## Imports and Runtime Setup


In [ ]:
import os, time, random, warnings, json, hashlib
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    confusion_matrix, ConfusionMatrixDisplay,
    roc_curve, auc, precision_recall_curve,
    classification_report, accuracy_score, f1_score
)
from torch.utils.data import TensorDataset, DataLoader
from collections import deque
from copy import deepcopy
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

warnings.filterwarnings("ignore")
torch.manual_seed(42); np.random.seed(42); random.seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
PIN_MEMORY = device.type == "cuda"
print("PyTorch:", torch.__version__, "| Device:", device, "| pin_memory:", PIN_MEMORY)

## Dataset Loading and Path Configuration


In [ ]:
# Dataset path for the GitHub-ready repository copy.
# Works when the notebook is launched from the repo root or from notebooks/.
from pathlib import Path
_repo_candidates = [Path.cwd(), Path.cwd().parent]
base_path = None
for _repo_root in _repo_candidates:
    _candidate = _repo_root / "data" / "dataset3"
    if (_candidate / "dataset_A.csv").exists():
        base_path = str(_candidate.resolve())
        break
if base_path is None:
    raise FileNotFoundError("Could not find data/dataset3/dataset_A.csv. Run from the repository root or notebooks/.")

# Runtime controls. Keep MAX_STEPS unchanged later; use caches/validation to reduce runtime.
FORCE_REBUILD_FEATURES = False
FORCE_RETRAIN_OFFLOADNET = False
FORCE_RETRAIN_FED_DDQN = False
FORCE_RETRAIN_BASELINES = False
FORCE_RETRAIN_ALLOCATOR = False
FORCE_REBUILD_ALLOCATOR_TARGETS = False
USE_ALLOCATOR_CACHES = True  # Stage-2 ResourceAllocator caches enabled; old allocator artifacts were purged.
FORCE_REBUILD_EVALUATION_CACHE = False
RUN_ALLOCATOR_ABLATION = True
ALLOCATOR_VERSION = "v66_rejection_residual_allocator_2026_05_30_cachefix"
RUN_FULL_ANALYTICS = True
RUN_PLOTS = True
LOAD_EXPERIMENT_RESULTS_ONLY = True
STAGE1_CACHE_ONLY = True  # Prevent accidental long retraining of stage-1/Fed-DDQN caches.

# v6.6 experiment-suite controls.
# Efficient default uses three seeds. For full reporting runs, set MULTI_SEED_QUICK_MODE=False.
RUN_MULTI_SEED_ABLATION = True
FORCE_RETRAIN_MULTI_SEED_ABLATION = False
MULTI_SEED_QUICK_MODE = True
FINAL_PAPER_SEEDS = [42, 77, 123, 2025, 999]
QUICK_EXPERIMENT_SEEDS = [42, 77, 123]
EXPERIMENT_SEEDS = QUICK_EXPERIMENT_SEEDS if MULTI_SEED_QUICK_MODE else FINAL_PAPER_SEEDS
CACHE_VERSION = "v66_rebalanced_fed_ddqn_2026_05_22"
RUN_CACHE_VERSION = "v66_full_run_cache_2026_05_29"

# Optional experiment/reproducibility hooks. Defaults are cache-preserving.
RUN_GAMMA_ZERO_ABLATION = False
RUN_BASELINE_MULTI_SEED = False
FORCE_RETRAIN_BASELINE_MULTI_SEED = False
EXPORT_REPRO_BUNDLE = True
BASELINE_MULTI_SEED_METHODS = ["DDQN", "FL-DDPG"]
BASELINE_MULTI_SEED_SEEDS = [42, 77, 123]


# v6.6 proposed Fed-DDQN controls.
# The core research model remains Federated DDQN; these flags rebalance how it learns.
PROPOSED_USE_ACTION_AWARE_ENV = False
PROPOSED_USE_PRIORITIZED_REPLAY = False
PROPOSED_MU_PROXIMAL = 0.003
PROPOSED_TARGET_TAU = 0.012
PROPOSED_FEDPROX_INTERVAL = 4
PROPOSED_FEDPROX_MAX_DRIFT = 1.0e4
PROPOSED_EDGE_OVERUSE_SOFT_CAP = 78.0
PROPOSED_HEAVY_TASK_EDGE_PENALTY = True
EVALUATE_FULL_TEST_SPLIT = True

cache_dir = os.path.join(base_path, "_v64_cache")
os.makedirs(cache_dir, exist_ok=True)
# Keep trained model artifacts in _v64_cache; place new optional hooks/exports in _v66_cache.
v66_cache_dir = os.path.join(base_path, "_v66_cache")
experiment_export_dir = os.path.join(v66_cache_dir, "experiment_exports")
baseline_multiseed_cache_dir = os.path.join(v66_cache_dir, "baseline_multiseed")
gamma_zero_cache_dir = os.path.join(v66_cache_dir, "gamma_zero")
for _cache_subdir in [v66_cache_dir, experiment_export_dir, baseline_multiseed_cache_dir, gamma_zero_cache_dir]:
    os.makedirs(_cache_subdir, exist_ok=True)

tasks         = pd.read_csv(os.path.join(base_path, "dataset_A.csv"))
edge_nodes    = pd.read_csv(os.path.join(base_path, "edge_nodes.csv"))
edge_state    = pd.read_csv(os.path.join(base_path, "edge_state.csv"))
cloud_nodes   = pd.read_csv(os.path.join(base_path, "cloud_nodes.csv"))
cloud_state   = pd.read_csv(os.path.join(base_path, "cloud_state.csv"))
network_state = pd.read_csv(os.path.join(base_path, "network_state.csv"))

_DATASET_FILES = [
    ("tasks", "dataset_A.csv", tasks),
    ("edge_nodes", "edge_nodes.csv", edge_nodes),
    ("edge_state", "edge_state.csv", edge_state),
    ("cloud_nodes", "cloud_nodes.csv", cloud_nodes),
    ("cloud_state", "cloud_state.csv", cloud_state),
    ("network_state", "network_state.csv", network_state),
]

def _build_cache_signature():
    payload = {"cache_version": CACHE_VERSION, "files": {}, "schemas": {}, "rows": {}}
    for name, filename, df in _DATASET_FILES:
        path = os.path.join(base_path, filename)
        stat = os.stat(path)
        payload["files"][filename] = {"size": stat.st_size, "mtime_ns": stat.st_mtime_ns}
        payload["schemas"][name] = list(df.columns)
        payload["rows"][name] = int(len(df))
    raw = json.dumps(payload, sort_keys=True).encode("utf-8")
    return hashlib.sha256(raw).hexdigest()

cache_signature = _build_cache_signature()
cache_prefix = f"v64_{cache_signature[:16]}"
labels_cache_path = os.path.join(cache_dir, f"{cache_prefix}_labels.npz")
features_cache_path = os.path.join(cache_dir, f"{cache_prefix}_features.npz")
offloadnet_model_path = os.path.join(cache_dir, f"{cache_prefix}_offloadnet.pt")
policy_eval_cache_path = os.path.join(cache_dir, f"{cache_prefix}_policy_eval.pkl")
allocator_cache_prefix = f"{cache_prefix}_{ALLOCATOR_VERSION}"
alloc_targets_cache_path = os.path.join(cache_dir, f"{allocator_cache_prefix}_alloc_targets.npz")
fed_model_path = os.path.join(cache_dir, f"{cache_prefix}_fed_ddqn_best.pt")
cent_model_path = os.path.join(cache_dir, f"{cache_prefix}_cent_ddqn.pt")
feddpg_model_path = os.path.join(cache_dir, f"{cache_prefix}_flddpg.pt")
baseline_meta_path = os.path.join(cache_dir, f"{cache_prefix}_baselines_meta.pt")
allocator_model_path = os.path.join(cache_dir, f"{allocator_cache_prefix}_allocator.pt")
allocator_stage2_cache_path = os.path.join(cache_dir, f"{allocator_cache_prefix}_stage2_allocations.pkl")
allocator_eval_cache_path = os.path.join(cache_dir, f"{allocator_cache_prefix}_allocator_eval.pkl")
allocator_ablation_cache_path = os.path.join(cache_dir, f"{allocator_cache_prefix}_allocator_ablation.pkl")
multiseed_cache_dir = os.path.join(cache_dir, "multiseed_ablation")
os.makedirs(multiseed_cache_dir, exist_ok=True)
results_cache_path = os.path.join(multiseed_cache_dir, f"{cache_prefix}_multiseed_ablation_results.pkl")
actions_cache_path = os.path.join(multiseed_cache_dir, f"{cache_prefix}_test_actions.pkl")
scenario_mask_cache_path = os.path.join(multiseed_cache_dir, f"{cache_prefix}_scenario_masks.npz")
gamma_zero_results_cache_path = os.path.join(gamma_zero_cache_dir, f"{cache_prefix}_gamma_zero_ablation_results.pkl")
baseline_multiseed_results_cache_path = os.path.join(baseline_multiseed_cache_dir, f"{cache_prefix}_baseline_multiseed_results.pkl")

def _safe_mtime(path):
    try:
        return os.path.getmtime(path) if os.path.exists(path) else None
    except OSError:
        return None

def _short_json_hash(payload):
    return hashlib.sha256(json.dumps(payload, sort_keys=True, default=str).encode("utf-8")).hexdigest()[:16]

def _short_array_hash(arr):
    arr = np.ascontiguousarray(np.asarray(arr))
    return hashlib.sha256(arr.view(np.uint8)).hexdigest()[:16]

def _cache_meta_matches(cached_meta, expected_meta):
    return dict(cached_meta or {}) == dict(expected_meta or {})

def install_numpy_pickle_compat():
    """Allow old/new NumPy pickle module paths to load across NumPy versions."""
    import importlib
    import sys

    aliases = {
        "numpy._core": "numpy.core",
        "numpy._core.numeric": "numpy.core.numeric",
        "numpy._core.multiarray": "numpy.core.multiarray",
        "numpy._core.umath": "numpy.core.umath",
        "numpy._core._multiarray_umath": "numpy.core._multiarray_umath",
        "numpy._core.numerictypes": "numpy.core.numerictypes",
        "numpy._core.fromnumeric": "numpy.core.fromnumeric",
        "numpy._core.shape_base": "numpy.core.shape_base",
        "numpy._core.function_base": "numpy.core.function_base",
    }
    for missing_name, available_name in aliases.items():
        if missing_name in sys.modules:
            continue
        try:
            sys.modules[missing_name] = importlib.import_module(available_name)
        except ModuleNotFoundError:
            pass


if "_PANDAS_READ_PICKLE" not in globals():
    _PANDAS_READ_PICKLE = pd.read_pickle


def read_pickle_compat(path):
    install_numpy_pickle_compat()
    return _PANDAS_READ_PICKLE(path)


install_numpy_pickle_compat()

for name, df in [("tasks",tasks),("edge_nodes",edge_nodes),("edge_state",edge_state),
                 ("cloud_nodes",cloud_nodes),("cloud_state",cloud_state),("network_state",network_state)]:
    print(f"{name:<15}: {df.shape}")
print("\nTask type distribution:\n", tasks["task_type"].value_counts())
print(f"\nCache directory: {cache_dir}")
print(f"v6.6 export directory: {experiment_export_dir}")
print(f"Cache signature: {cache_signature[:16]}")
print("Runtime flags:")
print(f"  FORCE_REBUILD_FEATURES={FORCE_REBUILD_FEATURES}")
print(f"  FORCE_RETRAIN_OFFLOADNET={FORCE_RETRAIN_OFFLOADNET}")
print(f"  FORCE_RETRAIN_FED_DDQN={FORCE_RETRAIN_FED_DDQN}")
print(f"  FORCE_RETRAIN_BASELINES={FORCE_RETRAIN_BASELINES}")
print(f"  FORCE_RETRAIN_ALLOCATOR={FORCE_RETRAIN_ALLOCATOR}")
print(f"  FORCE_REBUILD_ALLOCATOR_TARGETS={FORCE_REBUILD_ALLOCATOR_TARGETS}")
print(f"  USE_ALLOCATOR_CACHES={USE_ALLOCATOR_CACHES}")
print(f"  FORCE_REBUILD_EVALUATION_CACHE={FORCE_REBUILD_EVALUATION_CACHE}")
print(f"  RUN_ALLOCATOR_ABLATION={RUN_ALLOCATOR_ABLATION}")
print(f"  ALLOCATOR_VERSION={ALLOCATOR_VERSION}")
print(f"  RUN_FULL_ANALYTICS={RUN_FULL_ANALYTICS}")
print(f"  RUN_PLOTS={RUN_PLOTS}")
print(f"  LOAD_EXPERIMENT_RESULTS_ONLY={LOAD_EXPERIMENT_RESULTS_ONLY}")
print(f"  STAGE1_CACHE_ONLY={STAGE1_CACHE_ONLY}")
print(f"  RUN_MULTI_SEED_ABLATION={RUN_MULTI_SEED_ABLATION}")
print(f"  FORCE_RETRAIN_MULTI_SEED_ABLATION={FORCE_RETRAIN_MULTI_SEED_ABLATION}")
print(f"  RUN_GAMMA_ZERO_ABLATION={RUN_GAMMA_ZERO_ABLATION}")
print(f"  RUN_BASELINE_MULTI_SEED={RUN_BASELINE_MULTI_SEED}")
print(f"  FORCE_RETRAIN_BASELINE_MULTI_SEED={FORCE_RETRAIN_BASELINE_MULTI_SEED}")
print(f"  EXPORT_REPRO_BUNDLE={EXPORT_REPRO_BUNDLE}")
print(f"  MULTI_SEED_QUICK_MODE={MULTI_SEED_QUICK_MODE}")
print(f"  EXPERIMENT_SEEDS={EXPERIMENT_SEEDS}")


## QoS Parameters and Zone Mapping


In [ ]:
QUEUE_THRESHOLD        = 30
PACKET_LOSS_THRESHOLD  = 0.15
BANDWIDTH_THRESHOLD    = 80
SNR_THRESHOLD          = 5.0
ENERGY_FLOOR           = 50.0

# SLA targets per task type (ms)
SLA_MS = {
    "emergency": 50, "voice": 150, "sensor": 500, "telemetry": 800,
    "image": 1500, "video": 3000, "ai": 5000, "firmware_update": 60000,
}

EDGE_LAT_CAP  = 500.0
CLOUD_LAT_CAP = 800.0

# ── Zone mapping: edge_id -> location_zone ──────────────────────────────────
# This drives the non-IID federated split later.
edge_zone_map  = dict(zip(edge_nodes["edge_id"], edge_nodes["location_zone"]))
zone_edge_ids  = {}
for eid, zone in edge_zone_map.items():
    zone_edge_ids.setdefault(zone, []).append(eid)

# Attach zone to every task row
tasks["zone"] = tasks["assigned_edge_id"].map(edge_zone_map).fillna("urban")

print("Parameters loaded.")
print(f"Edge cap={EDGE_LAT_CAP} ms  |  Cloud cap={CLOUD_LAT_CAP} ms")
print("\nZone distribution:")
print(tasks["zone"].value_counts())
print("\nAvailable zones (client names):", sorted(zone_edge_ids.keys()))


## Lookup Tables for State Access


In [ ]:
print("Building lookup indexes ...")
edge_idx = {}
for row in edge_state.itertuples(index=False):
    edge_idx[(int(row.timestep), int(row.edge_id))] = row

net_idx = {}
for row in network_state.itertuples(index=False):
    net_idx[int(row.timestep)] = row

cloud_idx = {}
for row in cloud_state.itertuples(index=False):
    cloud_idx.setdefault(int(row.timestep), []).append(row)

print(f"edge_idx: {len(edge_idx):,} | net_idx: {len(net_idx):,} | cloud_idx: {len(cloud_idx):,}")


## Latency Computation Utilities


In [ ]:
def compute_latency(task_row):
    t       = int(task_row["arrival_time"])
    edge_id = int(task_row["assigned_edge_id"])

    edge   = edge_idx.get((t, edge_id))
    net    = net_idx.get(t)
    c_rows = cloud_idx.get(t, [])

    if edge is None or net is None or not c_rows:
        return 1, 1, EDGE_LAT_CAP, CLOUD_LAT_CAP
    if getattr(edge, "is_failed", 0) == 1:
        return 1, 1, EDGE_LAT_CAP, CLOUD_LAT_CAP
    if getattr(net, "is_outage", 0) == 1:
        return 1, 1, EDGE_LAT_CAP, CLOUD_LAT_CAP
    if task_row.get("is_corrupt", 0) == 1:
        return int(np.random.rand() > 0.5), 1, EDGE_LAT_CAP, CLOUD_LAT_CAP

    avail_c        = [r for r in c_rows if getattr(r, "is_in_maintenance", 0) == 0] or c_rows
    cloud_cpu      = float(np.mean([r.cloud_cpu_available   for r in avail_c]))
    cloud_lat_base = float(np.mean([r.cloud_latency_current for r in avail_c]))

    snr_factor   = float(np.clip(getattr(net, "snr_db", 25) / 25.0, 0.1, 1.5))
    effective_bw = max(float(net.uplink_bandwidth) * (1.0 - float(net.packet_loss_rate)) * snr_factor, 1.0)

    cpu_c  = float(task_row["cpu_cycles"])
    size_c = float(task_row["task_size_mb"])

    edge_latency = cpu_c / max(float(edge.edge_cpu_available), 1e-3) + float(edge.edge_queue_length) * 0.5
    if getattr(edge, "is_degrading", 0) == 1: edge_latency *= 1.35
    if task_row.get("is_low_battery", 0) == 1: edge_latency *= 1.20
    edge_latency = min(edge_latency, EDGE_LAT_CAP)

    cold_penalty  = 200.0 if (avail_c and getattr(avail_c[0], "had_cold_start", 0) == 1) else 0.0
    cloud_latency = min(
        size_c / effective_bw + cpu_c / max(cloud_cpu, 1e-3)
        + float(net.network_delay_ms) + cloud_lat_base + cold_penalty,
        CLOUD_LAT_CAP
    )
    if getattr(net, "is_jitter_storm", 0) == 1:
        cloud_latency = min(cloud_latency + float(np.random.exponential(30)), CLOUD_LAT_CAP)

    if task_row.get("impossible_deadline", 0) == 1:
        return 1, 1, edge_latency, cloud_latency

    rejected = int(
        float(edge.edge_queue_length)        > QUEUE_THRESHOLD        or
        float(net.packet_loss_rate)          > PACKET_LOSS_THRESHOLD  or
        effective_bw                         < BANDWIDTH_THRESHOLD    or
        float(edge.edge_energy_level)        < ENERGY_FLOOR           or
        float(getattr(net, "snr_db", 99.0)) < SNR_THRESHOLD
    )
    if task_row.get("task_type", "") == "emergency":
        rejected = 0   # emergency tasks are never QoS-rejected

    decision = 1 if rejected else (0 if edge_latency < cloud_latency else 1)
    return decision, rejected, edge_latency, cloud_latency

print("compute_latency() ready.")


## Label Generation and Rejection Targets


In [ ]:
labels_cache_loaded = False

if (not FORCE_REBUILD_FEATURES) and os.path.exists(labels_cache_path):
    try:
        cache = np.load(labels_cache_path)
        if int(cache["n_tasks"][0]) != len(tasks):
            raise ValueError("cached label row count does not match current tasks")
        tasks["offload_label"] = cache["offload_label"].astype(int)
        tasks["rejected"] = cache["rejected"].astype(int)
        tasks["edge_latency"] = cache["edge_latency"].astype(float)
        tasks["cloud_latency"] = cache["cloud_latency"].astype(float)
        tasks["rejection_flag"] = cache["rejection_flag"].astype(int)
        tasks["sla_violated"] = cache["sla_violated"].astype(int)
        labels_cache_loaded = True
        print(f"Loaded cached labels/latencies: {labels_cache_path}")
    except Exception as exc:
        print(f"Label cache ignored and rebuilt: {exc}")

if not labels_cache_loaded:
    print(f"Generating labels for {len(tasks):,} tasks ...")
    t0 = time.time()
    decisions, rejections, edge_lats, cloud_lats = [], [], [], []
    for _, row in tasks.iterrows():
        d, r, el, cl = compute_latency(row)
        decisions.append(d); rejections.append(r)
        edge_lats.append(el); cloud_lats.append(cl)

    tasks["offload_label"] = decisions
    tasks["rejected"] = rejections
    tasks["edge_latency"] = edge_lats
    tasks["cloud_latency"] = cloud_lats

    sla_arr = tasks["task_type"].map(SLA_MS).fillna(9999).values.astype(float)
    tasks["rejection_flag"] = (
        (tasks["impossible_deadline"] == 1) |
        ((tasks["edge_latency"] > sla_arr) & (tasks["cloud_latency"] > sla_arr))
    ).astype(int)
    chosen_latency = np.where(tasks["offload_label"].values == 0, tasks["edge_latency"].values, tasks["cloud_latency"].values)
    tasks["sla_violated"] = (chosen_latency > sla_arr).astype(int)

    np.savez_compressed(
        labels_cache_path,
        n_tasks=np.array([len(tasks)], dtype=np.int64),
        offload_label=tasks["offload_label"].values.astype(np.int8),
        rejected=tasks["rejected"].values.astype(np.int8),
        edge_latency=tasks["edge_latency"].values.astype(np.float32),
        cloud_latency=tasks["cloud_latency"].values.astype(np.float32),
        rejection_flag=tasks["rejection_flag"].values.astype(np.int8),
        sla_violated=tasks["sla_violated"].values.astype(np.int8),
    )
    print(f"Built labels in {time.time() - t0:.1f}s and saved cache: {labels_cache_path}")

print(f"\nLatency sanity:")
print(f"  edge_latency   min={tasks['edge_latency'].min():.2f}  max={tasks['edge_latency'].max():.2f}  mean={tasks['edge_latency'].mean():.2f}")
print(f"  cloud_latency  min={tasks['cloud_latency'].min():.2f}  max={tasks['cloud_latency'].max():.2f}  mean={tasks['cloud_latency'].mean():.2f}")
n_rej = int(tasks["rejection_flag"].sum())
print(f"\nrejection_flag=1: {n_rej:,}  ({n_rej/len(tasks)*100:.2f}%)")
print(f"  QoS rejected (rejected=1):  {tasks['rejected'].sum():,}  ({tasks['rejected'].mean()*100:.2f}%)")
print(f"  impossible_deadline=1:      {tasks['impossible_deadline'].sum():,}")
print(f"\nOffload label dist:\n{tasks['offload_label'].value_counts()}")
print(f"SLA violation rate: {tasks['sla_violated'].mean():.4f}")

_FEATURE_NAMES = [
    "task_size_mb","cpu_cycles","memory_req_mb","deadline_ms","priority_level",
    "energy_required","security_sensitivity","task_type","device_type",
    "is_real_time","is_encrypted","is_low_battery","has_dependency",
    "retransmission_count",
    "e_cpu","e_mem","e_queue","e_energy","e_fail","e_deg",
    "n_delay","n_bw","n_loss","n_snr","eff_bw","n_out","n_jit","n_cong",
    "c_cpu","c_over",
]
assert len(_FEATURE_NAMES) == 30, f"Feature list length mismatch: {len(_FEATURE_NAMES)}"
assert "rejection_flag" not in _FEATURE_NAMES, "rejection_flag leaked into features!"
print("\nLabel-leakage check: OK — rejection_flag is a target label only, not a feature.")
print(f"Feature count confirmed: {len(_FEATURE_NAMES)} inputs")

## Feature Engineering for Policy Inputs


In [ ]:
TASK_TYPES = ["sensor", "image", "ai", "video", "voice", "telemetry", "firmware_update", "emergency"]
DEVICE_TYPES = ["mobile", "sensor", "iot", "edge_device", "drone", "vehicle", "wearable", "industrial"]
TASK_TYPE_MAP = {t: i for i, t in enumerate(TASK_TYPES)}
DEVICE_TYPE_MAP = {d: i for i, d in enumerate(DEVICE_TYPES)}

FEATURE_NAMES = [
    "task_size_mb", "cpu_cycles", "memory_req_mb", "deadline_ms", "priority_level",
    "energy_required", "security_sensitivity", "task_type", "device_type",
    "is_real_time", "is_encrypted", "is_low_battery", "has_dependency",
    "retransmission_count",
    "e_cpu", "e_mem", "e_queue", "e_energy", "e_fail", "e_deg",
    "n_delay", "n_bw", "n_loss", "n_snr", "eff_bw", "n_out", "n_jit", "n_cong",
    "c_cpu", "c_over",
]
assert len(FEATURE_NAMES) == 30

_F_TASK_SIZE = 0
_F_CPU = 1
_F_MEM = 2
_F_DEADLINE = 3
_F_PRIORITY = 4
_F_ENERGY_REQ = 5
_F_SECURITY = 6
_F_TASK_TYPE = 7
_F_DEVICE_TYPE = 8
_F_REAL_TIME = 9
_F_ENCRYPTED = 10
_F_LOW_BATTERY = 11
_F_DEPENDENCY = 12
_F_RETX = 13
_F_E_CPU = 14
_F_E_MEM = 15
_F_E_QUEUE = 16
_F_E_ENERGY = 17
_F_E_FAIL = 18
_F_E_DEG = 19
_F_N_DELAY = 20
_F_N_BW = 21
_F_N_LOSS = 22
_F_N_SNR = 23
_F_EFF_BW = 24
_F_N_OUTAGE = 25
_F_N_JITTER = 26
_F_N_CONG = 27
_F_C_CPU = 28
_F_C_OVER = 29

def build_features(task_row):
    t = int(task_row["arrival_time"])
    edge_id = int(task_row["assigned_edge_id"])
    edge = edge_idx.get((t, edge_id))
    net = net_idx.get(t)
    c_rows = cloud_idx.get(t, [])

    e_cpu = float(edge.edge_cpu_available) if edge else 1.0
    e_mem = float(edge.edge_memory_available) if edge else 1.0
    e_queue = float(edge.edge_queue_length) if edge else 0.0
    e_energy = float(edge.edge_energy_level) if edge else 1000.0
    e_fail = float(getattr(edge, "is_failed", 0)) if edge else 0.0
    e_deg = float(getattr(edge, "is_degrading", 0)) if edge else 0.0
    n_delay = float(net.network_delay_ms) if net else 50.0
    n_bw = float(net.uplink_bandwidth) if net else 100.0
    n_loss = float(net.packet_loss_rate) if net else 0.05
    n_snr = float(getattr(net, "snr_db", 25.0)) if net else 25.0
    n_out = float(getattr(net, "is_outage", 0)) if net else 0.0
    n_jit = float(getattr(net, "is_jitter_storm", 0)) if net else 0.0
    n_cong = float(getattr(net, "is_congestion", 0)) if net else 0.0
    avail_c = [r for r in c_rows if getattr(r, "is_in_maintenance", 0) == 0]
    c_cpu = float(np.mean([r.cloud_cpu_available for r in avail_c])) if avail_c else 1.0
    c_over = float(np.mean([getattr(r, "is_overloaded", 0) for r in avail_c])) if avail_c else 0.0
    eff_bw = max(n_bw * (1.0 - n_loss) * float(np.clip(n_snr / 25.0, 0.1, 1.5)), 1.0)

    return [
        float(task_row["task_size_mb"]), float(task_row["cpu_cycles"]),
        float(task_row["memory_req_mb"]), float(task_row["deadline_ms"]),
        float(task_row["priority_level"]), float(task_row["energy_required"]),
        float(task_row["security_sensitivity"]),
        float(TASK_TYPE_MAP.get(task_row.get("task_type", "sensor"), 0)),
        float(DEVICE_TYPE_MAP.get(task_row.get("device_type", "iot"), 0)),
        float(task_row.get("is_real_time", 0)), float(task_row.get("is_encrypted", 0)),
        float(task_row.get("is_low_battery", 0)), float(task_row.get("has_dependency", 0)),
        float(task_row.get("retransmission_count", 0)),
        e_cpu, e_mem, e_queue, e_energy, e_fail, e_deg,
        n_delay, n_bw, n_loss, n_snr, eff_bw, n_out, n_jit, n_cong,
        c_cpu, c_over,
    ]

feature_cache_loaded = False
if (not FORCE_REBUILD_FEATURES) and os.path.exists(features_cache_path):
    try:
        cache = np.load(features_cache_path)
        features_raw = cache["features_raw"].astype(np.float32, copy=False)
        labels = cache["labels"].astype(np.int64, copy=False)
        if features_raw.shape != (len(tasks), len(FEATURE_NAMES)):
            raise ValueError(f"feature shape mismatch: {features_raw.shape}")
        if len(labels) != len(tasks):
            raise ValueError("label length mismatch")
        feature_cache_loaded = True
        print(f"Loaded cached feature matrix: {features_cache_path}")
    except Exception as exc:
        print(f"Feature cache ignored and rebuilt: {exc}")

if not feature_cache_loaded:
    print("Building feature matrix ...")
    t0 = time.time()
    features_raw = np.array(tasks.apply(build_features, axis=1).tolist(), dtype=np.float32)
    labels = tasks["offload_label"].values.astype(np.int64, copy=False)
    np.savez_compressed(features_cache_path, features_raw=features_raw, labels=labels)
    print(f"Built features in {time.time() - t0:.1f}s and saved cache: {features_cache_path}")

print(f"Feature matrix shape={features_raw.shape}")
print(f"NaN={np.isnan(features_raw).sum()}  Inf={np.isinf(features_raw).sum()}")
assert features_raw.shape == (len(tasks), 30), f"Expected features_raw shape {(len(tasks), 30)}, got {features_raw.shape}"
print("Feature constants ready for v6.6 optimized Fed-DDQN.")

## Temporal Split and Feature Scaling


In [ ]:
# Purged temporal train/validation/test split. Test remains untouched until final comparison.
train_mask = tasks["arrival_time"] <= 700
gap1_mask = tasks["arrival_time"].between(701, 715)
val_mask = tasks["arrival_time"].between(716, 850)
gap2_mask = tasks["arrival_time"].between(851, 865)
test_mask = tasks["arrival_time"].between(866, 1000)
gap_mask = gap1_mask | gap2_mask

assert not (train_mask & val_mask).any()
assert not (train_mask & test_mask).any()
assert not (val_mask & test_mask).any()
assert not (gap_mask & train_mask).any()
assert not (gap_mask & val_mask).any()
assert not (gap_mask & test_mask).any()

X_train_raw = features_raw[train_mask]
X_val_raw = features_raw[val_mask]
X_test_raw = features_raw[test_mask]
y_train_np = labels[train_mask]
y_val_np = labels[val_mask]
y_test_np = labels[test_mask]

train_df = tasks[train_mask].reset_index(drop=True)
val_df = tasks[val_mask].reset_index(drop=True)
test_df = tasks[test_mask].reset_index(drop=True)

print("Purged temporal split:")
print(f"  Train      arrival_time <= 700:       {len(train_df):,}")
print(f"  Gap 1      arrival_time 701-715:      {int(gap1_mask.sum()):,}")
print(f"  Validation arrival_time 716-850:      {len(val_df):,}")
print(f"  Gap 2      arrival_time 851-865:      {int(gap2_mask.sum()):,}")
print(f"  Test       arrival_time 866-1000:     {len(test_df):,}")
print(f"Train class dist: {np.bincount(y_train_np)}")
print(f"Val   class dist: {np.bincount(y_val_np)}")
print(f"Test  class dist: {np.bincount(y_test_np)}")

scaler = StandardScaler()
X_train_sc = np.clip(scaler.fit_transform(X_train_raw), -10, 10).astype(np.float32)
X_val_sc = np.clip(scaler.transform(X_val_raw), -10, 10).astype(np.float32)
X_test_sc = np.clip(scaler.transform(X_test_raw), -10, 10).astype(np.float32)
X_train_sc += np.random.normal(0, 0.01, X_train_sc.shape).astype(np.float32)

X_train = torch.tensor(X_train_sc, dtype=torch.float32)
X_val = torch.tensor(X_val_sc, dtype=torch.float32)
X_test = torch.tensor(X_test_sc, dtype=torch.float32)
X_train_dev = X_train.to(device)
X_val_dev = X_val.to(device)
X_test_dev = X_test.to(device)
y_train_t = torch.tensor(y_train_np.copy(), dtype=torch.long)
y_val_t = torch.tensor(y_val_np.copy(), dtype=torch.long)
y_test_t = torch.tensor(y_test_np.copy(), dtype=torch.long)
train_X_sc = X_train_sc
val_X_sc = X_val_sc
test_X_sc = X_test_sc

print(f"Tensors: X_train={X_train.shape}  X_val={X_val.shape}  X_test={X_test.shape}")
print("Scaler fit only on train split.")

## OffloadNet Training


In [ ]:

class OffloadNet(nn.Module):
    def __init__(self, inp):
        super().__init__()
        self.layers = nn.Sequential(
            nn.Linear(inp, 256), nn.BatchNorm1d(256), nn.ReLU(), nn.Dropout(0.30),
            nn.Linear(256, 128), nn.BatchNorm1d(128), nn.ReLU(), nn.Dropout(0.20),
            nn.Linear(128, 64),  nn.ReLU(),
            nn.Linear(64, 2)
        )
    def forward(self, x): return self.layers(x)


counts  = np.bincount(y_train_np)
weights = torch.tensor([1.0/counts[0], 1.0/counts[1]], dtype=torch.float32)
weights = (weights / weights.sum()).to(device)
print(f"Class weights  Edge={weights[0]:.4f}  Cloud={weights[1]:.4f}")

model     = OffloadNet(X_train.shape[1]).to(device)
criterion = nn.CrossEntropyLoss(weight=weights)
EPOCHS    = 20
tr_losses = []
offloadnet_loaded_from_cache = False

offloadnet_cache_meta = {
    "cache_signature": cache_signature,
    "run_cache_version": RUN_CACHE_VERSION,
    "model": "OffloadNet",
    "input_dim": int(X_train.shape[1]),
    "n_train": int(len(X_train)),
    "class_counts": counts.tolist(),
    "epochs": int(EPOCHS),
}

if (not FORCE_RETRAIN_OFFLOADNET) and os.path.exists(offloadnet_model_path):
    try:
        payload = torch.load(offloadnet_model_path, map_location=device)
        if not _cache_meta_matches(payload.get("meta"), offloadnet_cache_meta):
            raise ValueError("OffloadNet cache metadata mismatch")
        model.load_state_dict(payload["state_dict"])
        tr_losses = list(payload.get("tr_losses", []))
        offloadnet_loaded_from_cache = True
        print(f"Loaded cached OffloadNet: {offloadnet_model_path}")
    except Exception as exc:
        print(f"OffloadNet cache ignored and retrained: {exc}")

if globals().get("STAGE1_CACHE_ONLY", False) and not offloadnet_loaded_from_cache:
    raise FileNotFoundError(
        f"STAGE1_CACHE_ONLY=True and OffloadNet cache was not usable: {offloadnet_model_path}"
    )

if not offloadnet_loaded_from_cache:
    optimizer = optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=20)
    loader = DataLoader(TensorDataset(X_train, y_train_t), batch_size=512, shuffle=True, pin_memory=PIN_MEMORY)

    print("Training OffloadNet ...")
    for ep in range(EPOCHS):
        model.train()
        ep_loss = 0.0
        for xb, yb in loader:
            xb, yb = xb.to(device), yb.to(device)
            optimizer.zero_grad(set_to_none=True)
            loss = criterion(model(xb), yb)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            ep_loss += loss.item()
        scheduler.step()
        avg = ep_loss / len(loader)
        tr_losses.append(avg)
        print(f"  Epoch {ep+1:2d}/{EPOCHS}  loss={avg:.4f}")

    torch.save({
        "meta": offloadnet_cache_meta,
        "state_dict": _state_dict_cpu(model) if "_state_dict_cpu" in globals() else {k: v.detach().cpu().clone() for k, v in model.state_dict().items()},
        "tr_losses": tr_losses,
    }, offloadnet_model_path)
    print(f"Saved OffloadNet cache: {offloadnet_model_path}")

if RUN_FULL_ANALYTICS and RUN_PLOTS and tr_losses:
    plt.figure(figsize=(7,4))
    plt.plot(tr_losses, marker="o")
    plt.title("OffloadNet Training Loss")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.tight_layout()
    plt.show()
else:
    print("OffloadNet loss plot skipped or no loss history available.")

print(f"OffloadNet ready. cache_loaded={offloadnet_loaded_from_cache}")


## OffloadNet Evaluation


In [ ]:
model.eval()
with torch.no_grad():
    logits = model(X_test.to(device))
    preds = torch.argmax(logits, dim=1).cpu().numpy()
    probs = torch.softmax(logits, dim=1)[:, 1].cpu().numpy()

y_true = y_test_t.numpy()
print("=== OffloadNet Classification Report ===")
print(classification_report(y_true, preds, target_names=["Edge", "Cloud"]))

cm = confusion_matrix(y_true, preds)
fig, ax = plt.subplots(figsize=(5.8, 5.2))
ConfusionMatrixDisplay(cm, display_labels=["Edge", "Cloud"]).plot(ax=ax, colorbar=False)
ax.set_title("OffloadNet Confusion Matrix", fontweight="bold")
plt.tight_layout()
if RUN_PLOTS:
    plt.show()
else:
    plt.close()

fpr, tpr, _ = roc_curve(y_true, probs)
roc_auc = auc(fpr, tpr)
fig, ax = plt.subplots(figsize=(6.2, 5.0))
ax.plot(fpr, tpr, color="#378ADD", linewidth=2, label=f"AUC={roc_auc:.3f}")
ax.plot([0, 1], [0, 1], "--", color="gray", linewidth=1)
ax.set_title("OffloadNet ROC Curve", fontweight="bold")
ax.set_xlabel("False Positive Rate")
ax.set_ylabel("True Positive Rate")
ax.legend()
ax.grid(alpha=0.25)
plt.tight_layout()
if RUN_PLOTS:
    plt.show()
else:
    plt.close()

prec, rec, _ = precision_recall_curve(y_true, probs)
fig, ax = plt.subplots(figsize=(6.2, 5.0))
ax.plot(rec, prec, color="#1D9E75", linewidth=2)
ax.set_title("OffloadNet Precision-Recall Curve", fontweight="bold")
ax.set_xlabel("Recall")
ax.set_ylabel("Precision")
ax.grid(alpha=0.25)
plt.tight_layout()
if RUN_PLOTS:
    plt.show()
else:
    plt.close()

print("\n=== Per-Task-Type Accuracy ===")
test_tasks_eval = tasks[test_mask].copy().reset_index(drop=True)
test_tasks_eval["pred"] = preds
for tt in sorted(tasks["task_type"].unique()):
    m = test_tasks_eval["task_type"] == tt
    if not m.any():
        continue
    acc = accuracy_score(test_tasks_eval.loc[m, "offload_label"], test_tasks_eval.loc[m, "pred"])
    e_p = (test_tasks_eval.loc[m, "pred"] == 0).mean() * 100
    c_p = (test_tasks_eval.loc[m, "pred"] == 1).mean() * 100
    print(f"  {tt:<20}: acc={acc*100:5.1f}%  Edge={e_p:5.1f}%  Cloud={c_p:5.1f}%  n={m.sum():,}")
print("[CHART] OffloadNet evaluation exported as separate figures.")


## Offloading Environment


In [ ]:
class OffloadEnv:
    """
    Array-backed offline RL environment for edge/cloud offloading.

    v6.6 changes the proposed Fed-DDQN training target from edge-first behavior
    to evaluation-aligned adaptive offloading. Runtime pressure can still be
    enabled for ablations, but the proposed model defaults to static CSV latency
    because final comparison is also computed on static precomputed test rows.
    """

    HEAVY_TASK_TYPES = {"ai", "video", "image", "firmware_update"}
    URGENT_TASK_TYPES = {"emergency", "voice", "sensor", "telemetry"}

    def __init__(self, task_df: pd.DataFrame, feature_matrix: np.ndarray,
                 action_aware=None, heavy_task_penalty=None):
        self.df = task_df.reset_index(drop=True)
        self.X = np.asarray(feature_matrix, dtype=np.float32)
        self.n = len(self.df)
        self.ptr = 0

        self.action_aware = bool(PROPOSED_USE_ACTION_AWARE_ENV if action_aware is None else action_aware)
        self.heavy_task_penalty = bool(
            PROPOSED_HEAVY_TASK_EDGE_PENALTY if heavy_task_penalty is None else heavy_task_penalty
        )

        self.edge_lat = self.df["edge_latency"].to_numpy(dtype=np.float32)
        self.cloud_lat = self.df["cloud_latency"].to_numpy(dtype=np.float32)
        self.energy_req = self.df["energy_required"].to_numpy(dtype=np.float32)
        self.task_size = self.df["task_size_mb"].to_numpy(dtype=np.float32)
        self.cpu_cycles = self.df["cpu_cycles"].to_numpy(dtype=np.float32)
        self.priority = self.df["priority_level"].to_numpy(dtype=np.float32)
        self.low_battery = self.df.get("is_low_battery", pd.Series(0, index=self.df.index)).to_numpy(dtype=np.float32)
        self.impossible = self.df.get("impossible_deadline", pd.Series(0, index=self.df.index)).to_numpy(dtype=np.float32)
        self.task_type = self.df["task_type"].astype(str).to_numpy()
        self.sla = self.df["task_type"].map(SLA_MS).fillna(9999).to_numpy(dtype=np.float32)

        self.edge_pressure = 0.0
        self.cloud_pressure = 0.0
        self.network_pressure = 0.0
        self.energy_debt = 0.0

    def __len__(self):
        return self.n

    def _base_state(self, idx: int) -> np.ndarray:
        return self.X[idx].copy()

    def _state(self, idx: int) -> torch.Tensor:
        obs = self._base_state(idx)
        if self.action_aware:
            obs[_F_E_QUEUE] += min(self.edge_pressure, 6.0) * 0.20
            obs[_F_E_ENERGY] -= min(self.energy_debt, 6.0) * 0.12
            obs[_F_EFF_BW] -= min(self.network_pressure, 6.0) * 0.18
            obs[_F_N_LOSS] += min(self.network_pressure, 6.0) * 0.08
            obs[_F_C_OVER] += min(self.cloud_pressure, 6.0) * 0.18
        return torch.from_numpy(np.clip(obs, -10.0, 10.0).astype(np.float32, copy=False))

    def reset(self) -> torch.Tensor:
        self.ptr = 0
        self.edge_pressure = 0.0
        self.cloud_pressure = 0.0
        self.network_pressure = 0.0
        self.energy_debt = 0.0
        return self._state(0)

    def _dynamic_latencies(self, idx: int):
        edge_lat = float(self.edge_lat[idx])
        cloud_lat = float(self.cloud_lat[idx])
        if self.action_aware:
            edge_lat = min(edge_lat * (1.0 + 0.055 * self.edge_pressure) + 2.5 * self.energy_debt, EDGE_LAT_CAP)
            cloud_lat = min(cloud_lat * (1.0 + 0.045 * self.cloud_pressure + 0.060 * self.network_pressure), CLOUD_LAT_CAP)
        return edge_lat, cloud_lat

    def _apply_action_pressure(self, idx: int, action: int):
        if not self.action_aware:
            self.edge_pressure = 0.0
            self.cloud_pressure = 0.0
            self.network_pressure = 0.0
            self.energy_debt = 0.0
            return

        size_norm = min(float(self.task_size[idx]) / 80.0, 3.0)
        prio_norm = min(float(self.priority[idx]) / 5.0, 1.0)
        urgent = 1.0 if self.task_type[idx] in self.URGENT_TASK_TYPES else 0.0

        self.edge_pressure *= 0.965
        self.cloud_pressure *= 0.970
        self.network_pressure *= 0.972
        self.energy_debt *= 0.975

        if action == 0:
            self.edge_pressure += 0.10 + 0.09 * size_norm + 0.06 * prio_norm + 0.06 * urgent
            self.energy_debt += 0.04 + 0.06 * min(float(self.energy_req[idx]) / 150.0, 2.0)
            if self.low_battery[idx] > 0.5:
                self.energy_debt += 0.08
        else:
            self.cloud_pressure += 0.10 + 0.10 * size_norm + 0.04 * prio_norm
            self.network_pressure += 0.09 + 0.12 * size_norm + 0.03 * urgent

        self.edge_pressure = float(np.clip(self.edge_pressure, 0.0, 8.0))
        self.cloud_pressure = float(np.clip(self.cloud_pressure, 0.0, 8.0))
        self.network_pressure = float(np.clip(self.network_pressure, 0.0, 8.0))
        self.energy_debt = float(np.clip(self.energy_debt, 0.0, 8.0))

    def step(self, action: int):
        idx = self.ptr
        lat_e, lat_c = self._dynamic_latencies(idx)
        lat = lat_e if action == 0 else lat_c
        sla = float(self.sla[idx])
        task_type = self.task_type[idx]

        best = min(lat_e, lat_c)
        worst = max(lat_e, lat_c)
        regret = (lat - best) / max(best, 1.0)
        lat_r = 1.25 * (1.0 - 2.0 * ((lat - best) / (worst - best + 1e-6)))
        regret_r = -0.75 * float(np.clip(regret, 0.0, 2.0))

        sla_margin = float(np.clip((sla - lat) / max(sla, 1.0), -1.5, 1.0))
        sla_r = 1.05 * sla_margin if sla_margin >= 0.0 else 2.10 * sla_margin
        selected_reject = int(lat >= sla or self.impossible[idx] > 0.5)
        reject_r = -2.25 * selected_reject

        size_norm = min(float(self.task_size[idx]) / 120.0, 2.0)
        cpu_norm = min(float(self.cpu_cycles[idx]) / 900.0, 2.0)
        edge_worse_pct = max(lat_e - lat_c, 0.0) / max(lat_c, 1.0)
        cloud_worse_pct = max(lat_c - lat_e, 0.0) / max(lat_e, 1.0)

        routing_r = 0.0
        if action == 0:
            if lat_e > lat_c:
                routing_r -= min(1.35, 0.30 + 0.85 * edge_worse_pct)
            if self.heavy_task_penalty and task_type in self.HEAVY_TASK_TYPES and lat_e >= lat_c * 0.98:
                routing_r -= 0.30 + 0.16 * size_norm + 0.12 * cpu_norm
            routing_r -= 0.08 * float(self.energy_req[idx] / (self.energy_req[idx] + 100.0))
            if self.low_battery[idx] > 0.5 and task_type not in self.URGENT_TASK_TYPES:
                routing_r -= 0.05
            if self.action_aware:
                routing_r -= 0.035 * self.edge_pressure
        else:
            if task_type in self.URGENT_TASK_TYPES and lat_e < lat_c and lat_e < sla:
                routing_r -= min(0.70, 0.16 + 0.35 * cloud_worse_pct)
            if self.action_aware:
                routing_r -= 0.030 * (self.cloud_pressure + self.network_pressure)

        urgent_r = 0.0
        if task_type == "emergency":
            if action == 0 and lat_e <= lat_c and lat < sla:
                urgent_r = 0.35
            elif action == 1 and lat_e < lat_c and lat_e < sla:
                urgent_r = -0.35

        reward = float(np.clip(lat_r + regret_r + sla_r + reject_r + routing_r + urgent_r, -5.0, 5.0))
        self._apply_action_pressure(idx, action)

        done = self.ptr >= self.n - 2
        self.ptr = min(self.ptr + 1, self.n - 1)
        next_obs = self._state(self.ptr)
        info = {
            "latency": lat,
            "sla_met": int(lat < sla),
            "selected_reject": selected_reject,
            "edge_pressure": self.edge_pressure,
            "cloud_pressure": self.cloud_pressure,
            "network_pressure": self.network_pressure,
            "task_type": task_type,
        }
        return next_obs, reward, done, info


print("OffloadEnv v6.6 ready: evaluation-aligned reward, optional action-aware pressure.")
print(f"  action_aware_default={PROPOSED_USE_ACTION_AWARE_ENV} heavy_task_penalty={PROPOSED_HEAVY_TASK_EDGE_PENALTY}")


## Federated Client Partitioning


In [ ]:
# Collect training tasks and their scaled features.
train_df = tasks[train_mask].reset_index(drop=True)
train_X_sc = X_train_sc

zone_names = sorted(zone_edge_ids.keys())
zone_envs = {}

for zone in zone_names:
    zmask = train_df["zone"] == zone
    z_idx = zmask.values.nonzero()[0]

    if len(z_idx) == 0:
        print(f"  Zone {zone}: 0 tasks - skipping")
        continue

    z_df = train_df.iloc[z_idx].reset_index(drop=True)
    z_X = train_X_sc[z_idx]

    zone_envs[zone] = OffloadEnv(
        z_df,
        z_X,
        action_aware=PROPOSED_USE_ACTION_AWARE_ENV,
        heavy_task_penalty=PROPOSED_HEAVY_TASK_EDGE_PENALTY,
    )
    print(f"  Zone {zone:12s}: {len(z_df):,} training tasks")

    tt_dist = z_df["task_type"].value_counts(normalize=True)
    print(f"    Top task types: {dict(tt_dist.head(3).round(2))}")

zone_names = list(zone_envs.keys())
print(f"\nFederated clients: {len(zone_names)} zones = {zone_names}")
print("[v6.6] Non-IID split: each federated client trains on its own zone.")
print("[v6.6] Proposed Fed-DDQN env is evaluation-aligned by default.")


## Q-Network and Replay Buffer Utilities


In [ ]:
class QNetwork(nn.Module):
    '''Dueling DDQN: shared extractor -> separate Value + Advantage streams.'''
    def __init__(self, inp, act=2):
        super().__init__()
        self.feature = nn.Sequential(
            nn.Linear(inp, 256), nn.ReLU(),
            nn.Linear(256, 256), nn.ReLU(),
        )
        self.value_stream = nn.Sequential(
            nn.Linear(256, 128), nn.ReLU(),
            nn.Linear(128, 1),
        )
        self.adv_stream = nn.Sequential(
            nn.Linear(256, 128), nn.ReLU(),
            nn.Linear(128, act),
        )

    def forward(self, x):
        feat = self.feature(x)
        val = self.value_stream(feat)
        adv = self.adv_stream(feat)
        return val + adv - adv.mean(dim=-1, keepdim=True)


class ReplayBuffer:
    '''Tensor-backed replay buffer with optional prioritized sampling.'''
    def __init__(self, cap=30000, alpha=0.60, eps=1e-3):
        self.cap = int(cap)
        self.alpha = float(alpha)
        self.eps = float(eps)
        self.ptr = 0
        self.size = 0
        self.states = None
        self.next_states = None
        self.actions = torch.empty(self.cap, dtype=torch.long)
        self.rewards = torch.empty(self.cap, dtype=torch.float32)
        self.dones = torch.empty(self.cap, dtype=torch.float32)
        self.priorities = torch.ones(self.cap, dtype=torch.float32)

    def _lazy_init(self, state):
        if self.states is not None:
            return
        dim = int(torch.as_tensor(state).numel())
        self.states = torch.empty((self.cap, dim), dtype=torch.float32)
        self.next_states = torch.empty((self.cap, dim), dtype=torch.float32)

    def push(self, s, a, r, ns, done=False):
        self._lazy_init(s)
        idx = self.ptr
        self.states[idx].copy_(torch.as_tensor(s, dtype=torch.float32).view(-1).cpu())
        self.next_states[idx].copy_(torch.as_tensor(ns, dtype=torch.float32).view(-1).cpu())
        self.actions[idx] = int(a)
        self.rewards[idx] = float(r)
        self.dones[idx] = float(done)
        max_prio = float(self.priorities[:self.size].max().item()) if self.size > 0 else 1.0
        self.priorities[idx] = max(max_prio, self.eps)
        self.ptr = (self.ptr + 1) % self.cap
        self.size = min(self.size + 1, self.cap)

    def sample(self, n):
        idx = torch.randint(0, self.size, (int(n),))
        return self.states[idx], self.actions[idx], self.rewards[idx], self.next_states[idx], self.dones[idx]

    def sample_prioritized(self, n, beta=0.45):
        n = int(n)
        prios = self.priorities[:self.size].clamp_min(self.eps).pow(self.alpha)
        probs = prios / prios.sum()
        idx = torch.multinomial(probs, n, replacement=True)
        weights = (self.size * probs[idx]).pow(-float(beta))
        weights = weights / weights.max().clamp_min(self.eps)
        return self.states[idx], self.actions[idx], self.rewards[idx], self.next_states[idx], self.dones[idx], idx, weights.float()

    def update_priorities(self, idx, td_errors):
        td = torch.as_tensor(td_errors, dtype=torch.float32).view(-1).abs().cpu() + self.eps
        self.priorities[torch.as_tensor(idx, dtype=torch.long).cpu()] = td

    def mean_priority(self):
        if self.size == 0:
            return 1.0
        return float(self.priorities[:self.size].mean().item())

    def __len__(self):
        return self.size


def soft_update(target_model, source_model, tau=0.01):
    with torch.no_grad():
        for tgt, src in zip(target_model.parameters(), source_model.parameters()):
            tgt.data.mul_(1.0 - tau).add_(src.data, alpha=tau)


def federated_average(models, weights=None):
    '''Weighted FedAvg. Works with CPU or GPU models.'''
    if weights is None:
        weights = [1.0] * len(models)
    weights = np.asarray(weights, dtype=np.float64)
    weights = weights / max(weights.sum(), 1e-12)
    averaged = deepcopy(models[0])
    avg_sd = averaged.state_dict()
    model_sds = [m.state_dict() for m in models]
    with torch.no_grad():
        for key in avg_sd:
            acc = None
            for weight, state_dict in zip(weights, model_sds):
                value = state_dict[key].detach().float().to(avg_sd[key].device)
                acc = value * float(weight) if acc is None else acc + value * float(weight)
            avg_sd[key].copy_(acc.to(dtype=avg_sd[key].dtype))
    averaged.load_state_dict(avg_sd)
    return averaged

print("Dueling QNetwork / tensor ReplayBuffer with terminal masks / prioritized replay / soft-update / weighted FedAvg ready.")

## Federated DDQN Training Loop


In [ ]:
GAMMA = 0.95
EPSILON_START = 1.0
EPSILON_MIN = 0.05
EPSILON_DECAY = 0.92
BATCH_SIZE = 512
ROUNDS = 15
MAX_STEPS = 6000
TARGET_UPDATE = 200
TARGET_TAU = PROPOSED_TARGET_TAU
MU_PROXIMAL = PROPOSED_MU_PROXIMAL
PRIORITY_BETA_START = 0.40
PRIORITY_BETA_FRAMES = max(ROUNDS - 1, 1)
PATIENCE = 6


def _state_dict_cpu(model):
    return {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}


def evaluate_q_model_on_split(q_model, X_np, split_df):
    q_model.eval()
    with torch.inference_mode():
        if X_np is X_val_sc:
            xb = X_val_dev
        elif X_np is X_test_sc:
            xb = X_test_dev
        else:
            xb = torch.tensor(X_np, dtype=torch.float32, device=device)
        actions = torch.argmax(q_model(xb), dim=1).cpu().numpy().astype(np.int64)

    lat_e_all = split_df["edge_latency"].to_numpy(dtype=np.float32)
    lat_c_all = split_df["cloud_latency"].to_numpy(dtype=np.float32)
    valid = (lat_e_all < EDGE_LAT_CAP) | (lat_c_all < CLOUD_LAT_CAP)
    actions_v = actions[valid]
    lat_e = lat_e_all[valid]
    lat_c = lat_c_all[valid]
    lat = np.where(actions_v == 0, lat_e, lat_c)
    split_v = split_df.loc[valid].reset_index(drop=True)
    sla = split_v["task_type"].map(SLA_MS).fillna(9999).to_numpy(dtype=np.float32)
    impossible = split_v.get("impossible_deadline", pd.Series(0, index=split_v.index)).to_numpy(dtype=np.float32)
    selected_reject = (lat >= sla) | (impossible > 0.5)
    heavy_mask = split_v["task_type"].astype(str).isin(list(OffloadEnv.HEAVY_TASK_TYPES)).to_numpy()
    heavy_edge_bad = heavy_mask & (actions_v == 0) & (lat_e > lat_c * 1.02)

    return {
        "Avg Latency": float(lat.mean()) if len(lat) else 0.0,
        "SLA Miss %": float((lat >= sla).mean() * 100.0) if len(lat) else 0.0,
        "Rejection %": float(selected_reject.mean() * 100.0) if len(lat) else 0.0,
        "SLA %": float((lat < sla).mean() * 100.0) if len(lat) else 0.0,
        "Edge Usage %": float((actions_v == 0).mean() * 100.0) if len(actions_v) else 0.0,
        "Heavy Edge Bad %": float(heavy_edge_bad.mean() * 100.0) if len(actions_v) else 0.0,
        "N Eval": int(len(lat)),
    }


def fed_validation_score(metrics):
    edge_overuse = max(metrics["Edge Usage %"] - PROPOSED_EDGE_OVERUSE_SOFT_CAP, 0.0)
    return (
        metrics["Avg Latency"]
        + 5.0 * metrics["SLA Miss %"]
        + 2.5 * metrics["Rejection %"]
        + 0.8 * edge_overuse
        + 1.2 * metrics.get("Heavy Edge Bad %", 0.0)
    )


global_q = QNetwork(X_train.shape[1]).to(device)
target_q = deepcopy(global_q).to(device)
fed_ddqn_loaded_from_cache = False
federated_losses = []
federated_rewards = []
validation_scores = []
federated_zone_rewards = {zone: [] for zone in zone_names}
best_val_score = float("inf")
best_round = 0
rounds_ran = 0

if (not FORCE_RETRAIN_FED_DDQN) and os.path.exists(fed_model_path):
    try:
        payload = torch.load(fed_model_path, map_location=device)
        global_q.load_state_dict(payload["global_state_dict"])
        target_q.load_state_dict(payload.get("target_state_dict", payload["global_state_dict"]))
        federated_losses = list(payload.get("federated_losses", []))
        federated_rewards = list(payload.get("federated_rewards", []))
        validation_scores = list(payload.get("validation_scores", []))
        federated_zone_rewards = payload.get("federated_zone_rewards", federated_zone_rewards)
        best_val_score = float(payload.get("best_val_score", float("inf")))
        best_round = int(payload.get("best_round", len(federated_losses)))
        rounds_ran = int(payload.get("rounds_ran", len(federated_losses)))
        fed_ddqn_loaded_from_cache = True
        print(f"Loaded cached Fed-DDQN v6.6 checkpoint: {fed_model_path}")
        print(f"  best_round={best_round}  best_val_score={best_val_score:.4f}  rounds_ran={rounds_ran}")
    except Exception as exc:
        print(f"Fed-DDQN cache ignored and retrained: {exc}")

if globals().get("STAGE1_CACHE_ONLY", False) and not fed_ddqn_loaded_from_cache:
    raise FileNotFoundError(
        f"STAGE1_CACHE_ONLY=True and Fed-DDQN cache was not usable: {fed_model_path}"
    )

if not fed_ddqn_loaded_from_cache:
    print("Training proposed Federated DDQN v6.6 ...")
    print("Focus: non-IID zone clients + dueling DDQN + weighted FedAvg + soft target updates")
    print(f"Replay prioritized={PROPOSED_USE_PRIORITIZED_REPLAY} | FedProx mu={MU_PROXIMAL} | tau={TARGET_TAU}")
    print(f"Validation cap: edge overuse above {PROPOSED_EDGE_OVERUSE_SOFT_CAP:.1f}% is penalized")

    zone_models = {zone: deepcopy(global_q).to(device) for zone in zone_names}
    zone_targets = {zone: deepcopy(global_q).to(device) for zone in zone_names}
    zone_opts = {zone: optim.AdamW(zone_models[zone].parameters(), lr=5e-4, weight_decay=1e-4) for zone in zone_names}
    zone_buffers = {zone: ReplayBuffer(30000) for zone in zone_names}
    zone_eps = {zone: EPSILON_START for zone in zone_names}
    huber_none = nn.SmoothL1Loss(reduction="none")
    best_global_state = _state_dict_cpu(global_q)
    best_target_state = _state_dict_cpu(target_q)
    bad_rounds = 0

    for rnd in range(ROUNDS):
        beta = min(1.0, PRIORITY_BETA_START + (1.0 - PRIORITY_BETA_START) * rnd / PRIORITY_BETA_FRAMES)
        local_models = []
        agg_weights = []
        rnd_loss = 0.0
        rnd_reward = 0.0
        rnd_batches = 0
        rnd_steps = 0

        for zone in zone_names:
            env = zone_envs[zone]
            lm = zone_models[zone]
            tgt = zone_targets[zone]
            opt = zone_opts[zone]
            buf = zone_buffers[zone]
            lm.load_state_dict(global_q.state_dict())
            tgt.load_state_dict(target_q.state_dict())
            fedprox_anchors = (
                [param.detach().clone() for param in global_q.parameters()]
                if MU_PROXIMAL > 0.0 else []
            )
            lm.train()
            state = env.reset().to(device)
            zone_reward = 0.0
            steps_this_zone = 0

            while steps_this_zone < MAX_STEPS:
                eps = zone_eps[zone]
                if np.random.rand() < eps:
                    action = int(np.random.randint(0, 2))
                else:
                    with torch.no_grad():
                        action = int(torch.argmax(lm(state.unsqueeze(0))).item())

                next_state, reward, done, _ = env.step(action)
                buf.push(state.detach().cpu(), action, reward, next_state.detach().cpu(), done)
                rnd_reward += reward
                zone_reward += reward
                rnd_steps += 1
                steps_this_zone += 1

                if len(buf) > BATCH_SIZE:
                    if PROPOSED_USE_PRIORITIZED_REPLAY:
                        sb, ab, rb, nsb, db, idxs, isw = buf.sample_prioritized(BATCH_SIZE, beta=beta)
                        isw = isw.to(device)
                    else:
                        sb, ab, rb, nsb, db = buf.sample(BATCH_SIZE)
                        idxs = None
                        isw = torch.ones(BATCH_SIZE, dtype=torch.float32, device=device)

                    sb = sb.to(device)
                    nsb = nsb.to(device)
                    ab = ab.to(device)
                    rb = rb.to(device)
                    db = db.to(device)
                    current_q = lm(sb).gather(1, ab.unsqueeze(1)).squeeze(1)
                    with torch.no_grad():
                        next_actions = torch.argmax(lm(nsb), dim=1)
                        next_q = tgt(nsb).gather(1, next_actions.unsqueeze(1)).squeeze(1)
                        target_values = rb + GAMMA * (1.0 - db) * next_q

                    td_errors = current_q - target_values
                    loss_td = (isw * huber_none(current_q, target_values)).mean()
                    if MU_PROXIMAL > 0.0 and (rnd_batches % PROPOSED_FEDPROX_INTERVAL == 0):
                        drift = torch.zeros((), dtype=torch.float32, device=device)
                        for param, anchor in zip(lm.parameters(), fedprox_anchors):
                            drift = drift + torch.sum((param - anchor) ** 2)
                        drift = torch.nan_to_num(
                            torch.clamp(drift, max=PROPOSED_FEDPROX_MAX_DRIFT),
                            nan=0.0,
                            posinf=PROPOSED_FEDPROX_MAX_DRIFT,
                            neginf=0.0,
                        )
                        loss = loss_td + (MU_PROXIMAL / 2.0) * drift
                    else:
                        loss = loss_td
                    if not torch.isfinite(loss):
                        opt.zero_grad(set_to_none=True)
                        continue
                    opt.zero_grad(set_to_none=True)
                    loss.backward()
                    nn.utils.clip_grad_norm_(lm.parameters(), 1.0)
                    opt.step()
                    soft_update(tgt, lm, tau=TARGET_TAU)
                    if PROPOSED_USE_PRIORITIZED_REPLAY and idxs is not None:
                        buf.update_priorities(idxs, td_errors.detach().abs().cpu())
                    rnd_loss += float(loss_td.item())
                    rnd_batches += 1

                state = next_state.to(device)
                if done:
                    state = env.reset().to(device)

            zone_avg_reward = zone_reward / max(steps_this_zone, 1)
            federated_zone_rewards[zone].append(zone_avg_reward)
            zone_eps[zone] = max(zone_eps[zone] * EPSILON_DECAY, EPSILON_MIN)
            perf_factor = float(np.clip(1.0 + 0.05 * zone_avg_reward, 0.75, 1.25))
            agg_weights.append(len(env.df) * perf_factor)
            local_models.append(deepcopy(lm).cpu())

        averaged_q = federated_average(local_models, weights=agg_weights).to(device)
        global_q.load_state_dict(averaged_q.state_dict())
        soft_update(target_q, global_q, tau=0.25)

        avg_loss = rnd_loss / max(rnd_batches, 1)
        avg_reward = rnd_reward / max(rnd_steps, 1)
        federated_losses.append(avg_loss)
        federated_rewards.append(avg_reward)
        val_metrics = evaluate_q_model_on_split(global_q, X_val_sc, val_df)
        val_score = fed_validation_score(val_metrics)
        validation_scores.append(val_score)
        rounds_ran = rnd + 1
        eps_report = float(np.mean(list(zone_eps.values()))) if zone_eps else EPSILON_MIN
        improved = val_score < best_val_score

        if improved:
            best_val_score = val_score
            best_round = rnd + 1
            best_global_state = _state_dict_cpu(global_q)
            best_target_state = _state_dict_cpu(target_q)
            bad_rounds = 0
        else:
            bad_rounds += 1

        print(
            f"Round {rnd + 1:2d}/{ROUNDS}  loss={avg_loss:.4f}  reward={avg_reward:.4f}  "
            f"val_score={val_score:.3f}  val_lat={val_metrics['Avg Latency']:.2f}  "
            f"val_edge={val_metrics['Edge Usage %']:.2f}%  heavy_bad={val_metrics['Heavy Edge Bad %']:.2f}%  "
            f"eps(avg)={eps_report:.3f}  best={best_round}"
        )

        if bad_rounds >= PATIENCE:
            print(f"Early stopping: validation did not improve for {PATIENCE} rounds.")
            break

    global_q.load_state_dict(best_global_state)
    target_q.load_state_dict(best_target_state)
    torch.save({
        "global_state_dict": best_global_state,
        "target_state_dict": best_target_state,
        "federated_losses": federated_losses,
        "federated_rewards": federated_rewards,
        "validation_scores": validation_scores,
        "federated_zone_rewards": federated_zone_rewards,
        "best_val_score": best_val_score,
        "best_round": best_round,
        "rounds_ran": rounds_ran,
        "terminal_masked_targets": True,
        "target_formula": "r + gamma * (1 - done) * target_q(next_state)",
        "MAX_STEPS": MAX_STEPS,
        "ROUNDS": ROUNDS,
        "proposed_config": {
            "action_aware_env": PROPOSED_USE_ACTION_AWARE_ENV,
            "prioritized_replay": PROPOSED_USE_PRIORITIZED_REPLAY,
            "mu_proximal": MU_PROXIMAL,
            "target_tau": TARGET_TAU,
            "edge_overuse_soft_cap": PROPOSED_EDGE_OVERUSE_SOFT_CAP,
            "heavy_task_edge_penalty": PROPOSED_HEAVY_TASK_EDGE_PENALTY,
        },
    }, fed_model_path)
    print(f"Saved Fed-DDQN v6.6 best checkpoint: {fed_model_path}")

EPSILON = EPSILON_MIN
print("\nProposed Federated DDQN v6.6 ready.")
print(f"Zones used: {zone_names}")
print(f"Fed-DDQN cache loaded: {fed_ddqn_loaded_from_cache}")
print(f"Best validation round: {best_round} | best_val_score={best_val_score:.4f} | rounds_ran={rounds_ran}")
print("Applied: evaluation-aligned reward | non-IID FedAvg | dueling DDQN | terminal-masked targets | soft targets | light FedProx | validation penalties")


## Baseline Method Setup


In [ ]:
# ======================================================================
# LITERATURE BASELINE TRAINING / LOADING
# DDQN and FL-DDPG are comparison baselines only. Fed-DDQN remains proposed.
# ======================================================================

print("=" * 68)
print("LITERATURE BASELINE TRAINING / CACHE")
print("=" * 68)

cent_q = QNetwork(X_train.shape[1]).to(device)
feddpg_q = QNetwork(X_train.shape[1]).to(device)
cent_losses = []
feddpg_losses = []
local_losses = cent_losses
baseline_models_loaded_from_cache = False

baseline_cache_ready = (
    os.path.exists(cent_model_path) and
    os.path.exists(feddpg_model_path) and
    os.path.exists(baseline_meta_path)
)

if (not FORCE_RETRAIN_BASELINES) and baseline_cache_ready:
    try:
        cent_q.load_state_dict(torch.load(cent_model_path, map_location=device))
        feddpg_q.load_state_dict(torch.load(feddpg_model_path, map_location=device))
        meta = torch.load(baseline_meta_path, map_location="cpu")
        cent_losses = list(meta.get("cent_losses", []))
        feddpg_losses = list(meta.get("feddpg_losses", []))
        local_losses = cent_losses
        baseline_models_loaded_from_cache = True
        print(f"Loaded cached DDQN baseline: {cent_model_path}")
        print(f"Loaded cached FL-DDPG baseline: {feddpg_model_path}")
    except Exception as exc:
        print(f"Baseline cache ignored and retrained: {exc}")

if globals().get("STAGE1_CACHE_ONLY", False) and not baseline_models_loaded_from_cache:
    raise FileNotFoundError(
        "STAGE1_CACHE_ONLY=True and DDQN/FL-DDPG baseline caches were not usable: "
        f"{cent_model_path}, {feddpg_model_path}, {baseline_meta_path}"
    )

if not baseline_models_loaded_from_cache:
    print("\n[BL1] Training DDQN (centralised, Ullah et al. 2023) ...")
    cent_env = OffloadEnv(train_df, X_train_sc)
    cent_tgt = deepcopy(cent_q).to(device)
    cent_eps = 1.0

    for rnd in range(ROUNDS):
        opt_c = optim.Adam(cent_q.parameters(), lr=5e-4)
        buf_c = ReplayBuffer(8000)
        s_c = cent_env.reset().to(device)
        ep_l = ep_s = 0
        for step in range(MAX_STEPS):
            cent_q.eval()
            with torch.no_grad():
                act_c = int(np.random.randint(0, 2)) if np.random.rand() < cent_eps else int(torch.argmax(cent_q(s_c.unsqueeze(0))).item())
            cent_q.train()
            ns_c, rew_c, done_c, _ = cent_env.step(act_c)
            buf_c.push(s_c.cpu(), act_c, rew_c, ns_c.cpu(), done_c)
            if len(buf_c) > BATCH_SIZE:
                sb, ab, rb, nsb, db = buf_c.sample(BATCH_SIZE)
                sb = sb.to(device); nsb = nsb.to(device); ab = ab.to(device); rb = rb.to(device); db = db.to(device)
                cq_c = cent_q(sb).gather(1, ab.unsqueeze(1)).squeeze()
                with torch.no_grad():
                    na_c = torch.argmax(cent_q(nsb), dim=1)
                    nq_c = cent_tgt(nsb).gather(1, na_c.unsqueeze(1)).squeeze()
                    tv_c = rb + GAMMA * (1.0 - db) * nq_c
                loss_c = nn.MSELoss()(cq_c, tv_c)
                opt_c.zero_grad(); loss_c.backward()
                nn.utils.clip_grad_norm_(cent_q.parameters(), 1.0); opt_c.step()
                ep_l += float(loss_c.item()); ep_s += 1
            if step % TARGET_UPDATE == 0:
                cent_tgt.load_state_dict(cent_q.state_dict())
            s_c = ns_c.to(device)
            if done_c:
                s_c = cent_env.reset().to(device)
        cent_eps = max(cent_eps * EPSILON_DECAY, EPSILON_MIN)
        avg_c = ep_l / max(ep_s, 1)
        cent_losses.append(avg_c)
        print(f"  Round {rnd + 1:2d}/{ROUNDS}  loss={avg_c:.4f}  eps={cent_eps:.3f}")
    print("  DDQN (centralised) training complete.")

    print("\n[BL2] Training FL-DDPG approx (federated IID, Chen & Liu 2022) ...")
    N_IID = len(zone_names)
    iid_sz = len(X_train_sc) // max(N_IID, 1)
    iid_dfs = [train_df.iloc[i * iid_sz:(i + 1) * iid_sz].reset_index(drop=True) for i in range(N_IID)]
    iid_Xs = [X_train_sc[i * iid_sz:(i + 1) * iid_sz] for i in range(N_IID)]
    iid_envs = [OffloadEnv(iid_dfs[i], iid_Xs[i]) for i in range(N_IID)]
    feddpg_eps = 1.0

    for rnd in range(ROUNDS):
        local_ms_iid = []
        rnd_loss_iid = 0.0
        rnd_batches_iid = 0
        for env_i in iid_envs:
            lm_i = deepcopy(feddpg_q).to(device)
            tgt_i = deepcopy(lm_i).to(device)
            opt_i = optim.Adam(lm_i.parameters(), lr=5e-4)
            buf_i = ReplayBuffer(8000)
            s_i = env_i.reset().to(device)
            steps_i = 0
            while steps_i < MAX_STEPS:
                lm_i.eval()
                with torch.no_grad():
                    act_i = int(np.random.randint(0, 2)) if np.random.rand() < feddpg_eps else int(torch.argmax(lm_i(s_i.unsqueeze(0))).item())
                lm_i.train()
                ns_i, rew_i, done_i, _ = env_i.step(act_i)
                buf_i.push(s_i.cpu(), act_i, rew_i, ns_i.cpu(), done_i)
                steps_i += 1
                if len(buf_i) > BATCH_SIZE:
                    sb, ab, rb, nsb, db = buf_i.sample(BATCH_SIZE)
                    sb = sb.to(device); nsb = nsb.to(device); ab = ab.to(device); rb = rb.to(device); db = db.to(device)
                    cq_i = lm_i(sb).gather(1, ab.unsqueeze(1)).squeeze()
                    with torch.no_grad():
                        na_i = torch.argmax(lm_i(nsb), dim=1)
                        nq_i = tgt_i(nsb).gather(1, na_i.unsqueeze(1)).squeeze()
                        tv_i = rb + GAMMA * (1.0 - db) * nq_i
                    loss_i = nn.MSELoss()(cq_i, tv_i)
                    opt_i.zero_grad(); loss_i.backward()
                    nn.utils.clip_grad_norm_(lm_i.parameters(), 1.0); opt_i.step()
                    rnd_loss_iid += float(loss_i.item()); rnd_batches_iid += 1
                if steps_i % TARGET_UPDATE == 0:
                    tgt_i.load_state_dict(lm_i.state_dict())
                s_i = ns_i.to(device)
                if done_i:
                    s_i = env_i.reset().to(device)
            local_ms_iid.append(lm_i.cpu())
        feddpg_q = federated_average(local_ms_iid).to(device)
        feddpg_eps = max(feddpg_eps * EPSILON_DECAY, EPSILON_MIN)
        al_iid = rnd_loss_iid / max(rnd_batches_iid, 1)
        feddpg_losses.append(al_iid)
        print(f"  Round {rnd + 1:2d}/{ROUNDS}  loss={al_iid:.4f}  eps={feddpg_eps:.3f}")
    print("  FL-DDPG approx training complete.")

    torch.save(cent_q.state_dict(), cent_model_path)
    torch.save(feddpg_q.state_dict(), feddpg_model_path)
    torch.save({
        "cent_losses": cent_losses,
        "feddpg_losses": feddpg_losses,
        "MAX_STEPS": MAX_STEPS,
        "ROUNDS": ROUNDS,
        "terminal_masked_targets": True,
        "target_formula": "r + gamma * (1 - done) * target_q(next_state)",
    }, baseline_meta_path)
    print(f"Saved baseline caches:\n  {cent_model_path}\n  {feddpg_model_path}\n  {baseline_meta_path}")

print("\n[BL3-BL6] MTOSA/GTPSO/PTS-RA/JTOS: heuristic policies, no training needed.")
print("All 6 literature baselines ready.")
print(f"Baseline cache loaded: {baseline_models_loaded_from_cache}")

## Centralized DDQN Evaluation


In [ ]:
global_q.eval()
with torch.inference_mode():
    t0 = time.time()
    qout = global_q(X_test.to(device))
    pred_ddqn = torch.argmax(qout, dim=1).cpu().numpy()
    dt = time.time() - t0
print(f"Decision time ({len(X_test):,} samples): {dt * 1000:.2f} ms | per sample: {dt / len(X_test) * 1e6:.2f} us")

_test_mask_np = np.asarray(test_mask)
test_tasks_all = tasks[test_mask].reset_index(drop=True)
test_X_all = X_test_sc.astype(np.float32, copy=False)
test_raw_all = features_raw[_test_mask_np].astype(np.float32, copy=False)

rng_eval = np.random.default_rng(42)
sample_n = len(test_tasks_all) if EVALUATE_FULL_TEST_SPLIT else min(5000, len(test_tasks_all))
sample_positions = np.arange(len(test_tasks_all)) if EVALUATE_FULL_TEST_SPLIT else rng_eval.choice(len(test_tasks_all), size=sample_n, replace=False)
sample_tasks = test_tasks_all.iloc[sample_positions].reset_index(drop=True)
sample_X = test_X_all[sample_positions]
sample_raw = test_raw_all[sample_positions]

lat_e = sample_tasks["edge_latency"].to_numpy(dtype=np.float32)
lat_c = sample_tasks["cloud_latency"].to_numpy(dtype=np.float32)
valid = (lat_e < EDGE_LAT_CAP) | (lat_c < CLOUD_LAT_CAP)

with torch.inference_mode():
    actions = torch.argmax(global_q(torch.tensor(sample_X, dtype=torch.float32, device=device)), dim=1).cpu().numpy()

actions_v = actions[valid]
lat_v = np.where(actions_v == 0, lat_e[valid], lat_c[valid])
base_v = lat_c[valid]
sla_v = sample_tasks.loc[valid, "task_type"].map(SLA_MS).fillna(9999).to_numpy(dtype=np.float32)
n_eval = len(lat_v)
model_lat = lat_v
base_lat = base_v
avg_m = float(model_lat.mean()) if n_eval else 0.0
avg_b = float(base_lat.mean()) if n_eval else 0.0
improve = (avg_b - avg_m) / max(avg_b, 1e-6) * 100.0
sla_r = float((lat_v < sla_v).mean() * 100.0) if n_eval else 0.0
edge_cnt = int((actions_v == 0).sum())
cloud_cnt = int((actions_v == 1).sum())
skipped = int((~valid).sum())

print("\n===== Fed-DDQN PERFORMANCE =====")
print(f"Evaluated tasks (excl dual-cap) : {n_eval:,}  (skipped {skipped})")
print(f"Avg Latency  Fed-DDQN           : {avg_m:.4f} ms")
print(f"Avg Latency  Cloud Baseline     : {avg_b:.4f} ms")
print(f"Latency Improvement             : {improve:.2f}%")
print(f"Edge Usage                      : {edge_cnt / max(n_eval, 1) * 100:.2f}%")
print(f"Cloud Usage                     : {cloud_cnt / max(n_eval, 1) * 100:.2f}%")
print(f"SLA Compliance                  : {sla_r:.2f}%")
print(f"Rejection Rate (rejection_flag) : {tasks['rejection_flag'].mean():.4f}")


## Policy Evaluation Across Methods


In [ ]:
if "read_pickle_compat" not in globals():
    def install_numpy_pickle_compat():
        """Allow old/new NumPy pickle module paths to load across NumPy versions."""
        import importlib
        import sys

        aliases = {
            "numpy._core": "numpy.core",
            "numpy._core.numeric": "numpy.core.numeric",
            "numpy._core.multiarray": "numpy.core.multiarray",
            "numpy._core.umath": "numpy.core.umath",
            "numpy._core._multiarray_umath": "numpy.core._multiarray_umath",
            "numpy._core.numerictypes": "numpy.core.numerictypes",
            "numpy._core.fromnumeric": "numpy.core.fromnumeric",
            "numpy._core.shape_base": "numpy.core.shape_base",
            "numpy._core.function_base": "numpy.core.function_base",
        }
        for missing_name, available_name in aliases.items():
            if missing_name in sys.modules:
                continue
            try:
                sys.modules[missing_name] = importlib.import_module(available_name)
            except ModuleNotFoundError:
                pass

    def read_pickle_compat(path):
        install_numpy_pickle_compat()
        return pd.read_pickle(path)

install_numpy_pickle_compat()


def _model_actions(q_model, X_np):
    q_model.eval()
    with torch.inference_mode():
        if X_np is X_test_sc:
            xb = X_test_dev
        elif X_np is X_val_sc:
            xb = X_val_dev
        else:
            xb = torch.tensor(X_np, dtype=torch.float32, device=device)
        return torch.argmax(q_model(xb), dim=1).cpu().numpy().astype(np.int64)


def get_policy_actions(method, X_np, raw_np, eval_df):
    edge_possible = raw_np[:, _F_E_FAIL] < 0.5
    if method == "DDQN":
        return _model_actions(cent_q, X_np)
    if method == "FL-DDPG":
        return _model_actions(feddpg_q, X_np)
    if method == "Fed-DDQN (Proposed)":
        return _model_actions(global_q, X_np)
    if method == "Oracle":
        return (eval_df["cloud_latency"].to_numpy(dtype=np.float32) < eval_df["edge_latency"].to_numpy(dtype=np.float32)).astype(np.int64)
    if method == "MTOSA":
        return np.where((raw_np[:, _F_E_QUEUE] < 25.0) & (raw_np[:, _F_EFF_BW] > 100.0) & edge_possible, 0, 1)
    if method == "GTPSO":
        return np.where((raw_np[:, _F_E_ENERGY] > 80.0) & (raw_np[:, _F_C_OVER] > 0.25) & edge_possible, 0, 1)
    if method == "PTS-RA":
        return np.where((raw_np[:, _F_PRIORITY] >= 3.0) & (raw_np[:, _F_E_QUEUE] < 40.0) & edge_possible, 0, 1)
    if method == "JTOS":
        return np.where((raw_np[:, _F_DEADLINE] <= 800.0) & (raw_np[:, _F_E_QUEUE] < 35.0) & edge_possible, 0, 1)
    raise ValueError(f"Unknown policy method: {method}")


def evaluate_actions(method, actions, eval_df, raw_np):
    lat_e = eval_df["edge_latency"].to_numpy(dtype=np.float32)
    lat_c = eval_df["cloud_latency"].to_numpy(dtype=np.float32)
    valid = (lat_e < EDGE_LAT_CAP) | (lat_c < CLOUD_LAT_CAP)
    actions = np.asarray(actions, dtype=np.int64)
    lat_e = lat_e[valid]
    lat_c = lat_c[valid]
    actions_v = actions[valid]
    raw_v = raw_np[valid]
    df_v = eval_df.loc[valid].reset_index(drop=True)
    lat = np.where(actions_v == 0, lat_e, lat_c)
    sla = df_v["task_type"].map(SLA_MS).fillna(9999).to_numpy(dtype=np.float32)
    impossible = df_v.get("impossible_deadline", pd.Series(0, index=df_v.index)).to_numpy(dtype=np.float32)
    sla_ok = lat < sla
    selected_reject = (~sla_ok) | (impossible > 0.5)

    eff_bw = np.maximum(raw_v[:, _F_EFF_BW], 1.0)
    cloud_comm = raw_v[:, _F_N_DELAY] + raw_v[:, _F_TASK_SIZE] / eff_bw
    edge_comm = np.clip(raw_v[:, _F_E_QUEUE] * 0.5, 0.0, EDGE_LAT_CAP)
    comm_delay = np.where(actions_v == 1, cloud_comm, edge_comm)
    cloud_bw = np.clip(raw_v[:, _F_TASK_SIZE] / eff_bw, 0.0, 1.0) * 100.0
    edge_bw = np.zeros_like(cloud_bw)
    bw_cons = np.where(actions_v == 1, cloud_bw, edge_bw)
    n = len(lat)
    return {
        "Method": method,
        "Avg Latency": round(float(lat.mean()), 3) if n else 0.0,
        "SLA %": round(float(sla_ok.mean()) * 100.0, 2) if n else 0.0,
        "SLA Miss %": round(float((~sla_ok).mean()) * 100.0, 2) if n else 0.0,
        "Rejection %": round(float(selected_reject.mean()) * 100.0, 2) if n else 0.0,
        "Edge Usage %": round(float((actions_v == 0).mean()) * 100.0, 2) if n else 0.0,
        "Comm Delay (ms)": round(float(comm_delay.mean()), 3) if n else 0.0,
        "BW Consump %": round(float(bw_cons.mean()), 2) if n else 0.0,
        "N Eval": n,
    }


def evaluate_method(method, X_np, raw_np, eval_df):
    actions = get_policy_actions(method, X_np, raw_np, eval_df)
    return evaluate_actions(method, actions, eval_df, raw_np), actions


POLICY_METHODS = [
    "DDQN", "MTOSA", "FL-DDPG", "GTPSO", "PTS-RA", "JTOS",
    "Fed-DDQN (Proposed)", "Oracle",
]

print("Evaluating 8 policies on the untouched test split with batched inference/cache ...")
if EVALUATE_FULL_TEST_SPLIT:
    eval_positions = np.arange(len(test_tasks_all))
else:
    rng_eval2 = np.random.default_rng(99)
    eval_n = min(3000, int(test_mask.sum()))
    eval_positions = rng_eval2.choice(len(test_tasks_all), size=eval_n, replace=False)

eval_sample = test_tasks_all.iloc[eval_positions].reset_index(drop=True)
eval_X = test_X_all[eval_positions]
eval_raw = test_raw_all[eval_positions]

policy_eval_cache_meta = {
    "cache_signature": cache_signature,
    "run_cache_version": RUN_CACHE_VERSION,
    "policy_methods": POLICY_METHODS,
    "eval_full_test": bool(EVALUATE_FULL_TEST_SPLIT),
    "eval_positions_hash": _short_array_hash(eval_positions.astype(np.int64)),
    "n_eval": int(len(eval_sample)),
    "fed_model_mtime": _safe_mtime(fed_model_path),
    "cent_model_mtime": _safe_mtime(cent_model_path),
    "feddpg_model_mtime": _safe_mtime(feddpg_model_path),
    "baseline_meta_mtime": _safe_mtime(baseline_meta_path),
}

policy_eval_loaded_from_cache = False
if (not FORCE_REBUILD_EVALUATION_CACHE) and os.path.exists(policy_eval_cache_path):
    try:
        cached = read_pickle_compat(policy_eval_cache_path)
        if not _cache_meta_matches(cached.get("meta"), policy_eval_cache_meta):
            raise ValueError("policy evaluation cache metadata mismatch")
        df_compare = cached["df_compare"].copy()
        actions_by_method = {k: np.asarray(v, dtype=np.int64) for k, v in cached["actions_by_method"].items()}
        policy_eval_loaded_from_cache = True
        print(f"Loaded cached policy comparison/actions: {policy_eval_cache_path}")
    except Exception as exc:
        print(f"Policy evaluation cache ignored and rebuilt: {exc}")

if not policy_eval_loaded_from_cache:
    print(f"  Eval sample: {len(eval_sample):,} tasks | full_test={EVALUATE_FULL_TEST_SPLIT}")
    actions_by_method = {}
    all_results = []
    for method in POLICY_METHODS:
        res, actions = evaluate_method(method, eval_X, eval_raw, eval_sample)
        actions_by_method[method] = actions
        all_results.append(res)
        print(f"  {method:22s}  Lat={res['Avg Latency']:7.2f}ms  SLA={res['SLA %']:5.1f}%  "
              f"Miss={res['SLA Miss %']:5.1f}%  Rej={res['Rejection %']:5.1f}%  "
              f"Edge={res['Edge Usage %']:5.1f}%  N={res['N Eval']:,}")

    df_compare = pd.DataFrame(all_results).set_index("Method")
    pd.to_pickle({
        "meta": policy_eval_cache_meta,
        "df_compare": df_compare,
        "actions_by_method": actions_by_method,
    }, policy_eval_cache_path)
    print(f"Saved policy comparison/action cache: {policy_eval_cache_path}")

print("\n-- Validation checks --")
for method, row in df_compare.iterrows():
    assert 0.0 <= row["SLA Miss %"] <= 100.0
    assert 0.0 <= row["Rejection %"] <= 100.0
    assert 0.0 <= row["Edge Usage %"] <= 100.0
    assert abs(row["SLA %"] + row["SLA Miss %"] - 100.0) < 0.05, f"SLA complement fail: {method}"
print("  Bounds and complement checks OK")

oracle_lat = df_compare.loc["Oracle", "Avg Latency"]
fed_lat = df_compare.loc["Fed-DDQN (Proposed)", "Avg Latency"]
if oracle_lat <= fed_lat + 1e-6:
    print(f"  Oracle ({oracle_lat:.2f} ms) <= Fed-DDQN ({fed_lat:.2f} ms) OK")
else:
    print(f"  Check: Oracle ({oracle_lat:.2f} ms) is above Fed-DDQN ({fed_lat:.2f} ms) on this sample")

print("\n" + "=" * 96)
print("LITERATURE COMPARISON TABLE - v6.6 proposed Federated DDQN")
print("=" * 96)
cols = ["Avg Latency", "SLA %", "SLA Miss %", "Rejection %", "Edge Usage %", "Comm Delay (ms)", "BW Consump %"]
print(df_compare[cols].to_string())
print("=" * 96)
print("\n-- Fed-DDQN latency improvement over each baseline --")
prop_lat = df_compare.loc["Fed-DDQN (Proposed)", "Avg Latency"]
for method in ["DDQN", "MTOSA", "FL-DDPG", "GTPSO", "PTS-RA", "JTOS"]:
    baseline_lat = df_compare.loc[method, "Avg Latency"]
    pct = (baseline_lat - prop_lat) / max(baseline_lat, 1e-3) * 100.0
    print(f"  vs {method:10s}: {pct:+.1f}% latency reduction")
print(f"Policy comparison cache loaded: {policy_eval_loaded_from_cache}")


## Main Comparison Charts


In [ ]:
methods = df_compare.index.tolist()
METHOD_COLORS = {
    "DDQN": "#378ADD", "MTOSA": "#639922", "FL-DDPG": "#7F77DD",
    "GTPSO": "#D85A30", "PTS-RA": "#1D9E75", "JTOS": "#888780",
    "Fed-DDQN (Proposed)": "#A32D2D", "Oracle": "#BA7517",
}
colors = [METHOD_COLORS.get(m, "#888780") for m in methods]
metrics = ["Avg Latency", "SLA %", "SLA Miss %", "Edge Usage %", "Comm Delay (ms)", "BW Consump %"]
titles = ["Avg Latency (ms)", "SLA Compliance (%)", "SLA Miss Rate (%)", "Edge Usage (%)", "Comm Delay (ms)", "BW Consumption (%)"]
lower_better = [True, False, True, None, True, True]

for metric, title, lb in zip(metrics, titles, lower_better):
    vals = df_compare[metric].values.astype(float)
    fig, ax = plt.subplots(figsize=(11.5, 5.8))
    bars = ax.bar(methods, vals, color=colors, edgecolor="none", width=0.65)
    ax.set_title(f"{title}: Fed-DDQN vs Literature Baselines", fontsize=12, fontweight="bold")
    ax.set_ylabel(title, fontsize=10)
    ax.tick_params(axis="x", rotation=35, labelsize=8)
    if "Fed-DDQN (Proposed)" in methods:
        pi = methods.index("Fed-DDQN (Proposed)")
        bars[pi].set_edgecolor("black")
        bars[pi].set_linewidth(2.4)
    non_oracle = [(i, value) for i, (method, value) in enumerate(zip(methods, vals)) if method != "Oracle"]
    if non_oracle and lb is not None:
        best_i = min(non_oracle, key=lambda item: item[1])[0] if lb else max(non_oracle, key=lambda item: item[1])[0]
        bars[best_i].set_edgecolor("#BA7517")
        bars[best_i].set_linewidth(1.8)
    if "Oracle" in df_compare.index and metric in ["Avg Latency", "SLA %", "SLA Miss %", "Comm Delay (ms)"]:
        oracle_value = float(df_compare.loc["Oracle", metric])
        ax.axhline(oracle_value, color="#BA7517", linestyle="--", linewidth=1.4, label=f"Oracle={oracle_value:.1f}")
        ax.legend(fontsize=8)
    y_pad = max(float(np.nanmax(vals)) * 0.035, 0.8)
    for bar, value in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + y_pad, f"{value:.1f}",
                ha="center", va="bottom", fontsize=7)
    ax.grid(axis="y", alpha=0.22)
    fig.text(0.01, 0.01, "Black border = proposed Fed-DDQN. Gold border = best non-Oracle method where applicable.", fontsize=8)
    plt.tight_layout()
    if RUN_PLOTS:
        plt.show()
    else:
        plt.close()

from matplotlib.patches import Patch
legend_handles = [Patch(color=METHOD_COLORS.get(m, "#888780"), label=m) for m in methods]
fig, ax = plt.subplots(figsize=(10, 1.4))
ax.axis("off")
ax.legend(handles=legend_handles, loc="center", ncol=4, fontsize=9, frameon=False, title="Methods", title_fontsize=10)
plt.tight_layout()
if RUN_PLOTS:
    plt.show()
else:
    plt.close()
print(f"[CHART] {len(metrics)} comparison figures exported separately across {len(methods)} methods.")


## Performance Curves and Convergence


In [ ]:
NODE_COUNTS = [50, 100, 150, 200, 250, 300]
PLOT_POLICIES = [
    ("DDQN", "#378ADD", "-o"),
    ("MTOSA", "#639922", "-s"),
    ("FL-DDPG", "#7F77DD", "-^"),
    ("GTPSO", "#D85A30", "-D"),
    ("PTS-RA", "#1D9E75", "-v"),
    ("JTOS", "#888780", "-x"),
    ("Fed-DDQN (Proposed)", "#A32D2D", "-*"),
]

def evaluate_subset(method, idx):
    sub_df = eval_sample.iloc[idx].reset_index(drop=True)
    sub_raw = eval_raw[idx]
    cached_actions = actions_by_method[method][idx]
    return evaluate_actions(method, cached_actions, sub_df, sub_raw)

if RUN_FULL_ANALYTICS:
    print("Computing per-node curves with cached actions/results ...")
    curves_lat = {name: [] for name, _, _ in PLOT_POLICIES}
    curves_sla = {name: [] for name, _, _ in PLOT_POLICIES}
    curves_rej = {name: [] for name, _, _ in PLOT_POLICIES}
    rng_curve = np.random.default_rng(42)
    for n in NODE_COUNTS:
        idx = rng_curve.choice(len(eval_sample), size=min(n, len(eval_sample)), replace=False)
        for method, _, _ in PLOT_POLICIES:
            res = evaluate_subset(method, idx)
            curves_lat[method].append(res["Avg Latency"])
            curves_sla[method].append(res["SLA %"])
            curves_rej[method].append(res["SLA Miss %"])
        print(f"  n={n} done")

    curve_specs = [
        ("Avg Latency vs Number of Tasks", "Avg Latency (ms)", curves_lat),
        ("SLA Compliance vs Number of Tasks", "SLA Compliance (%)", curves_sla),
        ("SLA Miss Rate vs Number of Tasks", "SLA Miss Rate (%)", curves_rej),
    ]
    for title, ylabel, curve_data in curve_specs:
        fig, ax = plt.subplots(figsize=(9.5, 5.8))
        for method, color, marker in PLOT_POLICIES:
            ax.plot(NODE_COUNTS, curve_data[method], marker, color=color, label=method, linewidth=1.9, markersize=6)
        ax.set_title(title, fontweight="bold")
        ax.set_xlabel("Number of Tasks")
        ax.set_ylabel(ylabel)
        ax.legend(fontsize=7, ncol=2)
        ax.grid(alpha=0.3)
        plt.tight_layout()
        if RUN_PLOTS:
            plt.show()
        else:
            plt.close()

    convergence_specs = [
        ("Training Loss vs Round", "Round", "TD / MSE Loss", [
            (range(1, len(federated_losses) + 1), federated_losses, "Fed-DDQN (Proposed)", "#A32D2D", "-o"),
            (range(1, len(cent_losses) + 1), cent_losses, "DDQN (centralised)", "#378ADD", "--s"),
            (range(1, len(feddpg_losses) + 1), feddpg_losses, "FL-DDPG (IID approx)", "#7F77DD", "-.^")
        ]),
        ("Fed-DDQN Average Reward vs Round", "Round", "Avg Reward", [
            (range(1, len(federated_rewards) + 1), federated_rewards, "Fed-DDQN (Proposed)", "#A32D2D", "-o")
        ]),
        ("Fed-DDQN Validation Score vs Round", "Round", "Lower is better", [
            (range(1, len(validation_scores) + 1), validation_scores, "Validation Score", "#5A4FCF", "-D")
        ]),
    ]
    for title, xlabel, ylabel, series_list in convergence_specs:
        fig, ax = plt.subplots(figsize=(8.8, 5.4))
        for x_vals, y_vals, label, color, style in series_list:
            ax.plot(list(x_vals), y_vals, style, color=color, label=label, linewidth=2, markersize=6)
        if "Reward" in title:
            ax.axhline(0, color="gray", linestyle="--", lw=0.8)
        if "Validation" in title and best_round:
            ax.axvline(best_round, color="#A32D2D", linestyle="--", label=f"Best round={best_round}")
        ax.set_title(title, fontweight="bold")
        ax.set_xlabel(xlabel)
        ax.set_ylabel(ylabel)
        ax.legend(fontsize=8)
        ax.grid(alpha=0.3)
        plt.tight_layout()
        if RUN_PLOTS:
            plt.show()
        else:
            plt.close()

    print("\n=== SLA Compliance by Task Type ===")
    tt_rows = []
    for task_type in sorted(eval_sample["task_type"].unique()):
        idx = np.flatnonzero(eval_sample["task_type"].to_numpy() == task_type)
        row_tt = {"Task Type": task_type}
        for method, _, _ in PLOT_POLICIES:
            row_tt[method] = evaluate_subset(method, idx)["SLA %"] if len(idx) else 0.0
        tt_rows.append(row_tt)
    df_tt = pd.DataFrame(tt_rows).set_index("Task Type")
    print(df_tt.to_string())
    fig, ax = plt.subplots(figsize=(18, 6))
    x_tt = np.arange(len(df_tt.index)); w = 0.11
    for i, (method, color, _) in enumerate(PLOT_POLICIES):
        ax.bar(x_tt + i * w - w * 3, df_tt[method].values, w, label=method, color=color)
    ax.set_xticks(x_tt)
    ax.set_xticklabels(df_tt.index, rotation=20, fontsize=9)
    ax.axhline(80, color="red", linestyle="--", lw=1.2, label="80% target")
    ax.set_title("SLA Compliance (%) by Task Type -- All Methods", fontweight="bold")
    ax.set_ylabel("SLA Compliance (%)")
    ax.legend(fontsize=7, ncol=4)
    ax.grid(alpha=0.2)
    plt.tight_layout()
    if RUN_PLOTS:
        plt.show()
    else:
        plt.close()
    print("Extended analytics complete. Separate research figures generated.")
else:
    print("RUN_FULL_ANALYTICS=False; skipped extended analytics and repeated plots.")


## Task-Type Policy Breakdown


In [ ]:
print("=== Fed-DDQN Performance by Task Type ===\n")
rows_out = []
fed_actions_sample = get_policy_actions("Fed-DDQN (Proposed)", sample_X, sample_raw, sample_tasks)

for task_type in sorted(sample_tasks["task_type"].unique()):
    idx = np.flatnonzero(sample_tasks["task_type"].to_numpy() == task_type)
    if len(idx) == 0:
        continue
    sub_df = sample_tasks.iloc[idx].reset_index(drop=True)
    sub_raw = sample_raw[idx]
    sub_actions = fed_actions_sample[idx]
    res = evaluate_actions("Fed-DDQN (Proposed)", sub_actions, sub_df, sub_raw)

    lat_e = sub_df["edge_latency"].to_numpy(dtype=np.float32)
    lat_c = sub_df["cloud_latency"].to_numpy(dtype=np.float32)
    valid = (lat_e < EDGE_LAT_CAP) | (lat_c < CLOUD_LAT_CAP)
    actions_v = sub_actions[valid]
    model_lats = np.where(actions_v == 0, lat_e[valid], lat_c[valid])
    cloud_lats = lat_c[valid]
    if len(model_lats) == 0:
        continue
    improvement = (cloud_lats.mean() - model_lats.mean()) / max(float(cloud_lats.mean()), 1e-6) * 100.0
    rows_out.append({
        "Type": task_type,
        "N": int(res["N Eval"]),
        "Avg_DDQN": f"{res['Avg Latency']:.1f}",
        "Avg_Base": f"{cloud_lats.mean():.1f}",
        "Improv%": f"{improvement:.1f}",
        "Edge%": f"{res['Edge Usage %']:.1f}",
        "Cloud%": f"{100.0 - res['Edge Usage %']:.1f}",
        "SLA%": f"{res['SLA %']:.1f}",
    })

fed_task_type_table = pd.DataFrame(rows_out)
print(fed_task_type_table.to_string(index=False))

types = [row["Type"] for row in rows_out]
imps = [float(row["Improv%"]) for row in rows_out]
cols = ["#1D9E75" if value >= 0 else "#D85A30" for value in imps]
fig, ax = plt.subplots(figsize=(9.5, 5.2))
ax.bar(types, imps, color=cols)
ax.axhline(0, color="black", lw=1)
ax.set_title("Fed-DDQN Latency Improvement by Task Type", fontweight="bold")
ax.set_ylabel("Improvement over cloud baseline (%)")
ax.tick_params(axis="x", rotation=30)
ax.grid(axis="y", alpha=0.22)
plt.tight_layout()
if RUN_PLOTS:
    plt.show()
else:
    plt.close()

slas = [float(row["SLA%"]) for row in rows_out]
fig, ax = plt.subplots(figsize=(9.5, 5.2))
ax.bar(types, slas, color="#378ADD")
ax.axhline(80, color="red", linestyle="--", label="80% target")
ax.set_title("Fed-DDQN SLA Compliance by Task Type", fontweight="bold")
ax.set_ylabel("SLA Compliance (%)")
ax.tick_params(axis="x", rotation=30)
ax.legend()
ax.grid(axis="y", alpha=0.22)
plt.tight_layout()
if RUN_PLOTS:
    plt.show()
else:
    plt.close()

edge_usage = [float(row["Edge%"]) for row in rows_out]
fig, ax = plt.subplots(figsize=(9.5, 5.2))
ax.bar(types, edge_usage, color="#A32D2D", alpha=0.88, label="Edge")
ax.bar(types, [100.0 - value for value in edge_usage], bottom=edge_usage, color="#7F77DD", alpha=0.88, label="Cloud")
ax.set_title("Fed-DDQN Edge/Cloud Decision Mix by Task Type", fontweight="bold")
ax.set_ylabel("Decision share (%)")
ax.tick_params(axis="x", rotation=30)
ax.legend()
plt.tight_layout()
if RUN_PLOTS:
    plt.show()
else:
    plt.close()
print("[CHART] Fed-DDQN task-type figures exported separately.")


## Multi-Seed Ablation and Scenario Tables


In [ ]:
def install_numpy_pickle_compat():
    """Allow old/new NumPy pickle module paths to load across NumPy versions."""
    import importlib
    import sys

    aliases = {
        "numpy._core": "numpy.core",
        "numpy._core.numeric": "numpy.core.numeric",
        "numpy._core.multiarray": "numpy.core.multiarray",
        "numpy._core.umath": "numpy.core.umath",
        "numpy._core._multiarray_umath": "numpy.core._multiarray_umath",
        "numpy._core.numerictypes": "numpy.core.numerictypes",
        "numpy._core.fromnumeric": "numpy.core.fromnumeric",
        "numpy._core.shape_base": "numpy.core.shape_base",
        "numpy._core.function_base": "numpy.core.function_base",
    }
    for missing_name, available_name in aliases.items():
        if missing_name in sys.modules:
            continue
        try:
            sys.modules[missing_name] = importlib.import_module(available_name)
        except ModuleNotFoundError:
            pass


if "_PANDAS_READ_PICKLE" not in globals():
    _PANDAS_READ_PICKLE = pd.read_pickle


def read_pickle_compat(path):
    install_numpy_pickle_compat()
    return _PANDAS_READ_PICKLE(path)


install_numpy_pickle_compat()

# Define optional hook flags and paths when the main config cell has not been rerun.
RUN_GAMMA_ZERO_ABLATION = globals().get("RUN_GAMMA_ZERO_ABLATION", False)
RUN_BASELINE_MULTI_SEED = globals().get("RUN_BASELINE_MULTI_SEED", False)
FORCE_RETRAIN_BASELINE_MULTI_SEED = globals().get("FORCE_RETRAIN_BASELINE_MULTI_SEED", False)
EXPORT_REPRO_BUNDLE = globals().get("EXPORT_REPRO_BUNDLE", True)
BASELINE_MULTI_SEED_METHODS = globals().get("BASELINE_MULTI_SEED_METHODS", ["DDQN", "FL-DDPG"])
BASELINE_MULTI_SEED_SEEDS = globals().get("BASELINE_MULTI_SEED_SEEDS", [42, 77, 123])
v66_cache_dir = globals().get("v66_cache_dir", os.path.join(base_path, "_v66_cache"))
baseline_multiseed_cache_dir = globals().get("baseline_multiseed_cache_dir", os.path.join(v66_cache_dir, "baseline_multiseed"))
gamma_zero_cache_dir = globals().get("gamma_zero_cache_dir", os.path.join(v66_cache_dir, "gamma_zero"))
for _cache_subdir in [v66_cache_dir, baseline_multiseed_cache_dir, gamma_zero_cache_dir]:
    os.makedirs(_cache_subdir, exist_ok=True)
gamma_zero_results_cache_path = globals().get("gamma_zero_results_cache_path", os.path.join(gamma_zero_cache_dir, f"{cache_prefix}_gamma_zero_ablation_results.pkl"))
baseline_multiseed_results_cache_path = globals().get("baseline_multiseed_results_cache_path", os.path.join(baseline_multiseed_cache_dir, f"{cache_prefix}_baseline_multiseed_results.pkl"))

def set_experiment_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def safe_name(name):
    return "".join(ch.lower() if ch.isalnum() else "_" for ch in name).strip("_")


def clone_q_network(source_model):
    model_copy = QNetwork(X_train.shape[1]).to(device)
    model_copy.load_state_dict(source_model.state_dict())
    return model_copy


class AblationOffloadEnv(OffloadEnv):
    """OffloadEnv variant for v6.6 ablation controls."""
    def __init__(self, task_df, feature_matrix, action_aware=None, heavy_task_penalty=None):
        super().__init__(
            task_df,
            feature_matrix,
            action_aware=action_aware,
            heavy_task_penalty=heavy_task_penalty,
        )


ABLATION_CONFIGS = [
    {
        "name": "Fed-DDQN Proposed v6.6",
        "use_prioritized": PROPOSED_USE_PRIORITIZED_REPLAY,
        "mu_proximal": PROPOSED_MU_PROXIMAL,
        "soft_tau": PROPOSED_TARGET_TAU,
        "action_aware_env": PROPOSED_USE_ACTION_AWARE_ENV,
        "heavy_task_penalty": PROPOSED_HEAVY_TASK_EDGE_PENALTY,
        "validation_checkpoint": True,
    },
    {
        "name": "+ Prioritized Replay",
        "use_prioritized": True,
        "mu_proximal": PROPOSED_MU_PROXIMAL,
        "soft_tau": PROPOSED_TARGET_TAU,
        "action_aware_env": PROPOSED_USE_ACTION_AWARE_ENV,
        "heavy_task_penalty": PROPOSED_HEAVY_TASK_EDGE_PENALTY,
        "validation_checkpoint": True,
    },
    {
        "name": "+ Action-Aware Pressure",
        "use_prioritized": PROPOSED_USE_PRIORITIZED_REPLAY,
        "mu_proximal": PROPOSED_MU_PROXIMAL,
        "soft_tau": PROPOSED_TARGET_TAU,
        "action_aware_env": True,
        "heavy_task_penalty": PROPOSED_HEAVY_TASK_EDGE_PENALTY,
        "validation_checkpoint": True,
    },
    {
        "name": "No FedProx",
        "use_prioritized": PROPOSED_USE_PRIORITIZED_REPLAY,
        "mu_proximal": 0.0,
        "soft_tau": PROPOSED_TARGET_TAU,
        "action_aware_env": PROPOSED_USE_ACTION_AWARE_ENV,
        "heavy_task_penalty": PROPOSED_HEAVY_TASK_EDGE_PENALTY,
        "validation_checkpoint": True,
    },
    {
        "name": "Hard Target Update",
        "use_prioritized": PROPOSED_USE_PRIORITIZED_REPLAY,
        "mu_proximal": PROPOSED_MU_PROXIMAL,
        "soft_tau": None,
        "action_aware_env": PROPOSED_USE_ACTION_AWARE_ENV,
        "heavy_task_penalty": PROPOSED_HEAVY_TASK_EDGE_PENALTY,
        "validation_checkpoint": True,
    },
    {
        "name": "No Heavy-Task Penalty",
        "use_prioritized": PROPOSED_USE_PRIORITIZED_REPLAY,
        "mu_proximal": PROPOSED_MU_PROXIMAL,
        "soft_tau": PROPOSED_TARGET_TAU,
        "action_aware_env": PROPOSED_USE_ACTION_AWARE_ENV,
        "heavy_task_penalty": False,
        "validation_checkpoint": True,
    },
]


def make_variant_cache_path(config, seed):
    variant_key = safe_name(config["name"])
    variant_dir = gamma_zero_cache_dir if config.get("optional_cache_group") == "gamma_zero" else multiseed_cache_dir
    os.makedirs(variant_dir, exist_ok=True)
    return os.path.join(variant_dir, f"{cache_prefix}_{variant_key}_seed{int(seed)}.pt")


def variant_cache_meta(config, seed):
    return {
        "cache_signature": cache_signature,
        "run_cache_version": RUN_CACHE_VERSION,
        "variant": config["name"],
        "variant_config_hash": _short_json_hash(config),
        "seed": int(seed),
        "MAX_STEPS": int(MAX_STEPS),
        "ROUNDS": int(ROUNDS),
        "PATIENCE": int(PATIENCE),
        "BATCH_SIZE": int(BATCH_SIZE),
        "GAMMA": float(config.get("gamma", GAMMA)),
    }


def experiment_suite_meta():
    return {
        "cache_signature": cache_signature,
        "run_cache_version": RUN_CACHE_VERSION,
        "experiment_seeds": list(map(int, EXPERIMENT_SEEDS)),
        "ablation_names": [cfg["name"] for cfg in ABLATION_CONFIGS],
        "ablation_hashes": [_short_json_hash(cfg) for cfg in ABLATION_CONFIGS],
        "MAX_STEPS": int(MAX_STEPS),
        "ROUNDS": int(ROUNDS),
        "PATIENCE": int(PATIENCE),
        "EVALUATE_FULL_TEST_SPLIT": bool(EVALUATE_FULL_TEST_SPLIT),
        "n_test": int(len(test_df)),
    }


def build_variant_zone_envs(config):
    envs = {}
    for zone in zone_names:
        zmask = train_df["zone"] == zone
        z_idx = zmask.values.nonzero()[0]
        if len(z_idx) == 0:
            continue
        z_df = train_df.iloc[z_idx].reset_index(drop=True)
        z_X = X_train_sc[z_idx]
        envs[zone] = AblationOffloadEnv(
            z_df,
            z_X,
            action_aware=bool(config.get("action_aware_env", PROPOSED_USE_ACTION_AWARE_ENV)),
            heavy_task_penalty=bool(config.get("heavy_task_penalty", PROPOSED_HEAVY_TASK_EDGE_PENALTY)),
        )
    return envs


def evaluate_model_metrics(q_model, X_np, eval_df, raw_np=None, mask=None):
    q_model.eval()
    if mask is None:
        idx = np.arange(len(eval_df))
    else:
        idx = np.flatnonzero(np.asarray(mask, dtype=bool))
    if len(idx) == 0:
        return {"N Eval": 0, "Avg Latency": 0.0, "SLA %": 0.0, "SLA Miss %": 0.0, "Rejection %": 0.0, "Edge Usage %": 0.0, "Comm Delay (ms)": 0.0, "BW Consump %": 0.0}, np.array([], dtype=np.int64)

    X_sub = X_np[idx]
    df_sub = eval_df.iloc[idx].reset_index(drop=True)
    raw_sub = raw_np[idx] if raw_np is not None else features_raw[test_mask][idx]
    with torch.inference_mode():
        if X_np is X_test_sc and len(idx) == len(test_df) and np.array_equal(idx, np.arange(len(test_df))):
            xb = X_test_dev
        elif X_np is X_val_sc and len(idx) == len(val_df) and np.array_equal(idx, np.arange(len(val_df))):
            xb = X_val_dev
        else:
            xb = torch.tensor(X_sub, dtype=torch.float32, device=device)
        actions = torch.argmax(q_model(xb), dim=1).cpu().numpy().astype(np.int64)
    metrics = evaluate_actions("Fed-DDQN Variant", actions, df_sub, raw_sub)
    lat_e = df_sub["edge_latency"].to_numpy(dtype=np.float32)
    lat_c = df_sub["cloud_latency"].to_numpy(dtype=np.float32)
    valid = (lat_e < EDGE_LAT_CAP) | (lat_c < CLOUD_LAT_CAP)
    if valid.any():
        actions_v = actions[valid]
        lat = np.where(actions_v == 0, lat_e[valid], lat_c[valid])
        sla = df_sub.loc[valid, "task_type"].map(SLA_MS).fillna(9999).to_numpy(dtype=np.float32)
        impossible = df_sub.loc[valid].get("impossible_deadline", pd.Series(0, index=df_sub.loc[valid].index)).to_numpy(dtype=np.float32)
        metrics["Rejection %"] = round(float(((lat >= sla) | (impossible > 0.5)).mean() * 100.0), 2)
    else:
        metrics["Rejection %"] = 0.0
    return metrics, actions


def train_or_load_fed_variant(config, seed):
    variant_path = make_variant_cache_path(config, seed)
    model = QNetwork(X_train.shape[1]).to(device)

    expected_meta = variant_cache_meta(config, seed)
    if (not FORCE_RETRAIN_MULTI_SEED_ABLATION) and os.path.exists(variant_path):
        try:
            payload = torch.load(variant_path, map_location=device)
            if not _cache_meta_matches(payload.get("meta"), expected_meta):
                raise ValueError("variant cache metadata mismatch")
            model.load_state_dict(payload["state_dict"])
            payload["loaded_from_cache"] = True
            return model, payload
        except Exception as exc:
            print(f"Variant cache ignored and retrained ({config['name']} seed={seed}): {exc}")

    if config["name"] == "Fed-DDQN Proposed v6.6" and int(seed) == 42 and not FORCE_RETRAIN_MULTI_SEED_ABLATION and "global_q" in globals():
        model.load_state_dict(global_q.state_dict())
        payload = {
            "meta": expected_meta,
            "state_dict": _state_dict_cpu(model),
            "seed": int(seed),
            "gamma": float(config.get("gamma", GAMMA)),
            "terminal_masked_targets": True,
            "ablation": config["name"],
            "losses": list(federated_losses),
            "rewards": list(federated_rewards),
            "validation_scores": list(validation_scores),
            "best_val_score": float(best_val_score),
            "best_round": int(best_round),
            "rounds_ran": int(rounds_ran),
            "loaded_from_cache": False,
            "reused_current_global_q": True,
        }
        torch.save(payload, variant_path)
        return model, payload

    if LOAD_EXPERIMENT_RESULTS_ONLY:
        raise FileNotFoundError(f"Missing cached variant while LOAD_EXPERIMENT_RESULTS_ONLY=True: {variant_path}")

    set_experiment_seed(seed)
    model = QNetwork(X_train.shape[1]).to(device)
    target_model = clone_q_network(model)
    local_envs = build_variant_zone_envs(config)
    local_models = {zone: clone_q_network(model) for zone in local_envs}
    local_targets = {zone: clone_q_network(model) for zone in local_envs}
    local_opts = {zone: optim.AdamW(local_models[zone].parameters(), lr=5e-4, weight_decay=1e-4) for zone in local_envs}
    local_buffers = {zone: ReplayBuffer(30000) for zone in local_envs}
    local_eps = {zone: EPSILON_START for zone in local_envs}
    losses, rewards, val_scores = [], [], []
    best_state = _state_dict_cpu(model)
    best_target_state = _state_dict_cpu(target_model)
    best_score = float("inf")
    best_rnd = 0
    bad_rounds = 0
    huber = nn.SmoothL1Loss(reduction="none")
    variant_gamma = float(config.get("gamma", GAMMA))

    for rnd in range(ROUNDS):
        beta = min(1.0, PRIORITY_BETA_START + (1.0 - PRIORITY_BETA_START) * rnd / PRIORITY_BETA_FRAMES)
        round_models, round_weights = [], []
        round_loss = 0.0
        round_reward = 0.0
        round_batches = 0
        round_steps = 0

        for zone, env in local_envs.items():
            lm = local_models[zone]
            tgt = local_targets[zone]
            opt = local_opts[zone]
            buf = local_buffers[zone]
            lm.load_state_dict(model.state_dict())
            tgt.load_state_dict(target_model.state_dict())
            fedprox_mu = float(config["mu_proximal"])
            fedprox_anchors = (
                [param.detach().clone() for param in model.parameters()]
                if fedprox_mu > 0.0 else []
            )
            lm.train()
            state = env.reset().to(device)
            zone_reward = 0.0

            for step in range(MAX_STEPS):
                eps = local_eps[zone]
                if np.random.rand() < eps:
                    action = int(np.random.randint(0, 2))
                else:
                    with torch.no_grad():
                        action = int(torch.argmax(lm(state.unsqueeze(0))).item())
                next_state, reward, done, _ = env.step(action)
                buf.push(state.detach().cpu(), action, reward, next_state.detach().cpu(), done)
                round_reward += reward
                zone_reward += reward
                round_steps += 1

                if len(buf) > BATCH_SIZE:
                    if config["use_prioritized"]:
                        sb, ab, rb, nsb, db, idxs, isw = buf.sample_prioritized(BATCH_SIZE, beta=beta)
                        isw = isw.to(device)
                    else:
                        sb, ab, rb, nsb, db = buf.sample(BATCH_SIZE)
                        idxs = None
                        isw = torch.ones(BATCH_SIZE, dtype=torch.float32, device=device)
                    sb = sb.to(device); nsb = nsb.to(device)
                    ab = ab.to(device); rb = rb.to(device); db = db.to(device)
                    current_q = lm(sb).gather(1, ab.unsqueeze(1)).squeeze(1)
                    with torch.no_grad():
                        next_actions = torch.argmax(lm(nsb), dim=1)
                        next_q = tgt(nsb).gather(1, next_actions.unsqueeze(1)).squeeze(1)
                        target_q_values = rb + variant_gamma * (1.0 - db) * next_q
                    td_errors = current_q - target_q_values
                    td_loss = (isw * huber(current_q, target_q_values)).mean()
                    if fedprox_mu > 0.0 and (round_batches % PROPOSED_FEDPROX_INTERVAL == 0):
                        drift = torch.zeros((), dtype=torch.float32, device=device)
                        for param, anchor in zip(lm.parameters(), fedprox_anchors):
                            drift = drift + torch.sum((param - anchor) ** 2)
                        drift = torch.nan_to_num(
                            torch.clamp(drift, max=PROPOSED_FEDPROX_MAX_DRIFT),
                            nan=0.0,
                            posinf=PROPOSED_FEDPROX_MAX_DRIFT,
                            neginf=0.0,
                        )
                        loss = td_loss + (fedprox_mu / 2.0) * drift
                    else:
                        loss = td_loss
                    if not torch.isfinite(loss):
                        opt.zero_grad(set_to_none=True)
                        continue
                    opt.zero_grad(set_to_none=True)
                    loss.backward()
                    nn.utils.clip_grad_norm_(lm.parameters(), 1.0)
                    opt.step()
                    if config["soft_tau"] is None:
                        if step % TARGET_UPDATE == 0:
                            tgt.load_state_dict(lm.state_dict())
                    else:
                        soft_update(tgt, lm, tau=float(config["soft_tau"]))
                    if config["use_prioritized"] and idxs is not None:
                        buf.update_priorities(idxs, td_errors.detach().abs().cpu())
                    round_loss += float(td_loss.item())
                    round_batches += 1

                state = next_state.to(device)
                if done:
                    state = env.reset().to(device)

            local_eps[zone] = max(local_eps[zone] * EPSILON_DECAY, EPSILON_MIN)
            zone_avg_reward = zone_reward / max(MAX_STEPS, 1)
            perf_factor = float(np.clip(1.0 + 0.10 * zone_avg_reward, 0.60, 1.40))
            round_weights.append(len(env.df) * perf_factor)
            exported = QNetwork(X_train.shape[1])
            exported.load_state_dict({k: v.detach().cpu() for k, v in lm.state_dict().items()})
            round_models.append(exported)

        averaged = federated_average(round_models, weights=round_weights).to(device)
        model.load_state_dict(averaged.state_dict())
        soft_update(target_model, model, tau=0.25)
        avg_loss = round_loss / max(round_batches, 1)
        avg_reward = round_reward / max(round_steps, 1)
        losses.append(avg_loss)
        rewards.append(avg_reward)
        val_metrics = evaluate_q_model_on_split(model, X_val_sc, val_df)
        score = fed_validation_score(val_metrics)
        val_scores.append(score)

        if score < best_score:
            best_score = score
            best_rnd = rnd + 1
            best_state = _state_dict_cpu(model)
            best_target_state = _state_dict_cpu(target_model)
            bad_rounds = 0
        else:
            bad_rounds += 1

        if config["validation_checkpoint"] and bad_rounds >= PATIENCE:
            break

    if config["validation_checkpoint"]:
        model.load_state_dict(best_state)
        target_model.load_state_dict(best_target_state)
    else:
        best_state = _state_dict_cpu(model)
        best_target_state = _state_dict_cpu(target_model)

    payload = {
        "meta": expected_meta,
        "state_dict": best_state,
        "target_state_dict": best_target_state,
        "seed": int(seed),
        "ablation": config["name"],
        "config": config,
        "losses": losses,
        "rewards": rewards,
        "validation_scores": val_scores,
        "best_val_score": float(best_score),
        "best_round": int(best_rnd),
        "rounds_ran": int(len(losses)),
        "gamma": float(variant_gamma),
        "terminal_masked_targets": True,
        "target_formula": "r + gamma * (1 - done) * target_q(next_state)",
        "loaded_from_cache": False,
        "reused_current_global_q": False,
    }
    torch.save(payload, variant_path)
    return model, payload


def mean_std_text(values, digits=3):
    arr = pd.to_numeric(pd.Series(values), errors="coerce").dropna().to_numpy(dtype=float)
    if len(arr) == 0:
        return "0.000 ± 0.000"
    return f"{arr.mean():.{digits}f} ± {arr.std(ddof=0):.{digits}f}"


def build_scenario_masks(eval_df, raw_np):
    task_type = eval_df["task_type"].astype(str)
    scenario_masks = {
        "All test tasks": np.ones(len(eval_df), dtype=bool),
        "Normal traffic": (
            (raw_np[:, _F_N_CONG] < 0.5) &
            (raw_np[:, _F_N_OUTAGE] < 0.5) &
            (raw_np[:, _F_N_JITTER] < 0.5) &
            (raw_np[:, _F_E_QUEUE] < np.percentile(raw_np[:, _F_E_QUEUE], 75))
        ),
        "Congestion": raw_np[:, _F_N_CONG] >= 0.5,
        "Outage": raw_np[:, _F_N_OUTAGE] >= 0.5,
        "Jitter storm": raw_np[:, _F_N_JITTER] >= 0.5,
        "High edge queue": raw_np[:, _F_E_QUEUE] >= np.percentile(raw_np[:, _F_E_QUEUE], 90),
        "Emergency": task_type.eq("emergency").to_numpy(),
        "Real-time": eval_df.get("is_real_time", pd.Series(0, index=eval_df.index)).to_numpy(dtype=int) == 1,
        "Low battery": eval_df.get("is_low_battery", pd.Series(0, index=eval_df.index)).to_numpy(dtype=int) == 1,
        "AI/video-heavy": task_type.isin(["ai", "video"]).to_numpy(),
        "Firmware": task_type.eq("firmware_update").to_numpy(),
    }
    return {name: mask for name, mask in scenario_masks.items() if int(np.asarray(mask).sum()) > 0}


def load_cached_experiment_results():
    if not os.path.exists(results_cache_path):
        return None
    cached = read_pickle_compat(results_cache_path)
    if not _cache_meta_matches(cached.get("suite_meta"), experiment_suite_meta()):
        return None
    return cached


def save_experiment_results(payload):
    pd.to_pickle(payload, results_cache_path)
    print(f"Saved multi-seed/ablation result-table cache: {results_cache_path}")


def load_or_build_scenario_masks(test_raw_for_suite):
    if os.path.exists(scenario_mask_cache_path):
        try:
            mask_cache = np.load(scenario_mask_cache_path, allow_pickle=True)
            names = [str(x) for x in mask_cache["names"].tolist()]
            masks = {name: mask_cache[f"mask_{i}"].astype(bool) for i, name in enumerate(names)}
            if all(len(mask) == len(test_df) for mask in masks.values()):
                return masks, True
        except Exception as exc:
            print(f"Scenario mask cache ignored and rebuilt: {exc}")
    masks = build_scenario_masks(test_df, test_raw_for_suite)
    save_payload = {"names": np.array(list(masks.keys()), dtype=object)}
    for i, (name, mask) in enumerate(masks.items()):
        save_payload[f"mask_{i}"] = np.asarray(mask, dtype=bool)
    np.savez_compressed(scenario_mask_cache_path, **save_payload)
    return masks, False


def load_or_build_baseline_actions(test_raw_for_suite):
    if (not FORCE_REBUILD_EVALUATION_CACHE) and os.path.exists(actions_cache_path):
        try:
            cached = read_pickle_compat(actions_cache_path)
            expected_meta = {
                "cache_signature": cache_signature,
                "run_cache_version": RUN_CACHE_VERSION,
                "n_test": int(len(test_df)),
                "fed_model_mtime": _safe_mtime(fed_model_path),
                "cent_model_mtime": _safe_mtime(cent_model_path),
                "feddpg_model_mtime": _safe_mtime(feddpg_model_path),
            }
            if _cache_meta_matches(cached.get("meta"), expected_meta):
                return cached["baseline_actions"], True
        except Exception as exc:
            print(f"Baseline action cache ignored and rebuilt: {exc}")
    baseline_methods = ["DDQN", "MTOSA", "FL-DDPG", "GTPSO", "PTS-RA", "JTOS", "Oracle"]
    baseline_actions = {method: get_policy_actions(method, X_test_sc, test_raw_for_suite, test_df) for method in baseline_methods}
    expected_meta = {
        "cache_signature": cache_signature,
        "run_cache_version": RUN_CACHE_VERSION,
        "n_test": int(len(test_df)),
        "fed_model_mtime": _safe_mtime(fed_model_path),
        "cent_model_mtime": _safe_mtime(cent_model_path),
        "feddpg_model_mtime": _safe_mtime(feddpg_model_path),
    }
    pd.to_pickle({"meta": expected_meta, "baseline_actions": baseline_actions}, actions_cache_path)
    return baseline_actions, False


multiseed_records = []
full_seed_actions = {}
variant_payloads = {}
results_loaded_from_cache = False

if RUN_MULTI_SEED_ABLATION:
    cached_results = None if FORCE_RETRAIN_MULTI_SEED_ABLATION else load_cached_experiment_results()
    if cached_results is not None:
        df_multiseed_raw = cached_results["df_multiseed_raw"]
        df_ablation_summary = cached_results["df_ablation_summary"]
        df_full_multiseed = cached_results["df_full_multiseed"]
        df_scenario_final = cached_results["df_scenario_final"]
        full_seed_actions = {int(k): np.asarray(v, dtype=np.int64) for k, v in cached_results.get("full_seed_actions", {}).items()}
        multiseed_records = cached_results.get("multiseed_records", df_multiseed_raw.to_dict("records"))
        results_loaded_from_cache = True
        print(f"Loaded cached multi-seed/ablation result tables: {results_cache_path}")
    else:
        if LOAD_EXPERIMENT_RESULTS_ONLY:
            raise FileNotFoundError(f"LOAD_EXPERIMENT_RESULTS_ONLY=True but result cache is missing or stale: {results_cache_path}")

        print("=" * 88)
        print("MULTI-SEED + ABLATION SUITE FOR PROPOSED FED-DDQN v6.6")
        print("=" * 88)
        print(f"Seeds: {EXPERIMENT_SEEDS}")
        print(f"Quick mode: {MULTI_SEED_QUICK_MODE} | Full reporting run seeds: {FINAL_PAPER_SEEDS}")
        print(f"MAX_STEPS remains {MAX_STEPS}")
        test_raw_for_suite = features_raw[test_mask].astype(np.float32, copy=False)

        for config in ABLATION_CONFIGS:
            for seed in EXPERIMENT_SEEDS:
                t0_suite = time.time()
                model_variant, payload = train_or_load_fed_variant(config, seed)
                metrics, actions = evaluate_model_metrics(model_variant, X_test_sc, test_df, test_raw_for_suite)
                if config["name"] == "Fed-DDQN Proposed v6.6":
                    full_seed_actions[int(seed)] = actions
                variant_payloads[(config["name"], int(seed))] = payload
                multiseed_records.append({
                    "Ablation": config["name"],
                    "Seed": int(seed),
                    "Avg Latency": metrics["Avg Latency"],
                    "SLA %": metrics["SLA %"],
                    "SLA Miss %": metrics["SLA Miss %"],
                    "Rejection %": metrics["Rejection %"],
                    "Edge Usage %": metrics["Edge Usage %"],
                    "Comm Delay (ms)": metrics["Comm Delay (ms)"],
                    "BW Consump %": metrics["BW Consump %"],
                    "Best Val Score": payload.get("best_val_score", np.nan),
                    "Best Round": payload.get("best_round", 0),
                    "Rounds Ran": payload.get("rounds_ran", 0),
                    "Loaded Cache": bool(payload.get("loaded_from_cache", False)),
                    "Runtime Sec": round(time.time() - t0_suite, 2),
                })
                print(f"{config['name']:<26s} seed={seed:<5d} "
                      f"lat={metrics['Avg Latency']:.3f} SLA={metrics['SLA %']:.2f}% "
                      f"rej={metrics['Rejection %']:.2f}% cache={payload.get('loaded_from_cache', False)}")

        df_multiseed_raw = pd.DataFrame(multiseed_records)
        ablation_rows = []
        for ablation_name, group in df_multiseed_raw.groupby("Ablation", sort=False):
            ablation_rows.append({
                "Ablation": ablation_name,
                "Runs": int(len(group)),
                "Avg Latency": mean_std_text(group["Avg Latency"], 3),
                "SLA %": mean_std_text(group["SLA %"], 2),
                "Rejection %": mean_std_text(group["Rejection %"], 2),
                "Edge Usage %": mean_std_text(group["Edge Usage %"], 2),
                "Best Val Score": mean_std_text(group["Best Val Score"], 3),
                "Rounds Ran": mean_std_text(group["Rounds Ran"], 1),
            })
        df_ablation_summary = pd.DataFrame(ablation_rows)
        df_full_multiseed = df_multiseed_raw[df_multiseed_raw["Ablation"] == "Fed-DDQN Proposed v6.6"].reset_index(drop=True)

        scenario_masks, scenario_masks_loaded_from_cache = load_or_build_scenario_masks(test_raw_for_suite)
        baseline_actions_full_test, baseline_actions_loaded_from_cache = load_or_build_baseline_actions(test_raw_for_suite)
        scenario_rows = []

        for scenario_name, mask in scenario_masks.items():
            mask = np.asarray(mask, dtype=bool)
            seed_metric_rows = []
            for seed, actions in full_seed_actions.items():
                seed_metric_rows.append(evaluate_actions("Fed-DDQN Proposed v6.6", actions[mask], test_df.loc[mask].reset_index(drop=True), test_raw_for_suite[mask]))
            scenario_rows.append({
                "Scenario": scenario_name,
                "Method": "Fed-DDQN Full (mean±std)",
                "N": int(round(np.mean([row["N Eval"] for row in seed_metric_rows]))) if seed_metric_rows else 0,
                "Avg Latency": mean_std_text([row["Avg Latency"] for row in seed_metric_rows], 3),
                "SLA %": mean_std_text([row["SLA %"] for row in seed_metric_rows], 2),
                "SLA Miss %": mean_std_text([row["SLA Miss %"] for row in seed_metric_rows], 2),
                "Edge Usage %": mean_std_text([row["Edge Usage %"] for row in seed_metric_rows], 2),
                "Comm Delay (ms)": mean_std_text([row["Comm Delay (ms)"] for row in seed_metric_rows], 3),
                "BW Consump %": mean_std_text([row["BW Consump %"] for row in seed_metric_rows], 2),
            })
            for method, actions in baseline_actions_full_test.items():
                metrics = evaluate_actions(method, actions[mask], test_df.loc[mask].reset_index(drop=True), test_raw_for_suite[mask])
                scenario_rows.append({
                    "Scenario": scenario_name,
                    "Method": method,
                    "N": int(metrics["N Eval"]),
                    "Avg Latency": f"{metrics['Avg Latency']:.3f}",
                    "SLA %": f"{metrics['SLA %']:.2f}",
                    "SLA Miss %": f"{metrics['SLA Miss %']:.2f}",
                    "Edge Usage %": f"{metrics['Edge Usage %']:.2f}",
                    "Comm Delay (ms)": f"{metrics['Comm Delay (ms)']:.3f}",
                    "BW Consump %": f"{metrics['BW Consump %']:.2f}",
                })

        df_scenario_final = pd.DataFrame(scenario_rows)
        save_experiment_results({
            "suite_meta": experiment_suite_meta(),
            "cache_signature": cache_signature,
            "experiment_seeds": list(map(int, EXPERIMENT_SEEDS)),
            "ablation_names": [cfg["name"] for cfg in ABLATION_CONFIGS],
            "df_multiseed_raw": df_multiseed_raw,
            "df_ablation_summary": df_ablation_summary,
            "df_full_multiseed": df_full_multiseed,
            "df_scenario_final": df_scenario_final,
            "multiseed_records": multiseed_records,
            "full_seed_actions": full_seed_actions,
        })

    print("\n=== Multi-Seed Raw Results ===")
    print(df_multiseed_raw.to_string(index=False))
    print("\n=== Ablation Summary (mean ± std across seeds) ===")
    print(df_ablation_summary.to_string(index=False))
    print("\n=== Scenario-Wise Final Test Table ===")
    print(df_scenario_final.to_string(index=False))

    if RUN_FULL_ANALYTICS:
        ablation_plot_specs = [
            ("Avg Latency", "Ablation: Avg Latency (mean +/- std)", "Avg Latency (ms)", "#A32D2D"),
            ("SLA %", "Ablation: SLA Compliance (mean +/- std)", "SLA %", "#1D9E75"),
        ]
        for metric, title, ylabel, color in ablation_plot_specs:
            metric_means = df_multiseed_raw.groupby("Ablation", sort=False)[metric].mean()
            metric_stds = df_multiseed_raw.groupby("Ablation", sort=False)[metric].std(ddof=0).fillna(0)
            fig, ax = plt.subplots(figsize=(10.5, 5.2))
            ax.bar(metric_means.index, metric_means.values, yerr=metric_stds.values, color=color, alpha=0.85, capsize=4)
            ax.set_title(title, fontweight="bold")
            ax.set_ylabel(ylabel)
            ax.tick_params(axis="x", rotation=30)
            ax.grid(axis="y", alpha=0.22)
            plt.tight_layout()
            if RUN_PLOTS:
                plt.show()
            else:
                plt.close()
else:
    df_multiseed_raw = pd.DataFrame()
    df_ablation_summary = pd.DataFrame()
    df_full_multiseed = pd.DataFrame()
    df_scenario_final = pd.DataFrame()
    results_loaded_from_cache = False
    print("RUN_MULTI_SEED_ABLATION=False; skipped v6.6 multi-seed ablation suite.")

# ----------------------------------------------------------------------
# Optional experiment hooks: gamma-zero ablation and baseline multi-seed.
# These are disabled by default so the current cached Stage-1 results load fast.
# ----------------------------------------------------------------------
GAMMA_ZERO_CONFIG = {
    "name": "No Bootstrapping (Gamma=0)",
    "use_prioritized": PROPOSED_USE_PRIORITIZED_REPLAY,
    "mu_proximal": PROPOSED_MU_PROXIMAL,
    "soft_tau": PROPOSED_TARGET_TAU,
    "action_aware_env": PROPOSED_USE_ACTION_AWARE_ENV,
    "heavy_task_penalty": PROPOSED_HEAVY_TASK_EDGE_PENALTY,
    "validation_checkpoint": True,
    "gamma": 0.0,
    "optional_cache_group": "gamma_zero",
}


def _summarize_multiseed_table(df_raw):
    if df_raw is None or len(df_raw) == 0:
        return pd.DataFrame()
    rows = []
    for name, group in df_raw.groupby("Ablation", sort=False):
        rows.append({
            "Ablation": name,
            "Runs": int(len(group)),
            "Avg Latency": mean_std_text(group["Avg Latency"], 3),
            "SLA %": mean_std_text(group["SLA %"], 2),
            "Rejection %": mean_std_text(group["Rejection %"], 2),
            "Edge Usage %": mean_std_text(group["Edge Usage %"], 2),
            "Best Val Score": mean_std_text(group["Best Val Score"], 3) if "Best Val Score" in group else "0.000 +/- 0.000",
            "Rounds Ran": mean_std_text(group["Rounds Ran"], 1) if "Rounds Ran" in group else "0.0 +/- 0.0",
        })
    return pd.DataFrame(rows)


def _gamma_zero_results_meta():
    return {
        "cache_signature": cache_signature,
        "run_cache_version": RUN_CACHE_VERSION,
        "config_hash": _short_json_hash(GAMMA_ZERO_CONFIG),
        "seeds": list(map(int, EXPERIMENT_SEEDS)),
        "MAX_STEPS": int(MAX_STEPS),
        "ROUNDS": int(ROUNDS),
        "BATCH_SIZE": int(BATCH_SIZE),
        "gamma": 0.0,
        "terminal_masked_targets": True,
    }


def _load_gamma_zero_results():
    if not os.path.exists(gamma_zero_results_cache_path):
        return None
    cached = read_pickle_compat(gamma_zero_results_cache_path)
    if not _cache_meta_matches(cached.get("meta"), _gamma_zero_results_meta()):
        return None
    return cached


df_gamma_zero_ablation = pd.DataFrame()
df_gamma_zero_summary = pd.DataFrame()
gamma_zero_loaded_from_cache = False
gamma_zero_skipped_cache_only = False

if RUN_GAMMA_ZERO_ABLATION:
    cached_gamma_zero = None if FORCE_RETRAIN_MULTI_SEED_ABLATION else _load_gamma_zero_results()
    if cached_gamma_zero is not None:
        df_gamma_zero_ablation = cached_gamma_zero["df_gamma_zero_ablation"].copy()
        df_gamma_zero_summary = cached_gamma_zero["df_gamma_zero_summary"].copy()
        gamma_zero_loaded_from_cache = True
        print(f"Loaded cached gamma-zero ablation hook: {gamma_zero_results_cache_path}")
    elif LOAD_EXPERIMENT_RESULTS_ONLY:
        gamma_zero_skipped_cache_only = True
        print("RUN_GAMMA_ZERO_ABLATION=True but no valid cache exists; skipped because LOAD_EXPERIMENT_RESULTS_ONLY=True.")
    else:
        gamma_rows = []
        test_raw_for_gamma = features_raw[test_mask].astype(np.float32, copy=False)
        for seed in EXPERIMENT_SEEDS:
            t0_gamma = time.time()
            model_variant, payload = train_or_load_fed_variant(GAMMA_ZERO_CONFIG, seed)
            metrics, _ = evaluate_model_metrics(model_variant, X_test_sc, test_df, test_raw_for_gamma)
            gamma_rows.append({
                "Ablation": GAMMA_ZERO_CONFIG["name"],
                "Seed": int(seed),
                "Avg Latency": metrics["Avg Latency"],
                "SLA %": metrics["SLA %"],
                "SLA Miss %": metrics["SLA Miss %"],
                "Rejection %": metrics["Rejection %"],
                "Edge Usage %": metrics["Edge Usage %"],
                "Comm Delay (ms)": metrics["Comm Delay (ms)"],
                "BW Consump %": metrics["BW Consump %"],
                "Best Val Score": payload.get("best_val_score", np.nan),
                "Best Round": payload.get("best_round", 0),
                "Rounds Ran": payload.get("rounds_ran", 0),
                "Loaded Cache": bool(payload.get("loaded_from_cache", False)),
                "Runtime Sec": round(time.time() - t0_gamma, 2),
            })
            print(
                f"{GAMMA_ZERO_CONFIG['name']:<26s} seed={seed:<5d} "
                f"lat={metrics['Avg Latency']:.3f} SLA={metrics['SLA %']:.2f}% "
                f"rej={metrics['Rejection %']:.2f}% cache={payload.get('loaded_from_cache', False)}"
            )
        df_gamma_zero_ablation = pd.DataFrame(gamma_rows)
        df_gamma_zero_summary = _summarize_multiseed_table(df_gamma_zero_ablation)
        pd.to_pickle({
            "meta": _gamma_zero_results_meta(),
            "df_gamma_zero_ablation": df_gamma_zero_ablation,
            "df_gamma_zero_summary": df_gamma_zero_summary,
        }, gamma_zero_results_cache_path)
        print(f"Saved gamma-zero ablation hook cache: {gamma_zero_results_cache_path}")

    if len(df_gamma_zero_ablation):
        existing_ms = df_multiseed_raw.copy() if "df_multiseed_raw" in globals() and len(df_multiseed_raw) else pd.DataFrame()
        if len(existing_ms) and "Ablation" in existing_ms.columns:
            existing_ms = existing_ms[existing_ms["Ablation"] != GAMMA_ZERO_CONFIG["name"]]
        df_multiseed_raw = pd.concat([existing_ms, df_gamma_zero_ablation], ignore_index=True)
        df_ablation_summary = _summarize_multiseed_table(df_multiseed_raw)


def _baseline_multiseed_meta():
    return {
        "cache_signature": cache_signature,
        "run_cache_version": RUN_CACHE_VERSION,
        "methods": list(BASELINE_MULTI_SEED_METHODS),
        "seeds": list(map(int, BASELINE_MULTI_SEED_SEEDS)),
        "MAX_STEPS": int(MAX_STEPS),
        "ROUNDS": int(ROUNDS),
        "BATCH_SIZE": int(BATCH_SIZE),
        "GAMMA": float(GAMMA),
        "terminal_masked_targets": True,
    }


def _load_baseline_multiseed_results():
    if not os.path.exists(baseline_multiseed_results_cache_path):
        return None
    cached = read_pickle_compat(baseline_multiseed_results_cache_path)
    if not _cache_meta_matches(cached.get("meta"), _baseline_multiseed_meta()):
        return None
    return cached


def _train_baseline_multiseed_model(method, seed):
    set_experiment_seed(seed)
    method = str(method)
    if method == "DDQN":
        q = QNetwork(X_train.shape[1]).to(device)
        tgt = deepcopy(q).to(device)
        opt = optim.Adam(q.parameters(), lr=5e-4)
        env = OffloadEnv(train_df, X_train_sc)
        eps = 1.0
        losses = []
        for rnd in range(ROUNDS):
            buf = ReplayBuffer(8000)
            state = env.reset().to(device)
            round_loss = 0.0
            round_batches = 0
            for step in range(MAX_STEPS):
                q.eval()
                with torch.no_grad():
                    action = int(np.random.randint(0, 2)) if np.random.rand() < eps else int(torch.argmax(q(state.unsqueeze(0))).item())
                q.train()
                next_state, reward, done, _ = env.step(action)
                buf.push(state.cpu(), action, reward, next_state.cpu(), done)
                if len(buf) > BATCH_SIZE:
                    sb, ab, rb, nsb, db = buf.sample(BATCH_SIZE)
                    sb = sb.to(device); nsb = nsb.to(device); ab = ab.to(device); rb = rb.to(device); db = db.to(device)
                    current_q = q(sb).gather(1, ab.unsqueeze(1)).squeeze(1)
                    with torch.no_grad():
                        next_actions = torch.argmax(q(nsb), dim=1)
                        next_q = tgt(nsb).gather(1, next_actions.unsqueeze(1)).squeeze(1)
                        target_q = rb + GAMMA * (1.0 - db) * next_q
                    loss = nn.MSELoss()(current_q, target_q)
                    opt.zero_grad(set_to_none=True)
                    loss.backward()
                    nn.utils.clip_grad_norm_(q.parameters(), 1.0)
                    opt.step()
                    round_loss += float(loss.item())
                    round_batches += 1
                if step % TARGET_UPDATE == 0:
                    tgt.load_state_dict(q.state_dict())
                state = next_state.to(device)
                if done:
                    state = env.reset().to(device)
            eps = max(eps * EPSILON_DECAY, EPSILON_MIN)
            losses.append(round_loss / max(round_batches, 1))
        return q, {"losses": losses, "rounds_ran": len(losses)}

    if method == "FL-DDPG":
        q = QNetwork(X_train.shape[1]).to(device)
        eps = 1.0
        losses = []
        n_iid = len(zone_names)
        iid_sz = len(X_train_sc) // max(n_iid, 1)
        iid_dfs = [train_df.iloc[i * iid_sz:(i + 1) * iid_sz].reset_index(drop=True) for i in range(n_iid)]
        iid_Xs = [X_train_sc[i * iid_sz:(i + 1) * iid_sz] for i in range(n_iid)]
        iid_envs = [OffloadEnv(iid_dfs[i], iid_Xs[i]) for i in range(n_iid)]
        for rnd in range(ROUNDS):
            local_models = []
            round_loss = 0.0
            round_batches = 0
            for env in iid_envs:
                lm = deepcopy(q).to(device)
                tgt = deepcopy(lm).to(device)
                opt = optim.Adam(lm.parameters(), lr=5e-4)
                buf = ReplayBuffer(8000)
                state = env.reset().to(device)
                for step in range(MAX_STEPS):
                    lm.eval()
                    with torch.no_grad():
                        action = int(np.random.randint(0, 2)) if np.random.rand() < eps else int(torch.argmax(lm(state.unsqueeze(0))).item())
                    lm.train()
                    next_state, reward, done, _ = env.step(action)
                    buf.push(state.cpu(), action, reward, next_state.cpu(), done)
                    if len(buf) > BATCH_SIZE:
                        sb, ab, rb, nsb, db = buf.sample(BATCH_SIZE)
                        sb = sb.to(device); nsb = nsb.to(device); ab = ab.to(device); rb = rb.to(device); db = db.to(device)
                        current_q = lm(sb).gather(1, ab.unsqueeze(1)).squeeze(1)
                        with torch.no_grad():
                            next_actions = torch.argmax(lm(nsb), dim=1)
                            next_q = tgt(nsb).gather(1, next_actions.unsqueeze(1)).squeeze(1)
                            target_q = rb + GAMMA * (1.0 - db) * next_q
                        loss = nn.MSELoss()(current_q, target_q)
                        opt.zero_grad(set_to_none=True)
                        loss.backward()
                        nn.utils.clip_grad_norm_(lm.parameters(), 1.0)
                        opt.step()
                        round_loss += float(loss.item())
                        round_batches += 1
                    if step % TARGET_UPDATE == 0:
                        tgt.load_state_dict(lm.state_dict())
                    state = next_state.to(device)
                    if done:
                        state = env.reset().to(device)
                local_models.append(lm.cpu())
            q = federated_average(local_models).to(device)
            eps = max(eps * EPSILON_DECAY, EPSILON_MIN)
            losses.append(round_loss / max(round_batches, 1))
        return q, {"losses": losses, "rounds_ran": len(losses)}

    raise ValueError(f"Unsupported baseline multi-seed method: {method}")


df_baseline_multiseed_raw = pd.DataFrame()
df_baseline_multiseed_summary = pd.DataFrame()
baseline_multiseed_loaded_from_cache = False
baseline_multiseed_skipped_cache_only = False

if RUN_BASELINE_MULTI_SEED:
    cached_baseline_ms = None if FORCE_RETRAIN_BASELINE_MULTI_SEED else _load_baseline_multiseed_results()
    if cached_baseline_ms is not None:
        df_baseline_multiseed_raw = cached_baseline_ms["df_baseline_multiseed_raw"].copy()
        df_baseline_multiseed_summary = cached_baseline_ms["df_baseline_multiseed_summary"].copy()
        baseline_multiseed_loaded_from_cache = True
        print(f"Loaded cached DDQN/FL-DDPG baseline multi-seed hook: {baseline_multiseed_results_cache_path}")
    elif LOAD_EXPERIMENT_RESULTS_ONLY:
        baseline_multiseed_skipped_cache_only = True
        print("RUN_BASELINE_MULTI_SEED=True but no valid cache exists; skipped because LOAD_EXPERIMENT_RESULTS_ONLY=True.")
    else:
        baseline_rows = []
        test_raw_for_baselines = features_raw[test_mask].astype(np.float32, copy=False)
        for method in BASELINE_MULTI_SEED_METHODS:
            for seed in BASELINE_MULTI_SEED_SEEDS:
                t0_base = time.time()
                q_model, payload = _train_baseline_multiseed_model(method, seed)
                metrics, _ = evaluate_model_metrics(q_model, X_test_sc, test_df, test_raw_for_baselines)
                baseline_rows.append({
                    "Method": method,
                    "Seed": int(seed),
                    "Avg Latency": metrics["Avg Latency"],
                    "SLA %": metrics["SLA %"],
                    "SLA Miss %": metrics["SLA Miss %"],
                    "Rejection %": metrics["Rejection %"],
                    "Edge Usage %": metrics["Edge Usage %"],
                    "Comm Delay (ms)": metrics["Comm Delay (ms)"],
                    "BW Consump %": metrics["BW Consump %"],
                    "Rounds Ran": payload.get("rounds_ran", ROUNDS),
                    "Runtime Sec": round(time.time() - t0_base, 2),
                })
                print(
                    f"Baseline multi-seed {method:<7s} seed={seed:<5d} "
                    f"lat={metrics['Avg Latency']:.3f} SLA={metrics['SLA %']:.2f}% rej={metrics['Rejection %']:.2f}%"
                )
        df_baseline_multiseed_raw = pd.DataFrame(baseline_rows)
        summary_rows = []
        for method, group in df_baseline_multiseed_raw.groupby("Method", sort=False):
            summary_rows.append({
                "Method": method,
                "Runs": int(len(group)),
                "Avg Latency": mean_std_text(group["Avg Latency"], 3),
                "SLA %": mean_std_text(group["SLA %"], 2),
                "Rejection %": mean_std_text(group["Rejection %"], 2),
                "Edge Usage %": mean_std_text(group["Edge Usage %"], 2),
                "Rounds Ran": mean_std_text(group["Rounds Ran"], 1),
            })
        df_baseline_multiseed_summary = pd.DataFrame(summary_rows)
        pd.to_pickle({
            "meta": _baseline_multiseed_meta(),
            "df_baseline_multiseed_raw": df_baseline_multiseed_raw,
            "df_baseline_multiseed_summary": df_baseline_multiseed_summary,
        }, baseline_multiseed_results_cache_path)
        print(f"Saved baseline multi-seed hook cache: {baseline_multiseed_results_cache_path}")
else:
    print("RUN_BASELINE_MULTI_SEED=False; DDQN/FL-DDPG multi-seed hook is available but not run.")


## Ablation Result Figures


In [ ]:
def _plot_or_close():
    if RUN_PLOTS:
        plt.show()
    else:
        plt.close()


def _numeric_metric(series):
    def parse_value(value):
        if isinstance(value, (int, float, np.integer, np.floating)):
            return float(value)
        text = str(value).replace("±", "+/-")
        return float(text.split("+/-")[0].strip())
    return pd.Series(series).map(parse_value).astype(float)


def _std_metric(series):
    def parse_std(value):
        if isinstance(value, (int, float, np.integer, np.floating)):
            return 0.0
        text = str(value).replace("±", "+/-")
        if "+/-" not in text:
            return 0.0
        return float(text.split("+/-", 1)[1].strip())
    return pd.Series(series).map(parse_std).astype(float)


if "df_multiseed_raw" not in globals() or df_multiseed_raw.empty:
    print("Ablation visual pack skipped: multi-seed results are not available in the current session.")
else:
    print("Generating ablation visual evidence pack as separate figures ...")
    ablation_order = list(dict.fromkeys(df_multiseed_raw["Ablation"].astype(str).tolist()))
    ablation_colors = {name: ("#A32D2D" if name == "Fed-DDQN Proposed v6.6" else "#7F77DD") for name in ablation_order}
    grouped = df_multiseed_raw.groupby("Ablation", sort=False)

    metric_specs = [
        ("Avg Latency", "Ablation Mean Latency", "Avg latency (ms)", True),
        ("SLA %", "Ablation Mean SLA Compliance", "SLA compliance (%)", False),
        ("Rejection %", "Ablation Mean Rejection Rate", "Rejection rate (%)", True),
        ("Edge Usage %", "Ablation Mean Edge Usage", "Edge usage (%)", None),
        ("Best Val Score", "Ablation Best Validation Score", "Validation score (lower is better)", True),
        ("Rounds Ran", "Ablation Training Rounds Used", "Rounds", True),
    ]

    for metric, title, ylabel, lower_is_better in metric_specs:
        means = grouped[metric].mean().reindex(ablation_order)
        stds = grouped[metric].std(ddof=0).fillna(0).reindex(ablation_order)
        fig, ax = plt.subplots(figsize=(11.0, 5.4))
        bars = ax.bar(means.index, means.values, yerr=stds.values, capsize=4,
                      color=[ablation_colors[name] for name in means.index], alpha=0.88)
        if lower_is_better is not None and len(means):
            best_name = means.idxmin() if lower_is_better else means.idxmax()
            bars[list(means.index).index(best_name)].set_edgecolor("black")
            bars[list(means.index).index(best_name)].set_linewidth(2.0)
        ax.set_title(title, fontweight="bold")
        ax.set_ylabel(ylabel)
        ax.tick_params(axis="x", rotation=28)
        ax.grid(axis="y", alpha=0.22)
        fig.text(0.01, 0.01, "Red = proposed Fed-DDQN v6.6. Error bars = std across seeds.", fontsize=8)
        plt.tight_layout()
        _plot_or_close()

    proposed_rows = df_multiseed_raw[df_multiseed_raw["Ablation"] == "Fed-DDQN Proposed v6.6"]
    if len(proposed_rows):
        proposed_means = proposed_rows[["Avg Latency", "SLA %", "Rejection %", "Edge Usage %"]].mean()
        delta_rows = []
        for ablation_name, group in grouped:
            if ablation_name == "Fed-DDQN Proposed v6.6":
                continue
            means = group[["Avg Latency", "SLA %", "Rejection %", "Edge Usage %"]].mean()
            delta_rows.append({
                "Ablation": ablation_name,
                "Latency Delta": means["Avg Latency"] - proposed_means["Avg Latency"],
                "SLA Delta": means["SLA %"] - proposed_means["SLA %"],
                "Rejection Delta": means["Rejection %"] - proposed_means["Rejection %"],
                "Edge Usage Delta": means["Edge Usage %"] - proposed_means["Edge Usage %"],
            })
        df_ablation_delta = pd.DataFrame(delta_rows)
        if len(df_ablation_delta):
            for metric, ylabel, color in [
                ("Latency Delta", "Latency delta vs proposed (ms)", "#A32D2D"),
                ("SLA Delta", "SLA delta vs proposed (%)", "#1D9E75"),
                ("Rejection Delta", "Rejection delta vs proposed (%)", "#D85A30"),
                ("Edge Usage Delta", "Edge usage delta vs proposed (%)", "#7F77DD"),
            ]:
                fig, ax = plt.subplots(figsize=(10.5, 5.0))
                vals = df_ablation_delta[metric].values.astype(float)
                ax.bar(df_ablation_delta["Ablation"], vals, color=color, alpha=0.85)
                ax.axhline(0, color="black", linewidth=1)
                ax.set_title(f"Ablation Effect: {metric}", fontweight="bold")
                ax.set_ylabel(ylabel)
                ax.tick_params(axis="x", rotation=28)
                ax.grid(axis="y", alpha=0.22)
                plt.tight_layout()
                _plot_or_close()
            print("\n=== Ablation Delta vs Proposed ===")
            print(df_ablation_delta.round(3).to_string(index=False))

    for metric, ylabel in [
        ("Avg Latency", "Avg latency (ms)"),
        ("SLA %", "SLA compliance (%)"),
        ("Rejection %", "Rejection rate (%)"),
        ("Edge Usage %", "Edge usage (%)"),
    ]:
        fig, ax = plt.subplots(figsize=(10.5, 5.2))
        data = [df_multiseed_raw.loc[df_multiseed_raw["Ablation"] == name, metric].to_numpy(dtype=float) for name in ablation_order]
        ax.boxplot(data, labels=ablation_order, showmeans=True)
        ax.set_title(f"Seed Stability: {metric}", fontweight="bold")
        ax.set_ylabel(ylabel)
        ax.tick_params(axis="x", rotation=28)
        ax.grid(axis="y", alpha=0.22)
        plt.tight_layout()
        _plot_or_close()

    rank_metrics = ["Avg Latency", "SLA %", "Rejection %", "Edge Usage %", "Best Val Score"]
    rank_rows = []
    for ablation_name, group in grouped:
        row = {"Ablation": ablation_name}
        for metric in rank_metrics:
            means = grouped[metric].mean()
            ascending = metric not in ["SLA %"]
            ranks = means.rank(method="min", ascending=ascending)
            row[metric] = ranks.loc[ablation_name]
        rank_rows.append(row)
    df_ablation_rank = pd.DataFrame(rank_rows).set_index("Ablation")
    fig, ax = plt.subplots(figsize=(8.8, 5.4))
    im = ax.imshow(df_ablation_rank.values, cmap="YlGnBu_r", aspect="auto")
    ax.set_xticks(np.arange(len(df_ablation_rank.columns)))
    ax.set_xticklabels(df_ablation_rank.columns, rotation=25, ha="right")
    ax.set_yticks(np.arange(len(df_ablation_rank.index)))
    ax.set_yticklabels(df_ablation_rank.index)
    for i in range(df_ablation_rank.shape[0]):
        for j in range(df_ablation_rank.shape[1]):
            ax.text(j, i, f"{df_ablation_rank.iloc[i, j]:.0f}", ha="center", va="center", fontsize=8)
    ax.set_title("Ablation Rank Heatmap (1 = best)", fontweight="bold")
    fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    plt.tight_layout()
    _plot_or_close()
    print("Ablation visual evidence pack complete.")


## Scenario-Specific Result Figures


In [ ]:
if "df_scenario_final" not in globals() or df_scenario_final.empty:
    print("Scenario visual pack skipped: scenario results are not available in the current session.")
else:
    print("Generating scenario-wise research figures as separate outputs ...")
    scenario_plot_df = df_scenario_final.copy()
    for metric in ["Avg Latency", "SLA %", "SLA Miss %", "Edge Usage %", "Comm Delay (ms)", "BW Consump %"]:
        scenario_plot_df[metric] = _numeric_metric(scenario_plot_df[metric])

    scenario_methods = ["Fed-DDQN Full (mean±std)", "DDQN", "FL-DDPG", "MTOSA", "GTPSO", "PTS-RA", "JTOS", "Oracle"]
    scenario_methods = [method for method in scenario_methods if method in set(scenario_plot_df["Method"])]

    heatmap_specs = [
        ("Avg Latency", "Scenario-Wise Avg Latency", "Avg latency (ms)", "YlOrRd"),
        ("SLA %", "Scenario-Wise SLA Compliance", "SLA compliance (%)", "YlGn"),
        ("SLA Miss %", "Scenario-Wise SLA Miss Rate", "SLA miss rate (%)", "OrRd"),
        ("Edge Usage %", "Scenario-Wise Edge Usage", "Edge usage (%)", "PuBu"),
    ]
    for metric, title, cbar_label, cmap in heatmap_specs:
        pivot = scenario_plot_df.pivot_table(index="Scenario", columns="Method", values=metric, aggfunc="mean")
        pivot = pivot[[method for method in scenario_methods if method in pivot.columns]]
        fig_h = max(5.0, 0.45 * len(pivot.index) + 2.0)
        fig, ax = plt.subplots(figsize=(11.5, fig_h))
        im = ax.imshow(pivot.values, aspect="auto", cmap=cmap)
        ax.set_xticks(np.arange(len(pivot.columns)))
        ax.set_xticklabels(pivot.columns, rotation=30, ha="right")
        ax.set_yticks(np.arange(len(pivot.index)))
        ax.set_yticklabels(pivot.index)
        for i in range(pivot.shape[0]):
            for j in range(pivot.shape[1]):
                ax.text(j, i, f"{pivot.iloc[i, j]:.1f}", ha="center", va="center", fontsize=7)
        ax.set_title(title, fontweight="bold")
        fig.colorbar(im, ax=ax, label=cbar_label, fraction=0.046, pad=0.04)
        plt.tight_layout()
        _plot_or_close()

    proposed_name = "Fed-DDQN Full (mean±std)"
    if proposed_name in set(scenario_plot_df["Method"]):
        for baseline in ["DDQN", "FL-DDPG", "MTOSA", "GTPSO", "PTS-RA", "JTOS"]:
            if baseline not in set(scenario_plot_df["Method"]):
                continue
            prop = scenario_plot_df[scenario_plot_df["Method"] == proposed_name].set_index("Scenario")
            base = scenario_plot_df[scenario_plot_df["Method"] == baseline].set_index("Scenario")
            common = prop.index.intersection(base.index)
            if len(common) == 0:
                continue
            improvement = (base.loc[common, "Avg Latency"] - prop.loc[common, "Avg Latency"]) / np.maximum(base.loc[common, "Avg Latency"], 1e-6) * 100.0
            fig, ax = plt.subplots(figsize=(10.5, max(4.8, 0.38 * len(common) + 1.5)))
            colors = ["#1D9E75" if value >= 0 else "#D85A30" for value in improvement.values]
            ax.barh(common, improvement.values, color=colors, alpha=0.88)
            ax.axvline(0, color="black", linewidth=1)
            ax.set_title(f"Fed-DDQN Latency Improvement vs {baseline} by Scenario", fontweight="bold")
            ax.set_xlabel("Latency reduction (%)")
            ax.grid(axis="x", alpha=0.22)
            plt.tight_layout()
            _plot_or_close()

        prop = scenario_plot_df[scenario_plot_df["Method"] == proposed_name].copy()
        prop = prop.sort_values("Avg Latency", ascending=False)
        fig, ax = plt.subplots(figsize=(10.5, max(4.8, 0.38 * len(prop) + 1.5)))
        ax.barh(prop["Scenario"], prop["Avg Latency"], color="#A32D2D", alpha=0.88)
        ax.set_title("Fed-DDQN Proposed Latency by Scenario", fontweight="bold")
        ax.set_xlabel("Avg latency (ms)")
        ax.invert_yaxis()
        ax.grid(axis="x", alpha=0.22)
        plt.tight_layout()
        _plot_or_close()

    print("Scenario-wise research visual pack complete.")


## Policy Behavior Diagnostics


In [ ]:
print("Generating policy behavior diagnostics as separate figures ...")

valid_eval = (eval_sample["edge_latency"].to_numpy(dtype=np.float32) < EDGE_LAT_CAP) | (eval_sample["cloud_latency"].to_numpy(dtype=np.float32) < CLOUD_LAT_CAP)
valid_idx = np.flatnonzero(valid_eval)
valid_df = eval_sample.iloc[valid_idx].reset_index(drop=True)
valid_raw = eval_raw[valid_idx]

fed_actions_eval = actions_by_method["Fed-DDQN (Proposed)"][valid_idx]
oracle_actions_eval = actions_by_method["Oracle"][valid_idx]

decision_rows = []
for task_type in sorted(valid_df["task_type"].unique()):
    idx = np.flatnonzero(valid_df["task_type"].to_numpy() == task_type)
    if len(idx) == 0:
        continue
    decision_rows.append({
        "Task Type": task_type,
        "N": int(len(idx)),
        "Fed Edge %": float((fed_actions_eval[idx] == 0).mean() * 100.0),
        "Oracle Edge %": float((oracle_actions_eval[idx] == 0).mean() * 100.0),
        "Agreement %": float((fed_actions_eval[idx] == oracle_actions_eval[idx]).mean() * 100.0),
    })
df_policy_behavior = pd.DataFrame(decision_rows)
print("\n=== Policy Behavior by Task Type ===")
print(df_policy_behavior.round(2).to_string(index=False))

fig, ax = plt.subplots(figsize=(10.5, 5.2))
x = np.arange(len(df_policy_behavior))
w = 0.38
ax.bar(x - w/2, df_policy_behavior["Fed Edge %"], width=w, color="#A32D2D", label="Fed-DDQN edge")
ax.bar(x + w/2, df_policy_behavior["Oracle Edge %"], width=w, color="#BA7517", label="Oracle edge")
ax.set_xticks(x)
ax.set_xticklabels(df_policy_behavior["Task Type"], rotation=30)
ax.set_title("Edge Decision Share by Task Type: Fed-DDQN vs Oracle", fontweight="bold")
ax.set_ylabel("Edge decision share (%)")
ax.legend()
ax.grid(axis="y", alpha=0.22)
plt.tight_layout()
_plot_or_close()

fig, ax = plt.subplots(figsize=(10.5, 5.2))
ax.bar(df_policy_behavior["Task Type"], df_policy_behavior["Agreement %"], color="#5A4FCF", alpha=0.88)
ax.axhline(df_policy_behavior["Agreement %"].mean(), color="black", linestyle="--", label=f"Mean={df_policy_behavior['Agreement %'].mean():.1f}%")
ax.set_title("Fed-DDQN Agreement with Oracle by Task Type", fontweight="bold")
ax.set_ylabel("Agreement (%)")
ax.tick_params(axis="x", rotation=30)
ax.legend()
ax.grid(axis="y", alpha=0.22)
plt.tight_layout()
_plot_or_close()

latency_distribution_methods = ["DDQN", "FL-DDPG", "Fed-DDQN (Proposed)", "Oracle"]
latency_data = []
labels = []
lat_e_valid = valid_df["edge_latency"].to_numpy(dtype=np.float32)
lat_c_valid = valid_df["cloud_latency"].to_numpy(dtype=np.float32)
for method in latency_distribution_methods:
    if method not in actions_by_method:
        continue
    method_actions = actions_by_method[method][valid_idx]
    method_lat = np.where(method_actions == 0, lat_e_valid, lat_c_valid)
    if len(method_lat) > 4000:
        rng_box = np.random.default_rng(2026)
        method_lat = method_lat[rng_box.choice(len(method_lat), size=4000, replace=False)]
    latency_data.append(method_lat)
    labels.append(method)
fig, ax = plt.subplots(figsize=(9.5, 5.2))
ax.boxplot(latency_data, labels=labels, showmeans=True, showfliers=False)
ax.set_title("Latency Distribution by Policy", fontweight="bold")
ax.set_ylabel("Latency (ms)")
ax.tick_params(axis="x", rotation=20)
ax.grid(axis="y", alpha=0.22)
plt.tight_layout()
_plot_or_close()

sla_by_type_rows = []
fed_lat = np.where(fed_actions_eval == 0, lat_e_valid, lat_c_valid)
fed_sla = valid_df["task_type"].map(SLA_MS).fillna(9999).to_numpy(dtype=np.float32)
fed_miss = fed_lat >= fed_sla
for task_type in sorted(valid_df["task_type"].unique()):
    idx = np.flatnonzero(valid_df["task_type"].to_numpy() == task_type)
    if len(idx) == 0:
        continue
    sla_by_type_rows.append({
        "Task Type": task_type,
        "Miss Count": int(fed_miss[idx].sum()),
        "Miss Rate %": float(fed_miss[idx].mean() * 100.0),
        "Share of All Misses %": float(fed_miss[idx].sum() / max(fed_miss.sum(), 1) * 100.0),
    })
df_fed_sla_miss = pd.DataFrame(sla_by_type_rows)
fig, ax = plt.subplots(figsize=(10.5, 5.2))
ax.bar(df_fed_sla_miss["Task Type"], df_fed_sla_miss["Share of All Misses %"], color="#D85A30", alpha=0.88)
ax.set_title("Fed-DDQN SLA Miss Concentration by Task Type", fontweight="bold")
ax.set_ylabel("Share of all Fed-DDQN SLA misses (%)")
ax.tick_params(axis="x", rotation=30)
ax.grid(axis="y", alpha=0.22)
plt.tight_layout()
_plot_or_close()

stress_flags = {
    "Congestion": valid_raw[:, _F_N_CONG] >= 0.5,
    "Outage": valid_raw[:, _F_N_OUTAGE] >= 0.5,
    "Jitter": valid_raw[:, _F_N_JITTER] >= 0.5,
    "High edge queue": valid_raw[:, _F_E_QUEUE] >= np.percentile(valid_raw[:, _F_E_QUEUE], 90),
    "Normal": (valid_raw[:, _F_N_CONG] < 0.5) & (valid_raw[:, _F_N_OUTAGE] < 0.5) & (valid_raw[:, _F_N_JITTER] < 0.5),
}
stress_rows = []
for stress_name, stress_mask in stress_flags.items():
    if int(stress_mask.sum()) == 0:
        continue
    sub_df = valid_df.loc[stress_mask].reset_index(drop=True)
    sub_raw = valid_raw[stress_mask]
    sub_actions = fed_actions_eval[stress_mask]
    res = evaluate_actions("Fed-DDQN (Proposed)", sub_actions, sub_df, sub_raw)
    stress_rows.append({"Stress": stress_name, **res})
df_policy_stress = pd.DataFrame(stress_rows)
for metric, ylabel, color in [
    ("Avg Latency", "Avg latency (ms)", "#A32D2D"),
    ("SLA %", "SLA compliance (%)", "#1D9E75"),
    ("Rejection %", "Rejection rate (%)", "#D85A30"),
    ("Edge Usage %", "Edge usage (%)", "#7F77DD"),
]:
    fig, ax = plt.subplots(figsize=(8.5, 5.0))
    ax.bar(df_policy_stress["Stress"], df_policy_stress[metric], color=color, alpha=0.88)
    ax.set_title(f"Fed-DDQN Under Network/Queue Stress: {metric}", fontweight="bold")
    ax.set_ylabel(ylabel)
    ax.tick_params(axis="x", rotation=25)
    ax.grid(axis="y", alpha=0.22)
    plt.tight_layout()
    _plot_or_close()

print("Policy behavior diagnostics complete.")


## Deadline and Deployment Stress Diagnostics


In [ ]:
print("Generating deadline and deployment stress figures as separate outputs ...")

valid_eval = (eval_sample["edge_latency"].to_numpy(dtype=np.float32) < EDGE_LAT_CAP) | (eval_sample["cloud_latency"].to_numpy(dtype=np.float32) < CLOUD_LAT_CAP)
valid_idx = np.flatnonzero(valid_eval)
valid_df = eval_sample.iloc[valid_idx].reset_index(drop=True)
valid_raw = eval_raw[valid_idx]
fed_actions_eval = actions_by_method["Fed-DDQN (Proposed)"][valid_idx]
lat_e_valid = valid_df["edge_latency"].to_numpy(dtype=np.float32)
lat_c_valid = valid_df["cloud_latency"].to_numpy(dtype=np.float32)
fed_lat = np.where(fed_actions_eval == 0, lat_e_valid, lat_c_valid)

deadline_values = valid_df["deadline_ms"].to_numpy(dtype=np.float32)
deadline_bins = pd.qcut(deadline_values, q=5, duplicates="drop")
deadline_rows = []
for interval in deadline_bins.categories:
    mask = np.asarray(deadline_bins == interval)
    if int(mask.sum()) == 0:
        continue
    sla = valid_df.loc[mask, "task_type"].map(SLA_MS).fillna(9999).to_numpy(dtype=np.float32)
    deadline_rows.append({
        "Deadline Bin": str(interval),
        "N": int(mask.sum()),
        "Avg Latency": float(fed_lat[mask].mean()),
        "SLA %": float((fed_lat[mask] < sla).mean() * 100.0),
        "Edge Usage %": float((fed_actions_eval[mask] == 0).mean() * 100.0),
    })
df_deadline_sensitivity = pd.DataFrame(deadline_rows)
for metric, ylabel, color in [
    ("Avg Latency", "Avg latency (ms)", "#A32D2D"),
    ("SLA %", "SLA compliance (%)", "#1D9E75"),
    ("Edge Usage %", "Edge usage (%)", "#7F77DD"),
]:
    fig, ax = plt.subplots(figsize=(10.5, 5.2))
    ax.plot(df_deadline_sensitivity["Deadline Bin"], df_deadline_sensitivity[metric], marker="o", color=color, linewidth=2)
    ax.set_title(f"Fed-DDQN Deadline Sensitivity: {metric}", fontweight="bold")
    ax.set_ylabel(ylabel)
    ax.set_xlabel("Deadline quantile bin")
    ax.tick_params(axis="x", rotation=25)
    ax.grid(alpha=0.25)
    plt.tight_layout()
    _plot_or_close()

snr_values = valid_raw[:, _F_N_SNR]
snr_bins = pd.qcut(snr_values, q=5, duplicates="drop")
snr_rows = []
for interval in snr_bins.categories:
    mask = np.asarray(snr_bins == interval)
    if int(mask.sum()) == 0:
        continue
    sub_df = valid_df.loc[mask].reset_index(drop=True)
    sub_raw = valid_raw[mask]
    sub_actions = fed_actions_eval[mask]
    res = evaluate_actions("Fed-DDQN (Proposed)", sub_actions, sub_df, sub_raw)
    snr_rows.append({"SNR Bin": str(interval), **res})
df_snr_sensitivity = pd.DataFrame(snr_rows)
for metric, ylabel, color in [
    ("Avg Latency", "Avg latency (ms)", "#A32D2D"),
    ("SLA %", "SLA compliance (%)", "#1D9E75"),
    ("Comm Delay (ms)", "Communication delay (ms)", "#5A4FCF"),
    ("Edge Usage %", "Edge usage (%)", "#7F77DD"),
]:
    fig, ax = plt.subplots(figsize=(10.5, 5.2))
    ax.plot(df_snr_sensitivity["SNR Bin"], df_snr_sensitivity[metric], marker="s", color=color, linewidth=2)
    ax.set_title(f"Fed-DDQN Channel Sensitivity by SNR: {metric}", fontweight="bold")
    ax.set_ylabel(ylabel)
    ax.set_xlabel("SNR quantile bin")
    ax.tick_params(axis="x", rotation=25)
    ax.grid(alpha=0.25)
    plt.tight_layout()
    _plot_or_close()

arrival_counts = eval_sample.groupby("arrival_time").size().sort_index()
fig, ax = plt.subplots(figsize=(12.0, 4.8))
ax.plot(arrival_counts.index, arrival_counts.values, color="#378ADD", linewidth=1.6)
ax.set_title("Test-Split Arrival Burst Profile", fontweight="bold")
ax.set_xlabel("Arrival timestep")
ax.set_ylabel("Tasks arriving")
ax.grid(alpha=0.25)
plt.tight_layout()
_plot_or_close()

arrival_map = arrival_counts.to_dict()
valid_load = valid_df["arrival_time"].map(arrival_map).fillna(0).to_numpy(dtype=np.float32)
load_bins = pd.qcut(valid_load, q=5, duplicates="drop")
load_rows = []
for interval in load_bins.categories:
    mask = np.asarray(load_bins == interval)
    if int(mask.sum()) == 0:
        continue
    sub_df = valid_df.loc[mask].reset_index(drop=True)
    sub_raw = valid_raw[mask]
    sub_actions = fed_actions_eval[mask]
    res = evaluate_actions("Fed-DDQN (Proposed)", sub_actions, sub_df, sub_raw)
    load_rows.append({"Arrival Load Bin": str(interval), **res})
df_arrival_load_sensitivity = pd.DataFrame(load_rows)
for metric, ylabel, color in [
    ("Avg Latency", "Avg latency (ms)", "#A32D2D"),
    ("SLA %", "SLA compliance (%)", "#1D9E75"),
    ("Rejection %", "Rejection rate (%)", "#D85A30"),
    ("Edge Usage %", "Edge usage (%)", "#7F77DD"),
]:
    fig, ax = plt.subplots(figsize=(10.5, 5.2))
    ax.plot(df_arrival_load_sensitivity["Arrival Load Bin"], df_arrival_load_sensitivity[metric], marker="D", color=color, linewidth=2)
    ax.set_title(f"Fed-DDQN Sensitivity to Arrival Burst Load: {metric}", fontweight="bold")
    ax.set_ylabel(ylabel)
    ax.set_xlabel("Per-timestep arrival-load quantile")
    ax.tick_params(axis="x", rotation=25)
    ax.grid(alpha=0.25)
    plt.tight_layout()
    _plot_or_close()

print("Deadline and deployment stress figures complete.")


## Resource Allocation Target Construction


In [ ]:
if "read_pickle_compat" not in globals():
    def install_numpy_pickle_compat():
        """Allow old/new NumPy pickle module paths to load across NumPy versions."""
        import importlib
        import sys

        aliases = {
            "numpy._core": "numpy.core",
            "numpy._core.numeric": "numpy.core.numeric",
            "numpy._core.multiarray": "numpy.core.multiarray",
            "numpy._core.umath": "numpy.core.umath",
            "numpy._core._multiarray_umath": "numpy.core._multiarray_umath",
            "numpy._core.numerictypes": "numpy.core.numerictypes",
            "numpy._core.fromnumeric": "numpy.core.fromnumeric",
            "numpy._core.shape_base": "numpy.core.shape_base",
            "numpy._core.function_base": "numpy.core.function_base",
        }
        for missing_name, available_name in aliases.items():
            if missing_name in sys.modules:
                continue
            try:
                sys.modules[missing_name] = importlib.import_module(available_name)
            except ModuleNotFoundError:
                pass

    def read_pickle_compat(path):
        install_numpy_pickle_compat()
        return pd.read_pickle(path)

install_numpy_pickle_compat()

ALLOC_TASK_PROFILES = {
    "sensor": (0.85, 0.80, 0.75),
    "telemetry": (0.95, 0.85, 0.85),
    "voice": (1.05, 0.90, 1.20),
    "image": (1.15, 1.25, 1.10),
    "video": (1.20, 1.15, 1.35),
    "ai": (1.35, 1.30, 1.10),
    "firmware_update": (1.05, 1.10, 1.45),
    "emergency": (1.35, 1.05, 1.35),
}


def _task_urgency(task_row):
    task_type = str(task_row.get("task_type", "sensor"))
    deadline = float(task_row.get("deadline_ms", SLA_MS.get(task_type, 1000.0)))
    sla = float(SLA_MS.get(task_type, deadline))
    priority = float(task_row.get("priority_level", 1.0))
    priority_term = np.clip((priority - 1.0) / 4.0, 0.0, 1.0)
    deadline_term = np.clip((sla - deadline) / max(sla, 1.0), 0.0, 1.0)
    realtime_term = 0.35 * float(task_row.get("is_real_time", 0))
    emergency_term = 0.45 if task_type == "emergency" else 0.0
    return float(np.clip(0.30 * priority_term + 0.45 * deadline_term + realtime_term + emergency_term, 0.0, 1.0))


def gen_alloc_target(task_row):
    """Return QoS-aware (cpu_share, mem_share, bw_share) in [0.01, 1.0]."""
    t = int(task_row["arrival_time"])
    edge_id = int(task_row["assigned_edge_id"])
    edge = edge_idx.get((t, edge_id))
    net = net_idx.get(t)
    if edge is None or getattr(edge, "is_failed", 0) == 1:
        return [0.5, 0.5, 0.5]

    task_type = str(task_row.get("task_type", "sensor"))
    cpu_profile, mem_profile, bw_profile = ALLOC_TASK_PROFILES.get(task_type, (1.0, 1.0, 1.0))
    urgency = _task_urgency(task_row)
    queue_pressure = float(np.clip(getattr(edge, "edge_queue_length", 0.0) / 80.0, 0.0, 1.0))
    edge_degraded = float(getattr(edge, "is_degrading", 0))
    edge_isolated = float(getattr(edge, "is_isolated", 0))

    if net is not None:
        snr_fac = float(np.clip(getattr(net, "snr_db", 25) / 25.0, 0.1, 1.5))
        packet_loss = float(getattr(net, "packet_loss_rate", 0.05))
        eff_bw = max(float(net.uplink_bandwidth) * (1.0 - packet_loss) * snr_fac, 1.0)
        net_stress = float(
            np.clip(
                0.45 * getattr(net, "is_congestion", 0)
                + 0.55 * getattr(net, "is_jitter_storm", 0)
                + 0.80 * getattr(net, "is_outage", 0)
                + packet_loss * 2.0,
                0.0,
                1.0,
            )
        )
    else:
        eff_bw = 100.0
        net_stress = 0.0

    cpu_base = float(task_row["cpu_cycles"]) / max(float(edge.edge_cpu_available), 1e-3)
    mem_base = float(task_row["memory_req_mb"]) / max(float(edge.edge_memory_available), 1e-3)
    bw_base = float(task_row["task_size_mb"]) / eff_bw

    priority_boost = 1.0 + 0.35 * urgency
    queue_boost = 1.0 + 0.25 * queue_pressure
    reliability_boost = 1.0 + 0.12 * edge_degraded + 0.08 * edge_isolated
    network_boost = 1.0 + 0.30 * net_stress
    low_battery_factor = 0.92 if int(task_row.get("is_low_battery", 0)) == 1 else 1.0
    dependency_boost = 1.08 if int(task_row.get("has_dependency", 0)) == 1 else 1.0

    cpu_s = cpu_base * cpu_profile * priority_boost * queue_boost * reliability_boost * low_battery_factor
    mem_s = mem_base * mem_profile * (1.0 + 0.20 * urgency) * reliability_boost * dependency_boost
    bw_s = bw_base * bw_profile * priority_boost * network_boost * dependency_boost

    if task_type in {"ai", "video", "image", "firmware_update"}:
        cpu_s *= 1.08
        mem_s *= 1.10
    if task_type in {"emergency", "voice"}:
        bw_s *= 1.12

    return [
        float(np.clip(cpu_s, 0.01, 1.0)),
        float(np.clip(mem_s, 0.01, 1.0)),
        float(np.clip(bw_s, 0.01, 1.0)),
    ]


# Stage-2 ResourceAllocator cache setup. This cell can be rerun from here onward.
import os, json, hashlib

ALLOCATOR_VERSION = "v66_rejection_residual_allocator_2026_05_30_cachefix"
USE_ALLOCATOR_CACHES = True
FORCE_REBUILD_ALLOCATOR_TARGETS = False
FORCE_RETRAIN_ALLOCATOR = False
FORCE_REBUILD_EVALUATION_CACHE = globals().get("FORCE_REBUILD_EVALUATION_CACHE", False)
CACHE_VERSION = globals().get("CACHE_VERSION", "v66_rebalanced_fed_ddqn_2026_05_22")
RUN_CACHE_VERSION = globals().get("RUN_CACHE_VERSION", "v66_full_run_cache_2026_05_29")
base_path = globals().get("base_path", r"C:\Users\mayan\Desktop\3STSEM\ai\model\dataset3")
cache_dir = globals().get("cache_dir", os.path.join(base_path, "_v64_cache"))
os.makedirs(cache_dir, exist_ok=True)

if "_safe_mtime" not in globals():
    def _safe_mtime(path):
        try:
            return os.path.getmtime(path) if os.path.exists(path) else None
        except OSError:
            return None

if "_short_array_hash" not in globals():
    def _short_array_hash(arr):
        arr = np.ascontiguousarray(np.asarray(arr))
        return hashlib.sha256(arr.view(np.uint8)).hexdigest()[:16]

if "_cache_meta_matches" not in globals():
    def _cache_meta_matches(cached_meta, expected_meta):
        return dict(cached_meta or {}) == dict(expected_meta or {})

if "cache_signature" not in globals():
    _DATASET_FILES = [
        ("tasks", "dataset_A.csv", tasks),
        ("edge_nodes", "edge_nodes.csv", edge_nodes),
        ("edge_state", "edge_state.csv", edge_state),
        ("cloud_nodes", "cloud_nodes.csv", cloud_nodes),
        ("cloud_state", "cloud_state.csv", cloud_state),
        ("network_state", "network_state.csv", network_state),
    ]

    def _build_cache_signature():
        payload = {"cache_version": CACHE_VERSION, "files": {}, "schemas": {}, "rows": {}}
        for name, filename, df in _DATASET_FILES:
            path = os.path.join(base_path, filename)
            stat = os.stat(path)
            payload["files"][filename] = {"size": stat.st_size, "mtime_ns": stat.st_mtime_ns}
            payload["schemas"][name] = list(df.columns)
            payload["rows"][name] = int(len(df))
        raw = json.dumps(payload, sort_keys=True).encode("utf-8")
        return hashlib.sha256(raw).hexdigest()

    cache_signature = _build_cache_signature()

cache_prefix = f"v64_{cache_signature[:16]}"
allocator_cache_prefix = f"{cache_prefix}_{ALLOCATOR_VERSION}"
alloc_targets_cache_path = os.path.join(cache_dir, f"{allocator_cache_prefix}_alloc_targets.npz")
allocator_model_path = os.path.join(cache_dir, f"{allocator_cache_prefix}_allocator.pt")
allocator_stage2_cache_path = os.path.join(cache_dir, f"{allocator_cache_prefix}_stage2_allocations.pkl")
allocator_eval_cache_path = os.path.join(cache_dir, f"{allocator_cache_prefix}_allocator_eval.pkl")
allocator_ablation_cache_path = os.path.join(cache_dir, f"{allocator_cache_prefix}_allocator_ablation.pkl")
policy_eval_cache_path = globals().get("policy_eval_cache_path", os.path.join(cache_dir, f"{cache_prefix}_policy_eval.pkl"))

alloc_targets_loaded_from_cache = False
if USE_ALLOCATOR_CACHES and (not FORCE_REBUILD_ALLOCATOR_TARGETS) and os.path.exists(alloc_targets_cache_path):
    try:
        cache = np.load(alloc_targets_cache_path, allow_pickle=True)
        alloc_targets = cache["alloc_targets"].astype(np.float32, copy=False)
        if alloc_targets.shape != (len(tasks), 3):
            raise ValueError(f"allocator target shape mismatch: {alloc_targets.shape}")
        cached_version = str(cache["allocator_version"][0]) if "allocator_version" in cache.files else ""
        if cached_version and cached_version != ALLOCATOR_VERSION:
            raise ValueError(f"allocator target version mismatch: {cached_version}")
        alloc_targets_loaded_from_cache = True
        print(f"Loaded cached QoS-aware allocation targets: {alloc_targets_cache_path}")
    except Exception as exc:
        print(f"Allocation target cache ignored and rebuilt: {exc}")

if not alloc_targets_loaded_from_cache:
    print("Generating QoS-aware allocation targets (CPU, Memory, Bandwidth) ...")
    alloc_targets = np.array(tasks.apply(gen_alloc_target, axis=1).tolist(), dtype=np.float32)
    if USE_ALLOCATOR_CACHES:
        np.savez_compressed(
            alloc_targets_cache_path,
            alloc_targets=alloc_targets,
            allocator_version=np.array([ALLOCATOR_VERSION], dtype=object),
        )
        print(f"Saved QoS-aware allocation target cache: {alloc_targets_cache_path}")

print(f"Shape={alloc_targets.shape}")
print(f"CPU: mean={alloc_targets[:,0].mean():.3f}  Mem: mean={alloc_targets[:,1].mean():.3f}  BW: mean={alloc_targets[:,2].mean():.3f}")
print(f"Target p90 CPU={np.percentile(alloc_targets[:,0],90):.3f}  Mem={np.percentile(alloc_targets[:,1],90):.3f}  BW={np.percentile(alloc_targets[:,2],90):.3f}")
assert alloc_targets.shape[1] == 3, "Expected 3 allocation outputs"


## Resource Allocator Training


In [ ]:

edge_mask = tasks["offload_label"] == 0
edge_train_df = tasks.loc[edge_mask].reset_index(drop=True)
X_alloc_raw = features_raw[edge_mask]
raw_alloc_targets = np.clip(alloc_targets[edge_mask], 0.01, 1.0).astype(np.float32)

ALLOCATOR_RESIDUAL_RANGE = 0.35
ALLOCATOR_EPS = 1e-6


def allocator_priority_weights(task_df):
    task_type = task_df["task_type"].astype(str)
    priority = task_df["priority_level"].astype(float).to_numpy()
    deadline = np.maximum(task_df["deadline_ms"].astype(float).to_numpy(), 1.0)
    deadline_weight = 1.0 / np.sqrt(deadline / np.median(deadline))
    realtime = task_df.get("is_real_time", pd.Series(0, index=task_df.index)).to_numpy(dtype=float)
    emergency = task_type.eq("emergency").to_numpy(dtype=float)
    heavy = task_type.isin(["ai", "video", "image", "firmware_update"]).to_numpy(dtype=float)
    low_battery = task_df.get("is_low_battery", pd.Series(0, index=task_df.index)).to_numpy(dtype=float)
    weights = 1.0 + 0.30 * priority + 0.35 * realtime + 0.75 * emergency + 0.20 * heavy + 0.12 * low_battery
    weights *= np.clip(deadline_weight, 0.70, 1.70)
    return np.clip(weights.astype(np.float32), 0.10, 6.00)


def normalize_allocations_by_edge_timestep(alloc, task_df, min_share=0.0, priority_aware=True):
    alloc_norm = np.clip(np.asarray(alloc, dtype=np.float32).copy(), 0.0, 1.0)
    if len(alloc_norm) == 0:
        return alloc_norm
    groups = task_df[["arrival_time", "assigned_edge_id"]].copy().reset_index(drop=True)
    priority_weights = allocator_priority_weights(task_df.reset_index(drop=True)) if priority_aware else np.ones(len(task_df), dtype=np.float32)
    for _, idx_values in groups.groupby(["arrival_time", "assigned_edge_id"], sort=False).groups.items():
        idx = np.asarray(list(idx_values), dtype=int)
        if len(idx) == 0:
            continue
        weights = priority_weights[idx]
        for resource_i in range(3):
            total = float(alloc_norm[idx, resource_i].sum())
            if total <= 1.0:
                continue
            weighted = alloc_norm[idx, resource_i] * weights
            denom = float(weighted.sum())
            if denom <= 1e-8:
                alloc_norm[idx, resource_i] = 1.0 / len(idx)
            else:
                alloc_norm[idx, resource_i] = weighted / denom
            if min_share > 0.0:
                alloc_norm[idx, resource_i] = np.maximum(alloc_norm[idx, resource_i], min_share)
                total_after_floor = float(alloc_norm[idx, resource_i].sum())
                if total_after_floor > 1.0:
                    alloc_norm[idx, resource_i] /= total_after_floor
    return np.clip(alloc_norm, 0.0, 1.0)


def allocator_raw_demand_scores(raw_np):
    raw_np = np.asarray(raw_np, dtype=np.float32)
    return np.stack([
        raw_np[:, _F_CPU] / np.maximum(raw_np[:, _F_E_CPU], 1.0),
        raw_np[:, _F_MEM] / np.maximum(raw_np[:, _F_E_MEM], 1.0),
        raw_np[:, _F_TASK_SIZE] / np.maximum(raw_np[:, _F_EFF_BW], 1.0),
    ], axis=1).astype(np.float32)


def demand_proportional_allocation(task_df, raw_np):
    demand = np.clip(allocator_raw_demand_scores(raw_np), 0.01, 1.0)
    return normalize_allocations_by_edge_timestep(demand, task_df, priority_aware=False)


def priority_weighted_allocation(task_df):
    out = np.zeros((len(task_df), 3), dtype=np.float32)
    if len(task_df) == 0:
        return out
    groups = task_df[["arrival_time", "assigned_edge_id"]].reset_index(drop=True)
    weights = allocator_priority_weights(task_df.reset_index(drop=True))
    for _, idx_values in groups.groupby(["arrival_time", "assigned_edge_id"], sort=False).groups.items():
        idx = np.asarray(list(idx_values), dtype=int)
        denom = float(weights[idx].sum())
        share = np.ones(len(idx), dtype=np.float32) / max(len(idx), 1) if denom <= 1e-8 else weights[idx] / denom
        out[idx, :] = share[:, None]
    return np.clip(out, 0.0, 1.0)


def deadline_aware_allocation(task_df):
    deadline = np.maximum(task_df["deadline_ms"].to_numpy(dtype=np.float32), 1.0)
    weights = allocator_priority_weights(task_df) * np.clip(np.median(deadline) / deadline, 0.45, 2.80)
    out = np.zeros((len(task_df), 3), dtype=np.float32)
    groups = task_df[["arrival_time", "assigned_edge_id"]].reset_index(drop=True)
    for _, idx_values in groups.groupby(["arrival_time", "assigned_edge_id"], sort=False).groups.items():
        idx = np.asarray(list(idx_values), dtype=int)
        denom = float(weights[idx].sum())
        share = np.ones(len(idx), dtype=np.float32) / max(len(idx), 1) if denom <= 1e-8 else weights[idx] / denom
        out[idx, :] = share[:, None]
    return np.clip(out, 0.0, 1.0)


def estimate_allocator_risk_features(task_df, raw_np):
    task_type = task_df["task_type"].astype(str)
    deadline = np.maximum(task_df["deadline_ms"].to_numpy(dtype=np.float32), 1.0)
    sla = task_type.map(SLA_MS).fillna(9999).to_numpy(dtype=np.float32)
    edge_lat = task_df["edge_latency"].to_numpy(dtype=np.float32)
    cloud_lat = task_df["cloud_latency"].to_numpy(dtype=np.float32)
    priority = task_df["priority_level"].to_numpy(dtype=np.float32)
    realtime = task_df.get("is_real_time", pd.Series(0, index=task_df.index)).to_numpy(dtype=np.float32)
    low_battery = task_df.get("is_low_battery", pd.Series(0, index=task_df.index)).to_numpy(dtype=np.float32)
    impossible = task_df.get("impossible_deadline", pd.Series(0, index=task_df.index)).to_numpy(dtype=np.float32)
    emergency = task_type.eq("emergency").to_numpy(dtype=np.float32)
    heavy = task_type.isin(["ai", "video", "image", "firmware_update"]).to_numpy(dtype=np.float32)
    deadline_tightness = np.clip((sla - deadline) / np.maximum(sla, 1.0), 0.0, 1.0)
    edge_sla_risk = np.clip((edge_lat - 0.85 * deadline) / np.maximum(0.35 * deadline, 1.0), 0.0, 1.0)
    cloud_advantage = np.clip((edge_lat - cloud_lat) / EDGE_LAT_CAP, -1.0, 1.0)
    priority_norm = np.clip((priority - 1.0) / 4.0, 0.0, 1.0)
    urgency = np.clip(
        0.35 * priority_norm + 0.35 * deadline_tightness + 0.35 * realtime
        + 0.45 * emergency + 0.20 * low_battery + 0.60 * impossible,
        0.0,
        2.0,
    )
    risk_weight = 1.0 + 1.25 * edge_sla_risk + 0.75 * urgency + 0.65 * impossible + 0.30 * heavy
    return {
        "deadline_tightness": deadline_tightness.astype(np.float32),
        "edge_sla_risk": edge_sla_risk.astype(np.float32),
        "cloud_advantage": cloud_advantage.astype(np.float32),
        "priority_norm": priority_norm.astype(np.float32),
        "urgency": urgency.astype(np.float32),
        "risk_weight": np.clip(risk_weight, 0.50, 5.00).astype(np.float32),
        "emergency": emergency.astype(np.float32),
        "heavy": heavy.astype(np.float32),
        "realtime": realtime.astype(np.float32),
        "low_battery": low_battery.astype(np.float32),
        "impossible": impossible.astype(np.float32),
    }


def build_allocator_input_matrix(task_df, raw_np, base_alloc):
    task_df = task_df.reset_index(drop=True)
    raw_np = np.asarray(raw_np, dtype=np.float32)
    base_alloc = np.asarray(base_alloc, dtype=np.float32)
    demand_scores = np.clip(allocator_raw_demand_scores(raw_np), 0.0, 3.0)
    risk = estimate_allocator_risk_features(task_df, raw_np)

    group_df = task_df[["arrival_time", "assigned_edge_id"]].copy().reset_index(drop=True)
    demand_df = pd.DataFrame(demand_scores, columns=["d_cpu", "d_mem", "d_bw"])
    demand_df[["arrival_time", "assigned_edge_id"]] = group_df
    grouped = demand_df.groupby(["arrival_time", "assigned_edge_id"], sort=False)
    group_size = grouped["d_cpu"].transform("count").to_numpy(dtype=np.float32)
    group_sum = grouped[["d_cpu", "d_mem", "d_bw"]].transform("sum").to_numpy(dtype=np.float32)

    priority_rank = task_df["priority_level"].groupby([group_df["arrival_time"], group_df["assigned_edge_id"]]).rank(method="average", ascending=False).to_numpy(dtype=np.float32)
    deadline_rank = task_df["deadline_ms"].groupby([group_df["arrival_time"], group_df["assigned_edge_id"]]).rank(method="average", ascending=True).to_numpy(dtype=np.float32)
    rank_denom = np.maximum(group_size, 1.0)
    priority_rank_pct = 1.0 - (priority_rank - 1.0) / rank_denom
    deadline_rank_pct = 1.0 - (deadline_rank - 1.0) / rank_denom

    aux = np.column_stack([
        base_alloc,
        demand_scores,
        np.clip(group_sum, 0.0, 5.0),
        np.clip(group_size / 20.0, 0.0, 5.0),
        priority_rank_pct,
        deadline_rank_pct,
        risk["deadline_tightness"],
        risk["edge_sla_risk"],
        risk["cloud_advantage"],
        risk["priority_norm"],
        risk["urgency"],
        risk["emergency"],
        risk["heavy"],
        risk["realtime"],
        risk["low_battery"],
        risk["impossible"],
        np.clip(raw_np[:, _F_E_QUEUE] / 100.0, 0.0, 5.0),
        np.clip(raw_np[:, _F_N_LOSS], 0.0, 1.0),
        np.clip(raw_np[:, _F_N_CONG], 0.0, 1.0),
        np.clip(raw_np[:, _F_N_OUTAGE], 0.0, 1.0),
        np.clip(raw_np[:, _F_N_JITTER], 0.0, 1.0),
    ]).astype(np.float32)
    return np.concatenate([raw_np.astype(np.float32, copy=False), aux], axis=1).astype(np.float32)


base_alloc_np = demand_proportional_allocation(edge_train_df, X_alloc_raw)
y_alloc_np = normalize_allocations_by_edge_timestep(raw_alloc_targets, edge_train_df, priority_aware=True)
allocator_input_np = build_allocator_input_matrix(edge_train_df, X_alloc_raw, base_alloc_np)
allocator_risk_np = estimate_allocator_risk_features(edge_train_df, X_alloc_raw)["risk_weight"]

scaler_alloc = StandardScaler()
X_alloc_sc = np.clip(scaler_alloc.fit_transform(allocator_input_np), -10, 10).astype(np.float32)
X_alloc = torch.tensor(X_alloc_sc, dtype=torch.float32)
base_alloc_t = torch.tensor(base_alloc_np, dtype=torch.float32)
y_alloc = torch.tensor(y_alloc_np, dtype=torch.float32)
allocator_risk_t = torch.tensor(allocator_risk_np, dtype=torch.float32)
print(f"Edge allocation samples: {X_alloc.shape[0]:,}  input_dim={X_alloc.shape[1]}  targets shape: {y_alloc.shape}")
print(f"Residual target MAE from base demand: {np.abs(base_alloc_np - y_alloc_np).mean():.4f}")


class ResidualResourceAllocator(nn.Module):
    """
    Rejection-aware residual allocator.
    The model predicts bounded corrections over a demand-proportional baseline.
    """
    def __init__(self, inp, residual_range=ALLOCATOR_RESIDUAL_RANGE):
        super().__init__()
        self.residual_range = float(residual_range)
        self.trunk = nn.Sequential(
            nn.Linear(inp, 160), nn.ReLU(), nn.Dropout(0.15),
            nn.Linear(160, 96), nn.ReLU(), nn.Dropout(0.10),
            nn.Linear(96, 48), nn.ReLU(),
            nn.Linear(48, 3),
            nn.Tanh(),
        )

    def forward(self, x, base_alloc=None, return_residual=False):
        residual = self.trunk(x) * self.residual_range
        if base_alloc is None:
            return residual
        alloc = torch.clamp(base_alloc + residual, 0.0, 1.0)
        return (alloc, residual) if return_residual else alloc


def allocator_training_loss(pred_alloc, pred_residual, base_alloc, target, risk_weight):
    target_residual = torch.clamp(target - base_alloc, -ALLOCATOR_RESIDUAL_RANGE, ALLOCATOR_RESIDUAL_RANGE)
    risk = risk_weight.view(-1, 1).clamp(0.5, 5.0)
    alloc_loss = nn.functional.smooth_l1_loss(pred_alloc, target)
    residual_loss = nn.functional.smooth_l1_loss(pred_residual, target_residual)
    under = torch.relu(target - pred_alloc)
    waste = torch.relu(pred_alloc - target)
    under_loss = torch.mean(risk * under.square())
    waste_loss = torch.mean(waste.square())
    return 0.55 * alloc_loss + 0.35 * residual_loss + 0.75 * under_loss + 0.04 * waste_loss


def allocator_predict_np(task_df, raw_np, base_alloc=None):
    if base_alloc is None:
        base_alloc = demand_proportional_allocation(task_df, raw_np)
    alloc_input = build_allocator_input_matrix(task_df.reset_index(drop=True), raw_np, base_alloc)
    alloc_sc = np.clip(scaler_alloc.transform(alloc_input), -10, 10).astype(np.float32)
    allocator.eval()
    with torch.inference_mode():
        pred = allocator(
            torch.tensor(alloc_sc, dtype=torch.float32, device=device),
            torch.tensor(base_alloc, dtype=torch.float32, device=device),
        ).cpu().numpy().astype(np.float32)
    return np.clip(pred, 0.0, 1.0)


ResourceAllocator = ResidualResourceAllocator
allocator = ResourceAllocator(X_alloc.shape[1]).to(device)
alloc_losses = []
ALLOC_EPOCHS = 35
allocator_loaded_from_cache = False

allocator_cache_meta = {
    "cache_signature": cache_signature,
    "run_cache_version": RUN_CACHE_VERSION,
    "allocator_version": ALLOCATOR_VERSION,
    "architecture": "demand_proportional_residual",
    "input_dim": int(X_alloc.shape[1]),
    "n_train": int(len(X_alloc)),
    "residual_range": float(ALLOCATOR_RESIDUAL_RANGE),
    "target": "capacity_feasible_qos_target",
}

if USE_ALLOCATOR_CACHES and (not FORCE_RETRAIN_ALLOCATOR) and os.path.exists(allocator_model_path):
    try:
        payload = torch.load(allocator_model_path, map_location=device)
        if payload.get("meta") != allocator_cache_meta:
            raise ValueError("residual allocator cache metadata mismatch")
        allocator.load_state_dict(payload["state_dict"])
        alloc_losses = list(payload.get("alloc_losses", []))
        allocator_loaded_from_cache = True
        print(f"Loaded cached rejection-aware residual ResourceAllocator: {allocator_model_path}")
    except Exception as exc:
        print(f"Allocator cache ignored and retrained: {exc}")

if not allocator_loaded_from_cache:
    opt_alloc = optim.AdamW(allocator.parameters(), lr=8e-4, weight_decay=8e-5)
    sch_alloc = optim.lr_scheduler.CosineAnnealingLR(opt_alloc, T_max=ALLOC_EPOCHS)
    alloc_loader = DataLoader(
        TensorDataset(X_alloc, base_alloc_t, y_alloc, allocator_risk_t),
        batch_size=512,
        shuffle=True,
        pin_memory=PIN_MEMORY,
    )
    print("Training rejection-aware residual ResourceAllocator ...")
    for ep in range(ALLOC_EPOCHS):
        allocator.train()
        total = 0.0
        for xb, baseb, yb, riskb in alloc_loader:
            xb = xb.to(device)
            baseb = baseb.to(device)
            yb = yb.to(device)
            riskb = riskb.to(device)
            opt_alloc.zero_grad(set_to_none=True)
            pred_alloc, pred_residual = allocator(xb, baseb, return_residual=True)
            loss = allocator_training_loss(pred_alloc, pred_residual, baseb, yb, riskb)
            loss.backward()
            nn.utils.clip_grad_norm_(allocator.parameters(), 1.0)
            opt_alloc.step()
            total += float(loss.item())
        sch_alloc.step()
        avg = total / len(alloc_loader)
        alloc_losses.append(avg)
        if (ep + 1) % 5 == 0 or ep == 0:
            print(f"  Epoch {ep+1:2d}/{ALLOC_EPOCHS}  loss={avg:.4f}")
    if USE_ALLOCATOR_CACHES:
        torch.save({
            "meta": allocator_cache_meta,
            "state_dict": allocator.state_dict(),
            "alloc_losses": alloc_losses,
            "ALLOC_EPOCHS": ALLOC_EPOCHS,
            "allocator_version": ALLOCATOR_VERSION,
        }, allocator_model_path)
        print(f"Saved rejection-aware residual ResourceAllocator cache: {allocator_model_path}")

if RUN_FULL_ANALYTICS and RUN_PLOTS and alloc_losses:
    plt.figure(figsize=(7,4))
    plt.plot(alloc_losses, marker="o", color="purple")
    plt.title("Rejection-Aware Residual Allocator Loss")
    plt.xlabel("Epoch")
    plt.ylabel("Composite loss")
    plt.tight_layout()
    plt.show()
else:
    print("Allocator loss plot skipped or no loss history available.")


## Resource Allocator Evaluation


In [ ]:

preds_alloc_np = allocator_predict_np(edge_train_df, X_alloc_raw, base_alloc_np)
preds_alloc = torch.tensor(preds_alloc_np, dtype=torch.float32)

errors = np.abs(preds_alloc_np - y_alloc_np)
base_errors = np.abs(base_alloc_np - y_alloc_np)
print("[Stage 2] Rejection-aware residual allocator MAE:")
print(f"  Base Demand Overall MAE={base_errors.mean():.4f}")
print(f"  Residual CPU MAE={errors[:,0].mean():.4f}  "
      f"Mem MAE={errors[:,1].mean():.4f}  "
      f"BW MAE={errors[:,2].mean():.4f}  "
      f"Overall={errors.mean():.4f}")

fig, ax = plt.subplots(figsize=(7.5, 5.0))
ax.hist(errors.flatten(), bins=50, color="steelblue", alpha=0.9)
ax.set_title("Residual Resource Allocator Absolute Error Distribution", fontweight="bold")
ax.set_xlabel("Absolute Error")
ax.set_ylabel("Frequency")
ax.grid(axis="y", alpha=0.22)
plt.tight_layout()
if RUN_PLOTS:
    plt.show()
else:
    plt.close()

scatter_n = min(8000, len(errors))
scatter_idx = np.linspace(0, len(errors) - 1, scatter_n, dtype=int) if len(errors) else np.array([], dtype=int)
for col_i, (name, clr) in enumerate([("CPU", "steelblue"), ("Memory", "darkorange"), ("Bandwidth", "green")]):
    fig, ax = plt.subplots(figsize=(6.2, 5.4))
    ax.scatter(y_alloc[scatter_idx, col_i].numpy(), preds_alloc[scatter_idx, col_i].numpy(), alpha=0.18, s=5, color=clr)
    ax.plot([0, 1], [0, 1], "r--", linewidth=1.2)
    ax.set_title(f"Residual Allocator: {name} Target vs Predicted", fontweight="bold")
    ax.set_xlabel("Capacity-feasible target share")
    ax.set_ylabel("Predicted share")
    ax.grid(alpha=0.22)
    plt.tight_layout()
    if RUN_PLOTS:
        plt.show()
    else:
        plt.close()
print("[CHART] Residual allocator evaluation figures exported separately.")


## Allocator Baselines and Capacity Normalization


In [ ]:
if "read_pickle_compat" not in globals():
    def install_numpy_pickle_compat():
        """Allow old/new NumPy pickle module paths to load across NumPy versions."""
        import importlib
        import sys

        aliases = {
            "numpy._core": "numpy.core",
            "numpy._core.numeric": "numpy.core.numeric",
            "numpy._core.multiarray": "numpy.core.multiarray",
            "numpy._core.umath": "numpy.core.umath",
            "numpy._core._multiarray_umath": "numpy.core._multiarray_umath",
            "numpy._core.numerictypes": "numpy.core.numerictypes",
            "numpy._core.fromnumeric": "numpy.core.fromnumeric",
            "numpy._core.shape_base": "numpy.core.shape_base",
            "numpy._core.function_base": "numpy.core.function_base",
        }
        for missing_name, available_name in aliases.items():
            if missing_name in sys.modules:
                continue
            try:
                sys.modules[missing_name] = importlib.import_module(available_name)
            except ModuleNotFoundError:
                pass

    def read_pickle_compat(path):
        install_numpy_pickle_compat()
        return pd.read_pickle(path)

install_numpy_pickle_compat()


def _group_equal_alloc(task_df):
    out = np.zeros((len(task_df), 3), dtype=np.float32)
    groups = task_df[["arrival_time", "assigned_edge_id"]].reset_index(drop=True)
    for _, idx_values in groups.groupby(["arrival_time", "assigned_edge_id"], sort=False).groups.items():
        idx = np.asarray(list(idx_values), dtype=int)
        out[idx, :] = 1.0 / max(len(idx), 1)
    return out


def rejection_aware_demand_allocation(task_df, raw_np):
    demand = np.clip(allocator_raw_demand_scores(raw_np), 0.01, 1.0)
    risk = estimate_allocator_risk_features(task_df.reset_index(drop=True), raw_np)
    weights = np.clip(1.0 + 0.55 * risk["edge_sla_risk"] + 0.35 * risk["urgency"], 0.5, 3.0)
    return normalize_allocations_by_edge_timestep(demand * weights[:, None], task_df, priority_aware=True)


def capacity_violation_rate(alloc, task_df):
    alloc = np.asarray(alloc, dtype=np.float32)
    if len(alloc) == 0:
        return 0.0
    tmp = task_df[["arrival_time", "assigned_edge_id"]].copy().reset_index(drop=True)
    tmp["cpu_alloc"] = alloc[:, 0]
    tmp["mem_alloc"] = alloc[:, 1]
    tmp["bw_alloc"] = alloc[:, 2]
    grouped = tmp.groupby(["arrival_time", "assigned_edge_id"])[["cpu_alloc", "mem_alloc", "bw_alloc"]].sum()
    violated = (grouped > 1.0 + 1e-6).any(axis=1)
    return float(violated.mean() * 100.0) if len(violated) else 0.0


def build_allocator_baselines(task_df, raw_np):
    baselines = {}
    baselines["Equal Allocation"] = _group_equal_alloc(task_df)
    baselines["Demand-Proportional"] = demand_proportional_allocation(task_df, raw_np)
    baselines["Priority-Weighted"] = priority_weighted_allocation(task_df)
    baselines["Deadline-Aware"] = deadline_aware_allocation(task_df)
    baselines["Rejection-Aware Demand"] = rejection_aware_demand_allocation(task_df, raw_np)
    return baselines


valid_eval_for_alloc = (eval_sample["edge_latency"].to_numpy(dtype=np.float32) < EDGE_LAT_CAP) | (eval_sample["cloud_latency"].to_numpy(dtype=np.float32) < CLOUD_LAT_CAP)
fed_eval_actions = actions_by_method["Fed-DDQN (Proposed)"]
stage2_mask = valid_eval_for_alloc & (fed_eval_actions == 0)
stage2_eval_df = eval_sample.loc[stage2_mask].reset_index(drop=True)
stage2_eval_raw = eval_raw[stage2_mask]

def allocator_stage2_cache_meta(stage2_mask):
    fed_actions = np.asarray(actions_by_method["Fed-DDQN (Proposed)"], dtype=np.int64)
    return {
        "cache_signature": cache_signature,
        "run_cache_version": RUN_CACHE_VERSION,
        "allocator_version": ALLOCATOR_VERSION,
        "allocator_architecture": "demand_proportional_residual",
        "n_eval": int(len(eval_sample)),
        "stage2_mask_hash": _short_array_hash(np.asarray(stage2_mask, dtype=np.bool_)),
        "fed_actions_hash": _short_array_hash(fed_actions),
        "allocator_model_mtime": _safe_mtime(allocator_model_path),
        "risk_projection_learned_weight": 0.10,
        "policy_eval_mtime": _safe_mtime(policy_eval_cache_path),
    }


stage2_meta = allocator_stage2_cache_meta(stage2_mask)
allocator_stage2_loaded_from_cache = False

if USE_ALLOCATOR_CACHES and (not FORCE_REBUILD_EVALUATION_CACHE) and (not FORCE_RETRAIN_ALLOCATOR) and os.path.exists(allocator_stage2_cache_path):
    try:
        cached = read_pickle_compat(allocator_stage2_cache_path)
        if not _cache_meta_matches(cached.get("meta"), stage2_meta):
            raise ValueError("allocator stage-2 cache metadata mismatch")
        stage2_targets_raw = cached["stage2_targets_raw"].astype(np.float32, copy=False)
        stage2_targets_feasible = cached["stage2_targets_feasible"].astype(np.float32, copy=False)
        stage2_base_alloc = cached["stage2_base_alloc"].astype(np.float32, copy=False)
        residual_raw = cached["residual_raw"].astype(np.float32, copy=False)
        residual_norm = cached["residual_norm"].astype(np.float32, copy=False)
        residual_risk_projection = cached["residual_risk_projection"].astype(np.float32, copy=False)
        allocator_method_allocations = {k: np.asarray(v, dtype=np.float32) for k, v in cached["allocator_method_allocations"].items()}
        allocator_stage2_loaded_from_cache = True
        print(f"Loaded cached residual stage-2 allocator allocations: {allocator_stage2_cache_path}")
    except Exception as exc:
        print(f"Allocator stage-2 cache ignored and rebuilt: {exc}")

print(f"Stage-2 allocator evaluation tasks selected by Fed-DDQN for edge: {len(stage2_eval_df):,}")

if len(stage2_eval_df) == 0:
    allocator_method_allocations = {}
    df_allocator_compare = pd.DataFrame()
    stage2_targets_raw = np.empty((0, 3), dtype=np.float32)
    stage2_targets_feasible = np.empty((0, 3), dtype=np.float32)
    stage2_base_alloc = np.empty((0, 3), dtype=np.float32)
    residual_raw = np.empty((0, 3), dtype=np.float32)
    residual_norm = np.empty((0, 3), dtype=np.float32)
    residual_risk_projection = np.empty((0, 3), dtype=np.float32)
    print("No Fed-DDQN edge-selected tasks available for allocator evaluation.")
elif not allocator_stage2_loaded_from_cache:
    task_pos = stage2_eval_df["task_id"].map(pd.Series(np.arange(len(tasks)), index=tasks["task_id"]))
    if task_pos.isna().any():
        raise ValueError("Could not map some stage-2 task_id values back to the tasks table.")
    stage2_targets_raw = np.clip(alloc_targets[task_pos.to_numpy(dtype=int)], 0.01, 1.0).astype(np.float32)
    stage2_targets_feasible = normalize_allocations_by_edge_timestep(stage2_targets_raw, stage2_eval_df, priority_aware=True)
    stage2_base_alloc = demand_proportional_allocation(stage2_eval_df, stage2_eval_raw)

    residual_raw = allocator_predict_np(stage2_eval_df, stage2_eval_raw, stage2_base_alloc)
    residual_norm = normalize_allocations_by_edge_timestep(residual_raw, stage2_eval_df, priority_aware=True)

    allocator_method_allocations = build_allocator_baselines(stage2_eval_df, stage2_eval_raw)
    rejection_aware_demand = allocator_method_allocations["Rejection-Aware Demand"]
    residual_risk_projection = normalize_allocations_by_edge_timestep(
        0.10 * residual_norm + 0.90 * rejection_aware_demand,
        stage2_eval_df,
        priority_aware=True,
    )
    allocator_method_allocations["Residual Allocator Raw"] = np.clip(residual_raw, 0.0, 1.0)
    allocator_method_allocations["Residual + Capacity Norm"] = residual_norm
    allocator_method_allocations["Residual + Risk Projection"] = residual_risk_projection

    if USE_ALLOCATOR_CACHES:
        pd.to_pickle({
            "meta": stage2_meta,
            "stage2_targets_raw": stage2_targets_raw,
            "stage2_targets_feasible": stage2_targets_feasible,
            "stage2_base_alloc": stage2_base_alloc,
            "residual_raw": residual_raw,
            "residual_norm": residual_norm,
            "residual_risk_projection": residual_risk_projection,
            "allocator_method_allocations": allocator_method_allocations,
        }, allocator_stage2_cache_path)
        print(f"Saved residual stage-2 allocator allocation cache: {allocator_stage2_cache_path}")

if len(stage2_eval_df) > 0:
    print("Allocator methods ready:")
    for name, values in allocator_method_allocations.items():
        print(f"  {name:<28} shape={values.shape} cap_violation={capacity_violation_rate(values, stage2_eval_df):.2f}%")
print(f"Stage-2 allocator allocation cache loaded: {allocator_stage2_loaded_from_cache}")


## Allocator QoS Comparison


In [ ]:
if "read_pickle_compat" not in globals():
    def install_numpy_pickle_compat():
        """Allow old/new NumPy pickle module paths to load across NumPy versions."""
        import importlib
        import sys

        aliases = {
            "numpy._core": "numpy.core",
            "numpy._core.numeric": "numpy.core.numeric",
            "numpy._core.multiarray": "numpy.core.multiarray",
            "numpy._core.umath": "numpy.core.umath",
            "numpy._core._multiarray_umath": "numpy.core._multiarray_umath",
            "numpy._core.numerictypes": "numpy.core.numerictypes",
            "numpy._core.fromnumeric": "numpy.core.fromnumeric",
            "numpy._core.shape_base": "numpy.core.shape_base",
            "numpy._core.function_base": "numpy.core.function_base",
        }
        for missing_name, available_name in aliases.items():
            if missing_name in sys.modules:
                continue
            try:
                sys.modules[missing_name] = importlib.import_module(available_name)
            except ModuleNotFoundError:
                pass

    def read_pickle_compat(path):
        install_numpy_pickle_compat()
        return pd.read_pickle(path)

install_numpy_pickle_compat()

def estimate_post_allocation_latency(task_df, alloc, target):
    alloc = np.asarray(alloc, dtype=np.float32)
    target = np.maximum(np.asarray(target, dtype=np.float32), 1e-4)
    edge_lat = task_df["edge_latency"].to_numpy(dtype=np.float32)
    deficits = np.maximum(target - alloc, 0.0) / target
    surplus = np.maximum(alloc - target, 0.0)
    penalty = 1.0 + 0.55 * deficits[:, 0] + 0.40 * deficits[:, 1] + 0.35 * deficits[:, 2]
    useful_surplus = np.clip(0.04 * surplus[:, 0] + 0.025 * surplus[:, 1] + 0.035 * surplus[:, 2], 0.0, 0.10)
    return np.clip(edge_lat * penalty * (1.0 - useful_surplus), 0.0, EDGE_LAT_CAP)


def evaluate_allocator_method(method_name, alloc, task_df, target):
    alloc = np.clip(np.asarray(alloc, dtype=np.float32), 0.0, 1.0)
    target = np.clip(np.asarray(target, dtype=np.float32), 1e-4, 1.0)
    abs_err = np.abs(alloc - target)
    under_mask = alloc < (target * 0.85)
    waste = np.maximum(alloc - target, 0.0)
    under_gap = np.maximum(target - alloc, 0.0)
    post_lat = estimate_post_allocation_latency(task_df, alloc, target)
    sla = task_df["task_type"].map(SLA_MS).fillna(9999).to_numpy(dtype=np.float32)
    task_type = task_df["task_type"].astype(str)
    emergency_mask = task_type.eq("emergency").to_numpy()
    heavy_mask = task_type.isin(["ai", "video", "image", "firmware_update"]).to_numpy()
    cap_violation = capacity_violation_rate(alloc, task_df)
    under_pct = float(under_mask.mean() * 100.0)
    waste_pct = float((waste > 0.05).mean() * 100.0)
    sla_risk = float((post_lat >= sla).mean() * 100.0)
    efficiency = 100.0 - under_pct - 0.40 * waste_pct - 0.60 * cap_violation - 0.50 * sla_risk
    return {
        "Allocator Method": method_name,
        "N": int(len(task_df)),
        "CPU MAE": round(float(abs_err[:, 0].mean()), 4),
        "Memory MAE": round(float(abs_err[:, 1].mean()), 4),
        "Bandwidth MAE": round(float(abs_err[:, 2].mean()), 4),
        "Overall MAE": round(float(abs_err.mean()), 4),
        "Under-allocation %": round(under_pct, 2),
        "Waste %": round(waste_pct, 2),
        "Capacity Violation %": round(cap_violation, 2),
        "Estimated Latency": round(float(post_lat.mean()), 3),
        "Estimated SLA Risk %": round(sla_risk, 2),
        "Emergency Underalloc %": round(float(under_mask[emergency_mask].mean() * 100.0), 2) if emergency_mask.any() else 0.0,
        "Heavy Underalloc %": round(float(under_mask[heavy_mask].mean() * 100.0), 2) if heavy_mask.any() else 0.0,
        "Avg Under Gap": round(float(under_gap.mean()), 4),
        "Avg Waste Gap": round(float(waste.mean()), 4),
        "Resource Efficiency Score": round(float(efficiency), 3),
    }


allocator_compare_loaded_from_cache = False
allocator_compare_meta = dict(stage2_meta)
allocator_compare_meta.update({"method_names": list(allocator_method_allocations.keys())})

if len(stage2_eval_df) == 0:
    df_allocator_compare = pd.DataFrame()
    allocator_best_method = None
else:
    if USE_ALLOCATOR_CACHES and (not FORCE_REBUILD_EVALUATION_CACHE) and os.path.exists(allocator_eval_cache_path):
        try:
            cached = read_pickle_compat(allocator_eval_cache_path)
            if not _cache_meta_matches(cached.get("meta"), allocator_compare_meta):
                raise ValueError("allocator comparison cache metadata mismatch")
            df_allocator_compare = cached["df_allocator_compare"].copy()
            allocator_compare_loaded_from_cache = True
            print(f"Loaded cached allocator method comparison: {allocator_eval_cache_path}")
        except Exception as exc:
            print(f"Allocator comparison cache ignored and rebuilt: {exc}")

    if not allocator_compare_loaded_from_cache:
        allocator_compare_rows = [
            evaluate_allocator_method(name, alloc, stage2_eval_df, stage2_targets_feasible)
            for name, alloc in allocator_method_allocations.items()
        ]
        df_allocator_compare = pd.DataFrame(allocator_compare_rows)
        if USE_ALLOCATOR_CACHES:
            pd.to_pickle({
                "meta": allocator_compare_meta,
                "allocator_version": ALLOCATOR_VERSION,
                "df_allocator_compare": df_allocator_compare,
                "stage2_n": len(stage2_eval_df),
            }, allocator_eval_cache_path)
            print(f"Saved allocator method comparison cache: {allocator_eval_cache_path}")

    allocator_best_method = df_allocator_compare.sort_values(
        ["Resource Efficiency Score", "Overall MAE"],
        ascending=[False, True],
    ).iloc[0]["Allocator Method"]
    print("=== Allocator Method Comparison ===")
    print(df_allocator_compare.to_string(index=False))
    print(f"\nBest allocator by efficiency score: {allocator_best_method}")
    print(f"Allocator comparison cache loaded: {allocator_compare_loaded_from_cache}")


## Allocator Ablation Analysis


In [ ]:
if "read_pickle_compat" not in globals():
    def install_numpy_pickle_compat():
        """Allow old/new NumPy pickle module paths to load across NumPy versions."""
        import importlib
        import sys

        aliases = {
            "numpy._core": "numpy.core",
            "numpy._core.numeric": "numpy.core.numeric",
            "numpy._core.multiarray": "numpy.core.multiarray",
            "numpy._core.umath": "numpy.core.umath",
            "numpy._core._multiarray_umath": "numpy.core._multiarray_umath",
            "numpy._core.numerictypes": "numpy.core.numerictypes",
            "numpy._core.fromnumeric": "numpy.core.fromnumeric",
            "numpy._core.shape_base": "numpy.core.shape_base",
            "numpy._core.function_base": "numpy.core.function_base",
        }
        for missing_name, available_name in aliases.items():
            if missing_name in sys.modules:
                continue
            try:
                sys.modules[missing_name] = importlib.import_module(available_name)
            except ModuleNotFoundError:
                pass

    def read_pickle_compat(path):
        install_numpy_pickle_compat()
        return pd.read_pickle(path)

install_numpy_pickle_compat()

allocator_ablation_loaded_from_cache = False
ALLOCATOR_ABLATION_NAMES = [
    "Full Residual + Risk Projection",
    "Residual + Capacity Norm",
    "Base Demand Only",
    "Residual Raw",
    "Priority-Weighted",
    "Deadline-Aware",
    "Rejection-Aware Demand",
    "Equal Allocation",
]
allocator_ablation_meta = dict(stage2_meta)
allocator_ablation_meta.update({
    "run_allocator_ablation": bool(RUN_ALLOCATOR_ABLATION),
    "stage2_n": int(len(stage2_eval_df)),
    "ablation_names": ALLOCATOR_ABLATION_NAMES,
})

if not RUN_ALLOCATOR_ABLATION or len(stage2_eval_df) == 0:
    df_allocator_ablation = pd.DataFrame()
    print("Allocator ablation skipped.")
else:
    if USE_ALLOCATOR_CACHES and (not FORCE_REBUILD_EVALUATION_CACHE) and os.path.exists(allocator_ablation_cache_path):
        try:
            cached = read_pickle_compat(allocator_ablation_cache_path)
            if not _cache_meta_matches(cached.get("meta"), allocator_ablation_meta):
                raise ValueError("allocator ablation cache metadata mismatch")
            df_allocator_ablation = cached["df_allocator_ablation"].copy()
            allocator_ablation_loaded_from_cache = True
            print(f"Loaded cached residual allocator ablation table: {allocator_ablation_cache_path}")
        except Exception as exc:
            print(f"Allocator ablation cache ignored and rebuilt: {exc}")

    if not allocator_ablation_loaded_from_cache:
        ablation_allocs = {
            "Full Residual + Risk Projection": allocator_method_allocations["Residual + Risk Projection"],
            "Residual + Capacity Norm": allocator_method_allocations["Residual + Capacity Norm"],
            "Base Demand Only": allocator_method_allocations["Demand-Proportional"],
            "Residual Raw": allocator_method_allocations["Residual Allocator Raw"],
            "Priority-Weighted": allocator_method_allocations["Priority-Weighted"],
            "Deadline-Aware": allocator_method_allocations["Deadline-Aware"],
            "Rejection-Aware Demand": allocator_method_allocations["Rejection-Aware Demand"],
            "Equal Allocation": allocator_method_allocations["Equal Allocation"],
        }
        df_allocator_ablation = pd.DataFrame([
            evaluate_allocator_method(name, alloc, stage2_eval_df, stage2_targets_feasible)
            for name, alloc in ablation_allocs.items()
        ])
        if USE_ALLOCATOR_CACHES:
            pd.to_pickle({
                "meta": allocator_ablation_meta,
                "df_allocator_ablation": df_allocator_ablation,
            }, allocator_ablation_cache_path)
            print(f"Saved residual allocator ablation cache: {allocator_ablation_cache_path}")

    print("=== Residual Allocator Ablation Results ===")
    print(df_allocator_ablation.to_string(index=False))
    print(f"Allocator ablation cache loaded: {allocator_ablation_loaded_from_cache}")


## Allocator Visualization Pack


In [ ]:
if len(stage2_eval_df) == 0 or df_allocator_compare.empty:
    print("Allocator research figures skipped because allocator comparison table is empty.")
else:
    allocator_plot_specs = [
        ("Overall MAE", "Allocator Overall MAE", "Overall MAE", "#378ADD", True),
        ("Under-allocation %", "Allocator Under-allocation Rate", "Under-allocation (%)", "#D85A30", True),
        ("Waste %", "Allocator Resource Waste Rate", "Waste (%)", "#7F77DD", True),
        ("Capacity Violation %", "Allocator Capacity Violation Rate", "Capacity violation (%)", "#A32D2D", True),
        ("Estimated SLA Risk %", "Allocator Estimated SLA Risk", "Estimated SLA risk (%)", "#D85A30", True),
        ("Resource Efficiency Score", "Allocator Resource Efficiency Score", "Higher is better", "#1D9E75", False),
    ]
    for metric, title, ylabel, color, lower_better in allocator_plot_specs:
        fig, ax = plt.subplots(figsize=(11.0, 5.4))
        vals = df_allocator_compare[metric].to_numpy(dtype=float)
        bars = ax.bar(df_allocator_compare["Allocator Method"], vals, color=color, alpha=0.88)
        best_idx = int(np.argmin(vals) if lower_better else np.argmax(vals))
        bars[best_idx].set_edgecolor("black")
        bars[best_idx].set_linewidth(2.2)
        ax.set_title(title, fontweight="bold")
        ax.set_ylabel(ylabel)
        ax.tick_params(axis="x", rotation=28)
        ax.grid(axis="y", alpha=0.22)
        plt.tight_layout()
        if RUN_PLOTS:
            plt.show()
        else:
            plt.close()

    if "df_allocator_ablation" in globals() and len(df_allocator_ablation):
        for metric, title, color, lower_better in [
            ("Overall MAE", "Allocator Ablation: MAE", "#378ADD", True),
            ("Estimated SLA Risk %", "Allocator Ablation: SLA Risk", "#D85A30", True),
            ("Capacity Violation %", "Allocator Ablation: Capacity Violation", "#A32D2D", True),
            ("Resource Efficiency Score", "Allocator Ablation: Efficiency Score", "#1D9E75", False),
        ]:
            fig, ax = plt.subplots(figsize=(10.5, 5.2))
            vals = df_allocator_ablation[metric].to_numpy(dtype=float)
            bars = ax.bar(df_allocator_ablation["Allocator Method"], vals, color=color, alpha=0.88)
            best_idx = int(np.argmin(vals) if lower_better else np.argmax(vals))
            bars[best_idx].set_edgecolor("black")
            bars[best_idx].set_linewidth(2.2)
            ax.set_title(title, fontweight="bold")
            ax.set_ylabel(metric)
            ax.tick_params(axis="x", rotation=28)
            ax.grid(axis="y", alpha=0.22)
            plt.tight_layout()
            if RUN_PLOTS:
                plt.show()
            else:
                plt.close()

    best_alloc = allocator_method_allocations.get("Residual + Risk Projection")
    if best_alloc is not None:
        component_names = ["CPU", "Memory", "Bandwidth"]
        for resource_i, resource_name in enumerate(component_names):
            fig, ax = plt.subplots(figsize=(6.2, 5.4))
            scatter_n = min(8000, len(best_alloc))
            scatter_idx = np.linspace(0, len(best_alloc) - 1, scatter_n, dtype=int)
            ax.scatter(stage2_targets_feasible[scatter_idx, resource_i], best_alloc[scatter_idx, resource_i], alpha=0.18, s=5)
            ax.plot([0, 1], [0, 1], "r--", linewidth=1.2)
            ax.set_title(f"Residual QoS Allocator {resource_name}: Target vs Assigned", fontweight="bold")
            ax.set_xlabel("Capacity-feasible target")
            ax.set_ylabel("Assigned share")
            ax.grid(alpha=0.22)
            plt.tight_layout()
            if RUN_PLOTS:
                plt.show()
            else:
                plt.close()

        task_type_rows = []
        for task_type in sorted(stage2_eval_df["task_type"].unique()):
            mask = stage2_eval_df["task_type"].to_numpy() == task_type
            if int(mask.sum()) == 0:
                continue
            metric = evaluate_allocator_method("Residual + Risk Projection", best_alloc[mask], stage2_eval_df.loc[mask].reset_index(drop=True), stage2_targets_feasible[mask])
            metric["Task Type"] = task_type
            task_type_rows.append(metric)
        df_allocator_by_task_type = pd.DataFrame(task_type_rows)
        for metric, ylabel, color in [
            ("Overall MAE", "Overall MAE", "#378ADD"),
            ("Under-allocation %", "Under-allocation (%)", "#D85A30"),
            ("Estimated SLA Risk %", "Estimated SLA risk (%)", "#A32D2D"),
        ]:
            fig, ax = plt.subplots(figsize=(10.5, 5.2))
            ax.bar(df_allocator_by_task_type["Task Type"], df_allocator_by_task_type[metric], color=color, alpha=0.88)
            ax.set_title(f"Residual QoS Allocator by Task Type: {metric}", fontweight="bold")
            ax.set_ylabel(ylabel)
            ax.tick_params(axis="x", rotation=30)
            ax.grid(axis="y", alpha=0.22)
            plt.tight_layout()
            if RUN_PLOTS:
                plt.show()
            else:
                plt.close()

    print("Allocator research figures complete.")


## Edge Case Analysis


In [ ]:
print("=" * 70); print("EDGE CASE ANALYSIS"); print("=" * 70)

def safe_col(df, col):
    return df[col] if col in df.columns else pd.Series(0, index=df.index)

cases = {
    "Emergency tasks"     : tasks["task_type"] == "emergency",
    "Firmware update"     : tasks["task_type"] == "firmware_update",
    "Corrupt tasks"       : safe_col(tasks,"is_corrupt") == 1,
    "Low-battery tasks"   : safe_col(tasks,"is_low_battery") == 1,
    "Impossible deadline" : safe_col(tasks,"impossible_deadline") == 1,
    "Has dependency"      : safe_col(tasks,"has_dependency") == 1,
    "Encrypted tasks"     : safe_col(tasks,"is_encrypted") == 1,
    "Real-time tasks"     : safe_col(tasks,"is_real_time") == 1,
}
rows_out = []
for label, mask in cases.items():
    sub = tasks[mask]
    if len(sub) == 0: continue
    valid = sub[(sub["edge_latency"]<EDGE_LAT_CAP)|(sub["cloud_latency"]<CLOUD_LAT_CAP)]
    rows_out.append({
        "Case":        label,
        "Count":       len(sub),
        "Edge%":       f"{(sub['offload_label']==0).mean()*100:.1f}",
        "Cloud%":      f"{(sub['offload_label']==1).mean()*100:.1f}",
        "Rej_flag%":   f"{sub['rejection_flag'].mean()*100:.1f}",
        "SLA_viol%":   f"{sub['sla_violated'].mean()*100:.1f}",
        "AvgEdgeLat":  f"{valid['edge_latency'].mean():.1f}"  if len(valid) else "N/A",
        "AvgCloudLat": f"{valid['cloud_latency'].mean():.1f}" if len(valid) else "N/A",
    })
print(pd.DataFrame(rows_out).to_string(index=False))

print("\n-- Network Event Impact on Rejection Rate (rejection_flag) --")
for col in ["is_outage","is_congestion","is_jitter_storm"]:
    if col not in network_state.columns: continue
    ets   = set(network_state.loc[network_state[col]==1,"timestep"])
    in_ev = tasks["arrival_time"].isin(ets)
    print(f"  {col:<25}: during={tasks.loc[in_ev,'rejection_flag'].mean():.3f}  "
          f"outside={tasks.loc[~in_ev,'rejection_flag'].mean():.3f}")


## Complete System Dashboard


In [ ]:
print("Generating complete system figure pack as separate save-friendly figures ...")

fig, ax = plt.subplots(figsize=(8.4, 5.0))
ax.plot(federated_losses, marker="o", color="#A32D2D", label="Fed-DDQN")
ax.plot(cent_losses, marker="^", color="#1D9E75", label="Cent-DDQN", linestyle="-.")
ax.plot(local_losses, marker="s", color="#888780", label="SZ-DDQN", linestyle="--")
ax.set_title("DDQN Loss per Round", fontweight="bold")
ax.set_xlabel("Round")
ax.set_ylabel("MSE / TD loss")
ax.legend(fontsize=8)
ax.grid(alpha=0.25)
plt.tight_layout()
if RUN_PLOTS: plt.show()
else: plt.close()

fig, ax = plt.subplots(figsize=(8.4, 5.0))
ax.plot(federated_rewards, marker="s", color="green")
ax.axhline(0, color="gray", linestyle="--", lw=0.8)
ax.set_title("Fed-DDQN Reward per Round", fontweight="bold")
ax.set_xlabel("Round")
ax.set_ylabel("Average reward")
ax.grid(alpha=0.25)
plt.tight_layout()
if RUN_PLOTS: plt.show()
else: plt.close()

fig, ax = plt.subplots(figsize=(8.4, 5.0))
ax.plot(tr_losses, marker="^", color="darkorange")
ax.set_title("OffloadNet Training Loss", fontweight="bold")
ax.set_xlabel("Epoch")
ax.set_ylabel("Loss")
ax.grid(alpha=0.25)
plt.tight_layout()
if RUN_PLOTS: plt.show()
else: plt.close()

fig, ax = plt.subplots(figsize=(8.4, 5.0))
ax.plot(alloc_losses, marker="D", color="purple")
ax.set_title("Resource Allocator Loss", fontweight="bold")
ax.set_xlabel("Epoch")
ax.set_ylabel("MSE")
ax.grid(alpha=0.25)
plt.tight_layout()
if RUN_PLOTS: plt.show()
else: plt.close()

rej_rate = tasks["rejection_flag"].mean()
fig, ax = plt.subplots(figsize=(6.4, 4.8))
ax.bar(["Accepted", "Rejected"], [1 - rej_rate, rej_rate], color=["green", "red"])
ax.set_title("Dataset Rejection Rate", fontweight="bold")
ax.set_ylim(0, 1)
ax.set_ylabel("Ratio")
for i, value in enumerate([1 - rej_rate, rej_rate]):
    ax.text(i, value + 0.02, f"{value*100:.1f}%", ha="center", fontsize=9)
plt.tight_layout()
if RUN_PLOTS: plt.show()
else: plt.close()

methods_short = df_compare.index.tolist()
method_colors = [METHOD_COLORS.get(m, "#888780") if "METHOD_COLORS" in globals() else "#888780" for m in methods_short]
fig, ax = plt.subplots(figsize=(10.5, 5.2))
ax.bar(methods_short, df_compare["Avg Latency"].values, color=method_colors)
ax.set_title("Average Latency by Policy", fontweight="bold")
ax.set_ylabel("ms")
ax.tick_params(axis="x", rotation=30)
ax.grid(axis="y", alpha=0.22)
plt.tight_layout()
if RUN_PLOTS: plt.show()
else: plt.close()

fig, ax = plt.subplots(figsize=(10.5, 5.2))
ax.bar(methods_short, df_compare["SLA %"].values, color=method_colors)
ax.axhline(80, color="red", linestyle="--", label="80% target")
ax.legend()
ax.set_title("SLA Compliance by Policy", fontweight="bold")
ax.set_ylabel("%")
ax.tick_params(axis="x", rotation=30)
ax.grid(axis="y", alpha=0.22)
plt.tight_layout()
if RUN_PLOTS: plt.show()
else: plt.close()

zone_dist = train_df["zone"].value_counts()
fig, ax = plt.subplots(figsize=(7.8, 5.0))
zone_colors = plt.cm.Set2(np.linspace(0, 1, len(zone_dist)))
ax.bar(zone_dist.index, zone_dist.values, color=zone_colors)
ax.set_title("Federated Client Distribution", fontweight="bold")
ax.set_ylabel("Training tasks")
ax.tick_params(axis="x", rotation=15)
ax.grid(axis="y", alpha=0.22)
plt.tight_layout()
if RUN_PLOTS: plt.show()
else: plt.close()

fig, ax = plt.subplots(figsize=(12, 5.0))
ax.plot(network_state["timestep"], network_state["uplink_bandwidth"], label="Uplink BW (Mbps)", alpha=0.8)
ax.plot(network_state["timestep"], network_state["network_delay_ms"], label="Delay (ms)", alpha=0.8)
if "is_outage" in network_state.columns:
    for ts in network_state.loc[network_state["is_outage"] == 1, "timestep"]:
        ax.axvspan(ts - 0.5, ts + 0.5, color="red", alpha=0.20)
ax.set_title("Network Conditions Over Time", fontweight="bold")
ax.set_xlabel("Timestep")
ax.legend(fontsize=8)
ax.grid(alpha=0.22)
plt.tight_layout()
if RUN_PLOTS: plt.show()
else: plt.close()

sla_by_type = tasks.groupby("task_type")["sla_violated"].mean().sort_values(ascending=False)
fig, ax = plt.subplots(figsize=(9.5, 5.2))
cols10 = ["red" if value > 0.3 else "steelblue" for value in sla_by_type.values]
ax.bar(sla_by_type.index, sla_by_type.values, color=cols10)
ax.axhline(0.3, color="black", linestyle="--", lw=1, label="30% threshold")
ax.set_title("Dataset SLA Violation Rate by Task Type", fontweight="bold")
ax.set_ylabel("Rate")
ax.tick_params(axis="x", rotation=30)
ax.legend()
ax.grid(axis="y", alpha=0.22)
plt.tight_layout()
if RUN_PLOTS: plt.show()
else: plt.close()

mae_cpu = errors[:, 0].mean(); mae_mem = errors[:, 1].mean(); mae_bw = errors[:, 2].mean()
fig, ax = plt.subplots(figsize=(6.8, 4.8))
ax.bar(["CPU", "Memory", "Bandwidth"], [mae_cpu, mae_mem, mae_bw], color=["steelblue", "darkorange", "green"])
ax.set_title("Allocator MAE per Resource", fontweight="bold")
ax.set_ylabel("MAE")
ax.grid(axis="y", alpha=0.22)
plt.tight_layout()
if RUN_PLOTS: plt.show()
else: plt.close()

print("[CHART] Complete system dashboard was split into separate save-friendly figures.")


## Final Performance Summary


In [ ]:
model.eval()
with torch.no_grad():
    lg2 = model(X_test.to(device))
    pr2 = torch.argmax(lg2, dim=1).cpu().numpy()
    pb2 = torch.softmax(lg2, dim=1)[:, 1].cpu().numpy()

acc_net = accuracy_score(y_test_t.numpy(), pr2)
f1_net = f1_score(y_test_t.numpy(), pr2, average="weighted")
fpr2, tpr2, _ = roc_curve(y_test_t.numpy(), pb2)
auc_net = auc(fpr2, tpr2)

allocator.eval()
if "errors" in globals():
    mae_a = float(np.asarray(errors).mean())
else:
    ap_np = allocator_predict_np(edge_train_df, X_alloc_raw, base_alloc_np)
    mae_a = float(np.abs(ap_np - y_alloc_np).mean())

emerg_edge = float((tasks[tasks["task_type"] == "emergency"]["offload_label"] == 0).mean()) * 100
n_rej_flag = tasks["rejection_flag"].sum()
fed_cmp = df_compare.loc["Fed-DDQN (Proposed)"]
ddqn_cmp = df_compare.loc["DDQN"]
flddpg_cmp = df_compare.loc["FL-DDPG"]
oracle_cmp = df_compare.loc["Oracle"]

print("=" * 76)
print(f"  {'METRIC':<49} {'VALUE':>22}")
print("=" * 76)
print(f"  {'[OffloadNet] Accuracy':<49} {acc_net * 100:>21.2f}%")
print(f"  {'[OffloadNet] Weighted F1':<49} {f1_net:>22.4f}")
print(f"  {'[OffloadNet] ROC-AUC':<49} {auc_net:>22.4f}")
print("-" * 76)
print(f"  {'[Fed-DDQN v6.6] Avg Latency (ms)':<49} {fed_cmp['Avg Latency']:>22.4f}")
print(f"  {'[Fed-DDQN v6.6] SLA Compliance':<49} {fed_cmp['SLA %']:>21.2f}%")
print(f"  {'[Fed-DDQN v6.6] Rejection %':<49} {fed_cmp['Rejection %']:>21.2f}%")
print(f"  {'[Fed-DDQN v6.6] Edge Usage':<49} {fed_cmp['Edge Usage %']:>21.2f}%")
print(f"  {'[Fed-DDQN v6.6] Best Validation Round':<49} {best_round:>22}")
print(f"  {'[Fed-DDQN v6.6] Best Validation Score':<49} {best_val_score:>22.4f}")
print(f"  {'[Fed-DDQN v6.6] Rounds Actually Run':<49} {rounds_ran:>22}")
print("-" * 76)
print(f"  {'[Compare] DDQN Avg Latency':<49} {ddqn_cmp['Avg Latency']:>22.3f} ms")
print(f"  {'[Compare] FL-DDPG Avg Latency':<49} {flddpg_cmp['Avg Latency']:>22.3f} ms")
print(f"  {'[Compare] Oracle Avg Latency':<49} {oracle_cmp['Avg Latency']:>22.3f} ms")
print(f"  {'[Compare] Fed vs DDQN Improvement':<49} "
      f"{((ddqn_cmp['Avg Latency'] - fed_cmp['Avg Latency']) / max(ddqn_cmp['Avg Latency'], 1e-6) * 100.0):>21.2f}%")
print(f"  {'[Compare] Fed vs FL-DDPG Improvement':<49} "
      f"{((flddpg_cmp['Avg Latency'] - fed_cmp['Avg Latency']) / max(flddpg_cmp['Avg Latency'], 1e-6) * 100.0):>21.2f}%")
print("-" * 76)
print(f"  {'[Allocator] CPU MAE':<49} {errors[:, 0].mean():>22.4f}")
print(f"  {'[Allocator] Memory MAE':<49} {errors[:, 1].mean():>22.4f}")
print(f"  {'[Allocator] Bandwidth MAE':<49} {errors[:, 2].mean():>22.4f}")
print(f"  {'[Allocator] Overall MAE':<49} {mae_a:>22.4f}")
if "df_allocator_compare" in globals() and len(df_allocator_compare):
    _alloc_best = df_allocator_compare.sort_values(["Resource Efficiency Score", "Overall MAE"], ascending=[False, True]).iloc[0]
    print(f"  {'[Allocator QoS] Best Method':<49} {str(_alloc_best['Allocator Method']):>22}")
    print(f"  {'[Allocator QoS] Under-allocation %':<49} {_alloc_best['Under-allocation %']:>21.2f}%")
    print(f"  {'[Allocator QoS] Waste %':<49} {_alloc_best['Waste %']:>21.2f}%")
    print(f"  {'[Allocator QoS] Capacity Violation %':<49} {_alloc_best['Capacity Violation %']:>21.2f}%")
    print(f"  {'[Allocator QoS] Estimated SLA Risk %':<49} {_alloc_best['Estimated SLA Risk %']:>21.2f}%")
    print(f"  {'[Allocator QoS] Efficiency Score':<49} {_alloc_best['Resource Efficiency Score']:>22.3f}")
print("-" * 76)
print(f"  {'[System] Rejection Flag Count':<49} {n_rej_flag:>22,}")
print(f"  {'[System] Rejection Rate':<49} {tasks['rejection_flag'].mean():>22.4f}")
print(f"  {'[System] SLA Violation Rate':<49} {tasks['sla_violated'].mean():>22.4f}")
print(f"  {'[System] Emergency->Edge Label Rate':<49} {emerg_edge:>21.2f}%")
print(f"  {'[System] Federated Zones':<49} {str(zone_names):>22}")
print("=" * 76)
print("\nImprovements applied:")
print("  [v6.6] Proposed model remains Federated DDQN for edge-cloud process scheduling/offloading")
print("  [v6.6] Reward rebalanced against edge overuse and heavy-task edge misrouting")
print("  [v6.6] Proposed training uses evaluation-aligned environment by default")
print("  [v6.6] Validation score includes latency, SLA, rejection, edge overuse, and heavy-task routing")
print("  [v6.6] Main comparison uses the untouched test split when EVALUATE_FULL_TEST_SPLIT=True")
if "df_ablation_summary" in globals() and len(df_ablation_summary):
    print(f"  [v6.6] Multi-seed ablation variants: {len(df_ablation_summary)} | seeds: {EXPERIMENT_SEEDS}")
if "df_scenario_final" in globals() and len(df_scenario_final):
    print(f"  [v6.6] Scenario-wise rows: {len(df_scenario_final)}")
if "results_loaded_from_cache" in globals():
    print(f"  [v6.6] Result-table cache loaded: {results_loaded_from_cache}")
if "policy_eval_loaded_from_cache" in globals():
    print(f"  [v6.6] Policy comparison/action cache loaded: {policy_eval_loaded_from_cache}")
print(f"  [v6.6] Stage-2 allocator upgraded: rejection-aware residual over demand baseline + capacity normalization + baselines")
if "df_allocator_ablation" in globals() and len(df_allocator_ablation):
    print(f"  [v6.6] Allocator ablation variants: {len(df_allocator_ablation)}")
cache_flags = {
    "labels": globals().get("labels_cache_loaded", False),
    "features": globals().get("feature_cache_loaded", False),
    "offloadnet": globals().get("offloadnet_loaded_from_cache", False),
    "fed_ddqn": globals().get("fed_ddqn_loaded_from_cache", False),
    "baselines": globals().get("baseline_models_loaded_from_cache", False),
}
print(
    f"  [cache] labels={cache_flags['labels']} features={cache_flags['features']} "
    f"offloadnet={cache_flags['offloadnet']} fed_ddqn={cache_flags['fed_ddqn']} "
    f"baselines={cache_flags['baselines']}"
)
print(
    f"  [cache] allocator_targets={globals().get('alloc_targets_loaded_from_cache', False)} "
    f"allocator={globals().get('allocator_loaded_from_cache', False)} "
    f"use_allocator_caches={globals().get('USE_ALLOCATOR_CACHES', True)}"
)
print(
    f"  [cache] allocator_stage2={globals().get('allocator_stage2_loaded_from_cache', False)} "
    f"allocator_compare={globals().get('allocator_compare_loaded_from_cache', False)} "
    f"allocator_ablation={globals().get('allocator_ablation_loaded_from_cache', False)}"
)
print("\nSanity checks:")
print(f"  max edge_lat  = {tasks['edge_latency'].max():.2f} (cap={EDGE_LAT_CAP})")
print(f"  max cloud_lat = {tasks['cloud_latency'].max():.2f} (cap={CLOUD_LAT_CAP})")
print(f"  DDQN eval lat range = {model_lat.min():.2f} to {model_lat.max():.2f} ms")
assert np.isnan(errors).sum() == 0, "NaN in allocator errors"
assert alloc_targets.shape[1] == 3, "Allocator must have 3 outputs"
print("  Static sanity checks passed for v6.6.")


## Reproducibility Export Artifacts


In [ ]:
# Experiment reproducibility export bundle.
# Cache-only: writes dataset/reward/split/experiment metadata and optional completed hook outputs.
import os, json, hashlib, time
from pathlib import Path

exports_written = []
exports_skipped = []

if globals().get("EXPORT_REPRO_BUNDLE", True):
    experiment_export_dir = globals().get(
        "experiment_export_dir",
        os.path.join(base_path, "_v66_cache", "experiment_exports"),
    )
    os.makedirs(experiment_export_dir, exist_ok=True)
    export_root = Path(experiment_export_dir)

    def _json_safe(obj):
        if isinstance(obj, (np.integer,)):
            return int(obj)
        if isinstance(obj, (np.floating,)):
            return float(obj)
        if isinstance(obj, (np.bool_,)):
            return bool(obj)
        if isinstance(obj, (np.ndarray,)):
            return obj.tolist()
        if isinstance(obj, (pd.Series,)):
            return obj.tolist()
        if isinstance(obj, (pd.DataFrame,)):
            return obj.to_dict(orient="records")
        return str(obj)

    def _write_json(name, payload):
        path = export_root / name
        with open(path, "w", encoding="utf-8") as handle:
            json.dump(payload, handle, indent=2, sort_keys=True, default=_json_safe)
        exports_written.append(str(path))
        return path

    def _write_csv(name, df):
        path = export_root / name
        df.to_csv(path, index=False)
        exports_written.append(str(path))
        return path

    def _sha256_file(path):
        h = hashlib.sha256()
        with open(path, "rb") as handle:
            for chunk in iter(lambda: handle.read(1024 * 1024), b""):
                h.update(chunk)
        return h.hexdigest()

    dataset_rows = []
    for logical_name, filename, df_obj in globals().get("_DATASET_FILES", []):
        fpath = os.path.join(base_path, filename)
        stat = os.stat(fpath)
        dataset_rows.append({
            "logical_name": logical_name,
            "filename": filename,
            "rows": int(len(df_obj)),
            "columns": int(len(df_obj.columns)),
            "schema_sha256": hashlib.sha256(json.dumps(list(df_obj.columns), sort_keys=True).encode("utf-8")).hexdigest(),
            "file_size_bytes": int(stat.st_size),
            "mtime_ns": int(stat.st_mtime_ns),
            "file_sha256": _sha256_file(fpath),
        })
    _write_csv("dataset_checksums.csv", pd.DataFrame(dataset_rows))

    def _mask_summary(name, mask, arrival_window):
        mask = np.asarray(mask, dtype=bool)
        arr = tasks.loc[mask, "arrival_time"].to_numpy(dtype=np.int64) if mask.any() else np.array([], dtype=np.int64)
        return {
            "name": name,
            "arrival_time_window": arrival_window,
            "rows": int(mask.sum()),
            "min_arrival_time": int(arr.min()) if len(arr) else None,
            "max_arrival_time": int(arr.max()) if len(arr) else None,
            "mask_sha256": _short_array_hash(mask) if "_short_array_hash" in globals() else None,
        }

    if all(name in globals() for name in ["train_mask", "gap1_mask", "val_mask", "gap2_mask", "test_mask"]):
        split_manifest = {
            "protocol": "purged_temporal_train_validation_test",
            "train": _mask_summary("train", train_mask, "<=700"),
            "gap_1": _mask_summary("gap_1", gap1_mask, "701-715"),
            "validation": _mask_summary("validation", val_mask, "716-850"),
            "gap_2": _mask_summary("gap_2", gap2_mask, "851-865"),
            "test": _mask_summary("test", test_mask, "866-1000"),
            "test_is_tuning_free": True,
            "scaler_fit_scope": "train split only",
        }
    else:
        arrival = tasks["arrival_time"]
        split_manifest = {
            "protocol": "purged_temporal_train_validation_test",
            "train": {"arrival_time_window": "<=700", "rows": int((arrival <= 700).sum())},
            "gap_1": {"arrival_time_window": "701-715", "rows": int(arrival.between(701, 715).sum())},
            "validation": {"arrival_time_window": "716-850", "rows": int(arrival.between(716, 850).sum())},
            "gap_2": {"arrival_time_window": "851-865", "rows": int(arrival.between(851, 865).sum())},
            "test": {"arrival_time_window": "866-1000", "rows": int(arrival.between(866, 1000).sum())},
            "test_is_tuning_free": True,
            "scaler_fit_scope": "train split only",
        }
    _write_json("split_manifest.json", split_manifest)

    reward_config = {
        "stage_1_model": "Federated Double DQN",
        "action_space": {"0": "edge", "1": "cloud"},
        "rejection_is_outcome_not_action": True,
        "gamma_default": float(globals().get("GAMMA", 0.95)),
        "terminal_masked_target_formula": "r + gamma * (1 - done) * Q_target(next_state, argmax_a Q_online(next_state, a))",
        "terminal_masked_training_hooks_present": True,
        "fed_validation_score": {
            "avg_latency_weight": 1.0,
            "sla_miss_percent_weight": 5.0,
            "rejection_percent_weight": 2.5,
            "edge_overuse_weight": 0.8,
            "heavy_edge_bad_weight": 1.2,
            "edge_overuse_soft_cap_percent": float(globals().get("PROPOSED_EDGE_OVERUSE_SOFT_CAP", 78.0)),
        },
        "proposed_controls": {
            "action_aware_env": bool(globals().get("PROPOSED_USE_ACTION_AWARE_ENV", False)),
            "prioritized_replay": bool(globals().get("PROPOSED_USE_PRIORITIZED_REPLAY", False)),
            "mu_proximal": float(globals().get("PROPOSED_MU_PROXIMAL", 0.003)),
            "target_tau": float(globals().get("PROPOSED_TARGET_TAU", 0.012)),
            "heavy_task_edge_penalty": bool(globals().get("PROPOSED_HEAVY_TASK_EDGE_PENALTY", True)),
        },
        "cache_note": "Existing model caches are preserved; terminal masking is applied whenever affected models are retrained after this hook update.",
    }
    _write_json("reward_config.json", reward_config)

    allocator_spec = {
        "stage_2_model": "rejection-aware residual resource allocator",
        "allocator_version": globals().get("ALLOCATOR_VERSION", None),
        "scope": "edge-selected tasks from the proposed Fed-DDQN policy",
        "target_generation": [
            "demand-proportional base allocation",
            "deadline/priority/SLA-risk correction",
            "task type and urgency adjustment",
            "queue pressure and reliability adjustment",
            "low-battery/dependency/network-stress adjustment",
            "post-prediction capacity normalization",
        ],
        "risk_estimate_excludes": ["realized labels", "Oracle actions", "final generated allocation targets"],
        "resource_outputs": ["cpu", "memory", "bandwidth"],
        "allocator_caches_enabled": bool(globals().get("USE_ALLOCATOR_CACHES", True)),
    }
    if "df_allocator_compare" in globals() and len(df_allocator_compare):
        allocator_spec["allocator_comparison"] = df_allocator_compare.to_dict(orient="records")
    if "df_allocator_ablation" in globals() and len(df_allocator_ablation):
        allocator_spec["allocator_ablation"] = df_allocator_ablation.to_dict(orient="records")
    _write_json("allocator_target_risk_spec.json", allocator_spec)

    def _read_optional_pickle(path):
        if not path or not os.path.exists(path):
            return None
        try:
            if "read_pickle_compat" in globals():
                return read_pickle_compat(path)
            return pd.read_pickle(path)
        except Exception as exc:
            print(f"Optional cache ignored during export ({path}): {exc}")
            return None

    optional_artifacts = {}
    gamma_df = globals().get("df_gamma_zero_ablation", pd.DataFrame())
    gamma_summary = globals().get("df_gamma_zero_summary", pd.DataFrame())
    if (not isinstance(gamma_df, pd.DataFrame) or not len(gamma_df)) and "gamma_zero_results_cache_path" in globals():
        cached_gamma = _read_optional_pickle(gamma_zero_results_cache_path)
        if isinstance(cached_gamma, dict):
            gamma_df = cached_gamma.get("df_gamma_zero_ablation", pd.DataFrame())
            gamma_summary = cached_gamma.get("df_gamma_zero_summary", pd.DataFrame())
    if isinstance(gamma_df, pd.DataFrame) and len(gamma_df):
        _write_csv("gamma_zero_ablation_results.csv", gamma_df)
        _write_json("gamma_zero_ablation_summary.json", {
            "status": "available",
            "source": "notebook_run_or_loaded_cache",
            "summary": gamma_summary.to_dict(orient="records") if isinstance(gamma_summary, pd.DataFrame) else [],
        })
        optional_artifacts["gamma_zero_ablation"] = "available"
    else:
        optional_artifacts["gamma_zero_ablation"] = "not_run_or_no_valid_cache"
        exports_skipped.append("gamma_zero_ablation_results.csv")

    baseline_ms_df = globals().get("df_baseline_multiseed_raw", pd.DataFrame())
    baseline_ms_summary = globals().get("df_baseline_multiseed_summary", pd.DataFrame())
    if (not isinstance(baseline_ms_df, pd.DataFrame) or not len(baseline_ms_df)) and "baseline_multiseed_results_cache_path" in globals():
        cached_baseline_ms = _read_optional_pickle(baseline_multiseed_results_cache_path)
        if isinstance(cached_baseline_ms, dict):
            baseline_ms_df = cached_baseline_ms.get("df_baseline_multiseed_raw", pd.DataFrame())
            baseline_ms_summary = cached_baseline_ms.get("df_baseline_multiseed_summary", pd.DataFrame())
    if isinstance(baseline_ms_df, pd.DataFrame) and len(baseline_ms_df):
        _write_csv("baseline_multiseed_raw.csv", baseline_ms_df)
        _write_csv("baseline_multiseed_summary.csv", baseline_ms_summary if isinstance(baseline_ms_summary, pd.DataFrame) else pd.DataFrame())
        optional_artifacts["baseline_multiseed"] = "available"
    else:
        optional_artifacts["baseline_multiseed"] = "not_run_or_no_valid_cache"
        exports_skipped.append("baseline_multiseed_raw.csv")
        exports_skipped.append("baseline_multiseed_summary.csv")

    df_compare_export = globals().get("df_compare", pd.DataFrame())
    if (not isinstance(df_compare_export, pd.DataFrame) or not len(df_compare_export)) and "policy_eval_cache_path" in globals():
        cached_policy_eval = _read_optional_pickle(policy_eval_cache_path)
        if isinstance(cached_policy_eval, dict):
            df_compare_export = cached_policy_eval.get("df_compare", pd.DataFrame())
    if isinstance(df_compare_export, pd.DataFrame) and len(df_compare_export):
        _write_csv("main_policy_comparison_test_only.csv", df_compare_export.reset_index())

    fed_ablation_summary_export = globals().get("df_ablation_summary", pd.DataFrame())
    fed_multiseed_raw_export = globals().get("df_full_multiseed", pd.DataFrame())
    scenario_final_export = globals().get("df_scenario_final", pd.DataFrame())
    if (
        (not isinstance(fed_ablation_summary_export, pd.DataFrame) or not len(fed_ablation_summary_export))
        and "results_cache_path" in globals()
    ):
        cached_suite = _read_optional_pickle(results_cache_path)
        if isinstance(cached_suite, dict):
            fed_ablation_summary_export = cached_suite.get("df_ablation_summary", pd.DataFrame())
            fed_multiseed_raw_export = cached_suite.get("df_full_multiseed", pd.DataFrame())
            scenario_final_export = cached_suite.get("df_scenario_final", pd.DataFrame())
    if isinstance(fed_ablation_summary_export, pd.DataFrame) and len(fed_ablation_summary_export):
        _write_csv("fed_ddqn_ablation_summary.csv", fed_ablation_summary_export)
    if isinstance(fed_multiseed_raw_export, pd.DataFrame) and len(fed_multiseed_raw_export):
        _write_csv("fed_ddqn_multiseed_raw.csv", fed_multiseed_raw_export)
    if isinstance(scenario_final_export, pd.DataFrame) and len(scenario_final_export):
        _write_csv("scenario_wise_final_table.csv", scenario_final_export)

    reproducibility_manifest = {
        "created_unix_time": float(time.time()),
        "notebook": "v6.6.ipynb",
        "dataset_folder": base_path,
        "dataset_description": "real-world-inspired synthetic edge-cloud IoT dataset",
        "cache_signature": globals().get("cache_signature", None),
        "cache_prefix": globals().get("cache_prefix", None),
        "run_cache_version": globals().get("RUN_CACHE_VERSION", None),
        "trained_model_cache_dir": globals().get("cache_dir", None),
        "new_hook_cache_dir": globals().get("v66_cache_dir", None),
        "flags": {
            "RUN_GAMMA_ZERO_ABLATION": bool(globals().get("RUN_GAMMA_ZERO_ABLATION", False)),
            "RUN_BASELINE_MULTI_SEED": bool(globals().get("RUN_BASELINE_MULTI_SEED", False)),
            "FORCE_RETRAIN_BASELINE_MULTI_SEED": bool(globals().get("FORCE_RETRAIN_BASELINE_MULTI_SEED", False)),
            "EXPORT_REPRO_BUNDLE": bool(globals().get("EXPORT_REPRO_BUNDLE", False)),
            "LOAD_EXPERIMENT_RESULTS_ONLY": bool(globals().get("LOAD_EXPERIMENT_RESULTS_ONLY", False)),
            "STAGE1_CACHE_ONLY": bool(globals().get("STAGE1_CACHE_ONLY", False)),
        },
        "cache_loaded_flags": {
            "labels": bool(globals().get("labels_cache_loaded", False)),
            "features": bool(globals().get("feature_cache_loaded", False)),
            "fed_ddqn": bool(globals().get("fed_ddqn_loaded_from_cache", False)),
            "baselines": bool(globals().get("baseline_models_loaded_from_cache", False)),
            "policy_eval": bool(globals().get("policy_eval_loaded_from_cache", False)),
            "multi_seed_ablation_results": bool(globals().get("results_loaded_from_cache", False)),
            "gamma_zero": bool(globals().get("gamma_zero_loaded_from_cache", False)),
            "baseline_multiseed": bool(globals().get("baseline_multiseed_loaded_from_cache", False)),
        },
        "optional_artifacts": optional_artifacts,
        "reporting_rule": "Do not report gamma-zero or baseline-variance numbers unless the corresponding exported files are available.",
    }
    _write_json("reproducibility_manifest.json", reproducibility_manifest)

    export_index_path = export_root / "exports_index.json"
    export_index = {
        "written": exports_written + [str(export_index_path)],
        "skipped_optional": exports_skipped,
        "export_root": str(export_root),
    }
    _write_json("exports_index.json", export_index)
    print("Manuscript export bundle written:")
    for item in exports_written:
        print("  ", item)
    if exports_skipped:
        print("Optional exports not written because no completed run/cache exists:")
        for item in exports_skipped:
            print("  ", item)
else:
    print("EXPORT_REPRO_BUNDLE=False; experiment reproducibility export skipped.")


## Standalone Result Figure Exports


In [ ]:

# result result figure appendix.
# This cell does not change model training, evaluation, or cached results.
# It only generates additional standalone figures for result-reporting use.

import os
import re
import math
import textwrap
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch

RESULT_FIG_DIR = Path(os.getcwd()) / "result_figures" / "v66"
RESULT_FIG_DIR.mkdir(parents=True, exist_ok=True)
RESULT_FIG_DPI = 350
_result_figures_saved = []


def _num(x):
    """Parse floats from numbers or strings such as '136.855 +/- 0.599' / '136.855 ± 0.599'."""
    if isinstance(x, (int, float, np.integer, np.floating)):
        return float(x)
    if pd.isna(x):
        return np.nan
    match = re.search(r"[-+]?\d*\.?\d+", str(x))
    return float(match.group(0)) if match else np.nan


def _df_with_method(df):
    out = df.copy()
    if "Method" not in out.columns:
        out = out.reset_index().rename(columns={"index": "Method"})
    return out


def _result_save(fig, name):
    safe = re.sub(r"[^A-Za-z0-9_.-]+", "_", name).strip("_").lower()
    png = RESULT_FIG_DIR / f"{safe}.png"
    fig.savefig(png, dpi=RESULT_FIG_DPI, bbox_inches="tight", facecolor="white")
    _result_figures_saved.append((name, str(png)))
    if globals().get("RUN_PLOTS", True):
        plt.show()
    else:
        plt.close(fig)


def _wrap_label(text, width=18):
    parts = []
    for part in str(text).split("\n"):
        part = part.strip()
        if not part:
            parts.append("")
        else:
            parts.extend(textwrap.wrap(part, width=width, break_long_words=False))
    return "\n".join(parts)


def _box(ax, xy, wh, text, fc="#F6F8FA", ec="#2F3A45", fontsize=8.8, wrap_width=18):
    x, y = xy
    w, h = wh
    patch = FancyBboxPatch(
        (x, y), w, h,
        boxstyle="round,pad=0.012,rounding_size=0.018",
        linewidth=1.15,
        facecolor=fc,
        edgecolor=ec,
        zorder=2,
    )
    ax.add_patch(patch)
    ax.text(
        x + w / 2, y + h / 2,
        _wrap_label(text, wrap_width),
        ha="center", va="center",
        fontsize=fontsize,
        linespacing=1.18,
        color="#17202A",
        zorder=3,
    )
    return patch


def _arrow(ax, start, end, text=None, rad=0.0, label_offset=(0.0, 0.035), label_pos=0.5):
    arr = FancyArrowPatch(
        start, end,
        arrowstyle="-|>",
        mutation_scale=12,
        linewidth=1.15,
        color="#2F3A45",
        connectionstyle=f"arc3,rad={rad}",
        shrinkA=7,
        shrinkB=7,
        zorder=1,
    )
    ax.add_patch(arr)
    if text:
        mx = start[0] + (end[0] - start[0]) * label_pos
        my = start[1] + (end[1] - start[1]) * label_pos
        ax.text(
            mx + label_offset[0], my + label_offset[1],
            _wrap_label(text, 14),
            ha="center", va="center",
            fontsize=7.6,
            color="#2F3A45",
            bbox=dict(boxstyle="round,pad=0.18", fc="white", ec="none", alpha=0.92),
            zorder=4,
        )


def _has(name):
    return name in globals() and globals()[name] is not None


print(f"Saving result figure appendix to: {RESULT_FIG_DIR}")

# Figure A1: End-to-end system architecture aligned with the project title.
fig, ax = plt.subplots(figsize=(13.2, 6.2))
ax.set_axis_off()
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
_box(ax, (0.03, 0.68), (0.17, 0.18), "IoT Tasks\nsize, cycles, deadline,\npriority, battery", "#EAF4FF")
_box(ax, (0.03, 0.36), (0.17, 0.18), "Channel State\nSNR, bandwidth,\nloss, outage, jitter", "#FFF3E6")
_box(ax, (0.25, 0.52), (0.18, 0.20), "Feature Builder\n30 task + edge +\ncloud + network features", "#F5F0FF")
_box(ax, (0.49, 0.64), (0.18, 0.17), "Zone Clients\nurban / suburban /\nrural / industrial", "#EFFFF5")
_box(ax, (0.49, 0.37), (0.18, 0.17), "Federated DDQN\nDouble Q target,\nFedAvg, FedProx", "#E9FFF8")
_box(ax, (0.73, 0.62), (0.20, 0.15), "Offloading Action\nEdge / Cloud / Reject", "#EAF4FF")
_box(ax, (0.73, 0.37), (0.20, 0.15), "Residual Allocator\nCPU / memory / bandwidth\nfor edge-selected tasks", "#FFF0F0")
_box(ax, (0.73, 0.12), (0.20, 0.14), "Evaluation\nlatency, SLA,\nrejection, resource QoS", "#F6F8FA")
_arrow(ax, (0.20, 0.77), (0.25, 0.64))
_arrow(ax, (0.20, 0.45), (0.25, 0.59))
_arrow(ax, (0.43, 0.62), (0.49, 0.72), "non-IID zones")
_arrow(ax, (0.58, 0.64), (0.58, 0.54), "global sync")
_arrow(ax, (0.67, 0.46), (0.73, 0.69), "policy")
_arrow(ax, (0.83, 0.62), (0.83, 0.52), "edge tasks")
_arrow(ax, (0.83, 0.37), (0.83, 0.26), "QoS")
_arrow(ax, (0.83, 0.62), (0.83, 0.26), "cloud/reject metrics", rad=-0.20)
ax.text(0.5, 0.96, "Federated DDQN for Channel-Aware Edge-Cloud Offloading with Rejection-Aware Resource Allocation",
        ha="center", va="center", fontsize=13.5, fontweight="bold")
_result_save(fig, "fig_a1_system_architecture")

# Figure A2: Purged temporal split and stage roles.
if _has("tasks"):
    fig, ax = plt.subplots(figsize=(12.5, 3.1))
    ax.set_xlim(1, 1000)
    ax.set_ylim(0, 1)
    ax.set_yticks([])
    spans = [
        (1, 700, "Train\nFed-DDQN + baselines", "#E9FFF8"),
        (701, 715, "Gap", "#F3F3F3"),
        (716, 850, "Validation\ncheckpoint choice", "#FFF3E6"),
        (851, 865, "Gap", "#F3F3F3"),
        (866, 1000, "Test\nfull reporting run metrics", "#EAF4FF"),
    ]
    for a, b, label, color in spans:
        ax.axvspan(a, b, color=color, ec="#555", lw=0.8)
        ax.text((a + b) / 2, 0.50, label, ha="center", va="center", fontsize=10)
    ax.set_xlabel("arrival_time timestep")
    ax.set_title("Purged Temporal Evaluation Protocol", fontweight="bold")
    ax.spines[["left", "right", "top"]].set_visible(False)
    _result_save(fig, "fig_a2_temporal_split_protocol")

# Figure A3: Policy trade-off bubble chart.
if _has("df_compare") and len(df_compare):
    dfc = _df_with_method(df_compare)
    required = {"Method", "Avg Latency", "SLA %", "Rejection %", "Edge Usage %"}
    if required.issubset(dfc.columns):
        fig, ax = plt.subplots(figsize=(9.4, 6.2))
        x = dfc["Avg Latency"].map(_num)
        y = dfc["SLA %"].map(_num)
        s = 70 + 45 * dfc["Rejection %"].map(_num).fillna(0)
        c = dfc["Edge Usage %"].map(_num)
        sc = ax.scatter(x, y, s=s, c=c, cmap="viridis", alpha=0.85, edgecolor="#222", linewidth=0.7)
        for _, row in dfc.iterrows():
            weight = "bold" if "Fed-DDQN" in str(row["Method"]) else "normal"
            ax.annotate(str(row["Method"]), (_num(row["Avg Latency"]), _num(row["SLA %"])),
                        xytext=(5, 5), textcoords="offset points", fontsize=8.5, fontweight=weight)
        cb = fig.colorbar(sc, ax=ax)
        cb.set_label("Edge usage (%)")
        ax.set_xlabel("Average latency (ms), lower is better")
        ax.set_ylabel("SLA compliance (%), higher is better")
        ax.set_title("Stage-1 Policy Trade-off: Latency, SLA, Rejection, and Edge Usage", fontweight="bold")
        ax.grid(alpha=0.25)
        _result_save(fig, "fig_a3_policy_tradeoff_bubble")

# Figure A4: Fed-DDQN latency improvement over trainable/literature baselines.
if _has("df_compare") and "Fed-DDQN (Proposed)" in df_compare.index:
    dfc = _df_with_method(df_compare)
    fed_latency = float(df_compare.loc["Fed-DDQN (Proposed)", "Avg Latency"])
    rows = []
    for _, row in dfc.iterrows():
        method = str(row["Method"])
        if method in {"Fed-DDQN (Proposed)", "Oracle"}:
            continue
        base_lat = _num(row["Avg Latency"])
        if np.isfinite(base_lat) and base_lat > 0:
            rows.append((method, (base_lat - fed_latency) / base_lat * 100.0))
    if rows:
        rows = sorted(rows, key=lambda x: x[1])
        fig, ax = plt.subplots(figsize=(8.8, 5.4))
        y = np.arange(len(rows))
        vals = [v for _, v in rows]
        labels = [m for m, _ in rows]
        colors = ["#1D9E75" if v >= 0 else "#D85A30" for v in vals]
        ax.hlines(y, 0, vals, color=colors, lw=2.5)
        ax.scatter(vals, y, color=colors, s=70, zorder=3)
        ax.axvline(0, color="#555", lw=1.0)
        ax.set_yticks(y)
        ax.set_yticklabels(labels)
        ax.set_xlabel("Fed-DDQN latency reduction vs method (%)")
        ax.set_title("Stage-1 Latency Advantage of Fed-DDQN (Proposed)", fontweight="bold")
        ax.grid(axis="x", alpha=0.25)
        _result_save(fig, "fig_a4_fed_ddqn_latency_improvement")

# Figure A5: Edge/cloud/reject decision distribution by method.
if _has("actions_by_method"):
    selected_methods = [m for m in ["DDQN", "FL-DDPG", "Fed-DDQN (Proposed)", "Oracle"] if m in actions_by_method]
    dist_rows = []
    for method in selected_methods:
        arr = np.asarray(actions_by_method[method], dtype=int)
        total = max(len(arr), 1)
        dist_rows.append({
            "Method": method,
            "Edge": float((arr == 0).sum() / total * 100.0),
            "Cloud": float((arr == 1).sum() / total * 100.0),
            "Reject": float((arr == 2).sum() / total * 100.0),
        })
    if dist_rows:
        dist = pd.DataFrame(dist_rows).set_index("Method")
        fig, ax = plt.subplots(figsize=(9.0, 5.2))
        left = np.zeros(len(dist))
        colors = {"Edge": "#1D9E75", "Cloud": "#378ADD", "Reject": "#D85A30"}
        for col in ["Edge", "Cloud", "Reject"]:
            ax.barh(dist.index, dist[col], left=left, color=colors[col], label=col)
            left += dist[col].to_numpy()
        ax.set_xlim(0, 100)
        ax.set_xlabel("Decision share (%)")
        ax.set_title("Policy Action Distribution on the Test Split", fontweight="bold")
        ax.legend(loc="lower right")
        ax.grid(axis="x", alpha=0.25)
        _result_save(fig, "fig_a5_policy_action_distribution")

# Figure A6: Scenario-wise latency heatmap.
if _has("df_scenario_final") and len(df_scenario_final):
    sdf = df_scenario_final.copy()
    sdf["LatencyNum"] = sdf["Avg Latency"].map(_num)
    keep_methods = [
        "Fed-DDQN Full (mean±std)", "Fed-DDQN Full (mean+/-std)",
        "Fed-DDQN (Proposed)", "DDQN", "FL-DDPG", "Oracle"
    ]
    sdf = sdf[sdf["Method"].isin(keep_methods)]
    if len(sdf):
        piv = sdf.pivot_table(index="Scenario", columns="Method", values="LatencyNum", aggfunc="mean")
        fig_h = max(4.8, 0.36 * len(piv) + 1.8)
        fig, ax = plt.subplots(figsize=(8.8, fig_h))
        im = ax.imshow(piv.to_numpy(), aspect="auto", cmap="viridis_r")
        ax.set_xticks(np.arange(len(piv.columns)))
        ax.set_xticklabels(piv.columns, rotation=30, ha="right")
        ax.set_yticks(np.arange(len(piv.index)))
        ax.set_yticklabels(piv.index)
        ax.set_title("Scenario-wise Latency Robustness", fontweight="bold")
        cb = fig.colorbar(im, ax=ax)
        cb.set_label("Average latency (ms)")
        _result_save(fig, "fig_a6_scenario_latency_heatmap")

# Figure A7: Multi-seed ablation uncertainty for latency.
if _has("df_multiseed_raw") and len(df_multiseed_raw):
    raw = df_multiseed_raw.copy()
    if {"Ablation", "Avg Latency"}.issubset(raw.columns):
        agg = raw.groupby("Ablation")["Avg Latency"].agg(["mean", "std", "count"]).reset_index()
        agg["sem95"] = 1.96 * agg["std"].fillna(0) / np.sqrt(agg["count"].clip(lower=1))
        agg = agg.sort_values("mean", ascending=True)
        fig, ax = plt.subplots(figsize=(9.5, 5.8))
        y = np.arange(len(agg))
        ax.errorbar(agg["mean"], y, xerr=agg["sem95"], fmt="o", color="#378ADD",
                    ecolor="#555", elinewidth=1.4, capsize=4)
        ax.set_yticks(y)
        ax.set_yticklabels(agg["Ablation"])
        ax.set_xlabel("Average latency (ms), mean with approx. 95% CI")
        ax.set_title("Fed-DDQN Ablation Reliability Across Seeds", fontweight="bold")
        ax.grid(axis="x", alpha=0.25)
        _result_save(fig, "fig_a7_multiseed_ablation_latency_ci")

# Figure A8: Stage-2 allocator QoS trade-off.
if _has("df_allocator_compare") and len(df_allocator_compare):
    ac = df_allocator_compare.copy()
    req = {"Allocator Method", "Overall MAE", "Resource Efficiency Score", "Under-allocation %", "Capacity Violation %"}
    if req.issubset(ac.columns):
        fig, ax = plt.subplots(figsize=(9.4, 6.2))
        sizes = 70 + 18 * ac["Under-allocation %"].map(_num).fillna(0)
        colors = ac["Capacity Violation %"].map(_num).fillna(0)
        sc = ax.scatter(ac["Overall MAE"].map(_num), ac["Resource Efficiency Score"].map(_num),
                        s=sizes, c=colors, cmap="Reds", alpha=0.82, edgecolor="#222", linewidth=0.7)
        for _, row in ac.iterrows():
            weight = "bold" if "Residual + Risk" in str(row["Allocator Method"]) else "normal"
            ax.annotate(str(row["Allocator Method"]), (_num(row["Overall MAE"]), _num(row["Resource Efficiency Score"])),
                        xytext=(5, 5), textcoords="offset points", fontsize=8.3, fontweight=weight)
        cb = fig.colorbar(sc, ax=ax)
        cb.set_label("Capacity violation (%)")
        ax.set_xlabel("Overall MAE, lower is better")
        ax.set_ylabel("Resource efficiency score, higher is better")
        ax.set_title("Stage-2 Allocator QoS Trade-off", fontweight="bold")
        ax.grid(alpha=0.25)
        _result_save(fig, "fig_a8_allocator_qos_tradeoff")

# Figure A9: Target vs assigned resource distribution for the proposed allocator.
if _has("stage2_targets_feasible") and _has("residual_risk_projection") and len(stage2_targets_feasible):
    resources = ["CPU", "Memory", "Bandwidth"]
    data = []
    labels = []
    for i, res in enumerate(resources):
        data.extend([stage2_targets_feasible[:, i], residual_risk_projection[:, i]])
        labels.extend([f"{res}\nTarget", f"{res}\nAssigned"])
    fig, ax = plt.subplots(figsize=(9.4, 5.5))
    bp = ax.boxplot(data, labels=labels, patch_artist=True, showfliers=False)
    for j, patch in enumerate(bp["boxes"]):
        patch.set_facecolor("#EAF4FF" if j % 2 == 0 else "#EFFFF5")
        patch.set_edgecolor("#2F3A45")
    ax.set_ylabel("Normalized resource share")
    ax.set_title("Stage-2 Proposed Allocator: Target vs Assigned Resource Shares", fontweight="bold")
    ax.grid(axis="y", alpha=0.25)
    _result_save(fig, "fig_a9_allocator_target_vs_assigned_distribution")

# Figure A10: Task mix selected for edge-side allocation.
if _has("stage2_eval_df") and len(stage2_eval_df) and "task_type" in stage2_eval_df.columns:
    counts = stage2_eval_df["task_type"].value_counts().sort_values(ascending=True)
    fig, ax = plt.subplots(figsize=(8.8, 5.4))
    ax.barh(counts.index.astype(str), counts.values, color="#7F77DD", alpha=0.88)
    ax.set_xlabel("Fed-DDQN edge-selected tasks")
    ax.set_title("Workload Composition Entering Stage-2 Resource Allocation", fontweight="bold")
    ax.grid(axis="x", alpha=0.25)
    _result_save(fig, "fig_a10_stage2_edge_selected_task_mix")

# Figure A11: Allocator ablation effect size.
if _has("df_allocator_ablation") and len(df_allocator_ablation):
    ab = df_allocator_ablation.copy()
    if {"Allocator Method", "Resource Efficiency Score"}.issubset(ab.columns):
        best_name = "Full Residual + Risk Projection"
        base_name = "Base Demand Only"
        fig, ax = plt.subplots(figsize=(9.4, 5.4))
        ab = ab.sort_values("Resource Efficiency Score", ascending=True)
        colors = ["#1D9E75" if best_name in str(m) else "#9AA4AF" for m in ab["Allocator Method"]]
        ax.barh(ab["Allocator Method"], ab["Resource Efficiency Score"].map(_num), color=colors)
        ax.set_xlabel("Resource efficiency score")
        ax.set_title("Stage-2 Allocator Ablation: Component Contribution", fontweight="bold")
        ax.grid(axis="x", alpha=0.25)
        _result_save(fig, "fig_a11_allocator_ablation_efficiency")

# Figure A12: Stage-2 rejection-aware residual allocation pipeline.
fig, ax = plt.subplots(figsize=(15.2, 6.2))
ax.set_axis_off()
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)

_box(ax, (0.035, 0.60), (0.14, 0.18), "Fed-DDQN\nedge-selected\ntasks", "#EAF4FF", fontsize=8.2, wrap_width=15)
_box(ax, (0.225, 0.60), (0.14, 0.18), "Demand base\nCPU, memory,\nbandwidth", "#F6F8FA", fontsize=8.2, wrap_width=15)
_box(ax, (0.415, 0.60), (0.15, 0.18), "QoS risk\nurgency, priority,\nqueue pressure", "#FFF3E6", fontsize=8.1, wrap_width=15)
_box(ax, (0.610, 0.60), (0.15, 0.18), "Residual net\nlearns corrective\nallocation delta", "#F5F0FF", fontsize=8.1, wrap_width=15)
_box(ax, (0.815, 0.60), (0.14, 0.18), "Capacity norm\nedge-timestep\nfeasibility", "#EFFFF5", fontsize=8.1, wrap_width=15)

_box(ax, (0.225, 0.20), (0.14, 0.16), "Baseline target\nfair demand\nproportion", "#F6F8FA", fontsize=8.0, wrap_width=15)
_box(ax, (0.610, 0.20), (0.15, 0.16), "Training loss\nMAE + residual\nunderallocation", "#FFF0F0", fontsize=8.0, wrap_width=15)
_box(ax, (0.815, 0.20), (0.14, 0.16), "Final shares\nCPU, memory,\nbandwidth", "#F0FAFF", fontsize=8.0, wrap_width=15)

_arrow(ax, (0.175, 0.69), (0.225, 0.69))
_arrow(ax, (0.365, 0.69), (0.415, 0.69))
_arrow(ax, (0.565, 0.69), (0.610, 0.69))
_arrow(ax, (0.760, 0.69), (0.815, 0.69))
_arrow(ax, (0.295, 0.60), (0.295, 0.36), "anchor", label_offset=(0.055, 0.0))
_arrow(ax, (0.685, 0.60), (0.685, 0.36), "optimize", label_offset=(0.064, 0.0))
_arrow(ax, (0.885, 0.60), (0.885, 0.36), "project", label_offset=(0.060, 0.0))
_arrow(ax, (0.365, 0.28), (0.610, 0.28), "target vs prediction", rad=-0.05, label_offset=(0.0, -0.045))
_arrow(ax, (0.760, 0.28), (0.815, 0.28))

ax.text(0.5, 0.93, "Rejection-Aware Residual Resource Allocation for Edge-Selected Tasks",
        ha="center", va="center", fontsize=13, fontweight="bold")
ax.text(0.5, 0.06, "Design intent: keep Stage 1 offloading fixed, then adapt resources for accepted edge tasks under QoS and capacity limits.",
        ha="center", va="center", fontsize=8.7, color="#4E5A65")
_result_save(fig, "fig_a12_stage2_allocator_pipeline")

print("\nresult figure appendix generated.")
for title, png in _result_figures_saved:
    print(f"- {title}: {png} | {pdf}")


## Additional Metrics and Flowcharts


In [ ]:


# Additional result result figures.
# Run after the result tables and helper functions are available.
# This cell is guarded: figures are skipped if their required variables are absent.

import os
import re
import math
import textwrap
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch

if "RESULT_FIG_DIR" not in globals():
    RESULT_FIG_DIR = Path(os.getcwd()) / "result_figures" / "v66"
    RESULT_FIG_DIR.mkdir(parents=True, exist_ok=True)
if "RESULT_FIG_DPI" not in globals():
    RESULT_FIG_DPI = 350
if "_result_figures_saved" not in globals():
    _result_figures_saved = []

if "_wrap_label" not in globals():
    def _wrap_label(text, width=18):
        parts = []
        for part in str(text).split("\n"):
            parts.extend(textwrap.wrap(part.strip(), width=width, break_long_words=False) or [""])
        return "\n".join(parts)

if "_num" not in globals():
    def _num(x):
        if isinstance(x, (int, float, np.integer, np.floating)):
            return float(x)
        if pd.isna(x):
            return np.nan
        match = re.search(r"[-+]?\d*\.?\d+", str(x))
        return float(match.group(0)) if match else np.nan

if "_has" not in globals():
    def _has(name):
        return name in globals() and globals()[name] is not None

if "_df_with_method" not in globals():
    def _df_with_method(df):
        out = df.copy()
        if "Method" not in out.columns:
            out = out.reset_index().rename(columns={"index": "Method"})
        return out

if "_result_save" not in globals():
    def _result_save(fig, name):
        safe = re.sub(r"[^A-Za-z0-9_.-]+", "_", name).strip("_").lower()
        png = RESULT_FIG_DIR / f"{safe}.png"
        fig.savefig(png, dpi=RESULT_FIG_DPI, bbox_inches="tight", facecolor="white")
        _result_figures_saved.append((name, str(png)))
        if globals().get("RUN_PLOTS", True):
            plt.show()
        else:
            plt.close(fig)

if "_box" not in globals():
    def _box(ax, xy, wh, text, fc="#F6F8FA", ec="#2F3A45", fontsize=8.8, wrap_width=18):
        x, y = xy
        w, h = wh
        patch = FancyBboxPatch((x, y), w, h, boxstyle="round,pad=0.012,rounding_size=0.018",
                               linewidth=1.15, facecolor=fc, edgecolor=ec, zorder=2)
        ax.add_patch(patch)
        ax.text(x + w / 2, y + h / 2, _wrap_label(text, wrap_width),
                ha="center", va="center", fontsize=fontsize, linespacing=1.18, zorder=3)
        return patch

if "_arrow" not in globals():
    def _arrow(ax, start, end, text=None, rad=0.0, label_offset=(0.0, 0.035), label_pos=0.5):
        arr = FancyArrowPatch(start, end, arrowstyle="-|>", mutation_scale=12, linewidth=1.15,
                              color="#2F3A45", connectionstyle=f"arc3,rad={rad}",
                              shrinkA=7, shrinkB=7, zorder=1)
        ax.add_patch(arr)
        if text:
            mx = start[0] + (end[0] - start[0]) * label_pos
            my = start[1] + (end[1] - start[1]) * label_pos
            ax.text(mx + label_offset[0], my + label_offset[1], _wrap_label(text, 14),
                    ha="center", va="center", fontsize=7.6, color="#2F3A45",
                    bbox=dict(boxstyle="round,pad=0.18", fc="white", ec="none", alpha=0.92),
                    zorder=4)

def _clean_method_label(name):
    return str(name).replace("Fed-DDQN (Proposed)", "Fed-DDQN\n(Proposed)")

def _latency_for_actions(df, actions):
    n = min(len(df), len(actions))
    edge = df["edge_latency"].to_numpy(dtype=np.float32)[:n]
    cloud = df["cloud_latency"].to_numpy(dtype=np.float32)[:n]
    a = np.asarray(actions[:n], dtype=np.int64)
    return np.where(a == 0, edge, cloud), a

def _sla_for_df(df):
    if "task_type" in df.columns and _has("SLA_MS"):
        return df["task_type"].map(SLA_MS).fillna(9999).to_numpy(dtype=np.float32)
    if "sla_violated" in df.columns:
        return np.full(len(df), np.inf, dtype=np.float32)
    return np.full(len(df), np.inf, dtype=np.float32)

def _feature_col(idx_name):
    return globals().get(idx_name, None)

def _safe_quantile_bins(values, q=5):
    s = pd.Series(values).replace([np.inf, -np.inf], np.nan).dropna()
    if s.nunique() < 2:
        return None
    return pd.qcut(s, q=min(q, s.nunique()), duplicates="drop")

_extra_start = len(_result_figures_saved)

# Figure A13: Fed-DDQN learning dynamics.
if _has("federated_losses") and len(federated_losses):
    rounds = np.arange(1, len(federated_losses) + 1)
    fig, ax1 = plt.subplots(figsize=(10.4, 5.8))
    ax1.plot(rounds, federated_losses, marker="o", color="#A32D2D", label="Training TD loss")
    ax1.set_xlabel("Federated round")
    ax1.set_ylabel("TD loss", color="#A32D2D")
    ax1.tick_params(axis="y", labelcolor="#A32D2D")
    ax1.grid(alpha=0.25)
    ax2 = ax1.twinx()
    if _has("federated_rewards") and len(federated_rewards):
        ax2.plot(rounds[:len(federated_rewards)], federated_rewards, marker="s", color="#1D9E75", label="Mean reward")
        ax2.set_ylabel("Mean reward", color="#1D9E75")
        ax2.tick_params(axis="y", labelcolor="#1D9E75")
    if globals().get("best_round", 0):
        ax1.axvline(int(best_round), color="#2F3A45", linestyle="--", linewidth=1.2)
        ax1.text(int(best_round), ax1.get_ylim()[1], f" best round {int(best_round)} ",
                 ha="center", va="top", fontsize=8.2,
                 bbox=dict(boxstyle="round,pad=0.18", fc="white", ec="none", alpha=0.9))
    ax1.set_title("Fed-DDQN Learning Dynamics and Validation Checkpoint", fontweight="bold")
    _result_save(fig, "fig_a13_fed_ddqn_learning_dynamics")

# Figure A14: Multi-objective normalized scorecard.
if _has("df_compare") and len(df_compare):
    dfc = _df_with_method(df_compare)
    metrics = [
        ("Avg Latency", False),
        ("SLA Miss %", False),
        ("Rejection %", False),
        ("Comm Delay (ms)", False),
        ("BW Consump %", False),
        ("SLA %", True),
    ]
    usable = [(m, high_good) for m, high_good in metrics if m in dfc.columns]
    if usable:
        score = pd.DataFrame(index=dfc["Method"].astype(str))
        for metric, high_good in usable:
            vals = dfc[metric].map(_num).astype(float)
            lo, hi = vals.min(), vals.max()
            if not np.isfinite(lo) or not np.isfinite(hi) or hi == lo:
                norm = pd.Series(1.0, index=vals.index)
            elif high_good:
                norm = (vals - lo) / (hi - lo)
            else:
                norm = 1.0 - ((vals - lo) / (hi - lo))
            score[metric] = norm.to_numpy()
        fig, ax = plt.subplots(figsize=(10.8, max(4.8, 0.48 * len(score) + 2.2)))
        im = ax.imshow(score.to_numpy(), cmap="YlGnBu", aspect="auto", vmin=0, vmax=1)
        ax.set_yticks(np.arange(len(score.index)))
        ax.set_yticklabels([_clean_method_label(m) for m in score.index], fontsize=8.5)
        ax.set_xticks(np.arange(len(score.columns)))
        ax.set_xticklabels(score.columns, rotation=25, ha="right", fontsize=8.5)
        for i in range(score.shape[0]):
            for j in range(score.shape[1]):
                ax.text(j, i, f"{score.iat[i, j]:.2f}", ha="center", va="center", fontsize=7.5, color="#17202A")
        ax.set_title("Normalized Multi-Objective Policy Scorecard (Higher is Better)", fontweight="bold")
        fig.colorbar(im, ax=ax, fraction=0.035, pad=0.03)
        _result_save(fig, "fig_a14_policy_multiobjective_scorecard")

# Figure A15: Agreement with Oracle routing decisions.
if _has("actions_by_method") and "Fed-DDQN (Proposed)" in actions_by_method and "Oracle" in actions_by_method:
    fed = np.asarray(actions_by_method["Fed-DDQN (Proposed)"], dtype=np.int64)
    oracle = np.asarray(actions_by_method["Oracle"], dtype=np.int64)
    n = min(len(fed), len(oracle))
    if n:
        mat = np.zeros((2, 2), dtype=int)
        for o, f in zip(oracle[:n], fed[:n]):
            if o in (0, 1) and f in (0, 1):
                mat[int(o), int(f)] += 1
        pct = mat / max(mat.sum(), 1) * 100.0
        fig, ax = plt.subplots(figsize=(6.8, 5.6))
        im = ax.imshow(pct, cmap="Blues", vmin=0)
        ax.set_xticks([0, 1]); ax.set_xticklabels(["Fed edge", "Fed cloud"])
        ax.set_yticks([0, 1]); ax.set_yticklabels(["Oracle edge", "Oracle cloud"])
        for i in range(2):
            for j in range(2):
                ax.text(j, i, f"{pct[i, j]:.1f}%\n({mat[i, j]})", ha="center", va="center", fontsize=10)
        ax.set_title("Fed-DDQN Agreement with Oracle Routing Ceiling", fontweight="bold")
        fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
        _result_save(fig, "fig_a15_fed_oracle_agreement_matrix")

# Figure A16: Channel-awareness over SNR.
if _has("eval_raw") and _has("eval_sample") and _has("actions_by_method") and "Fed-DDQN (Proposed)" in actions_by_method and _feature_col("_F_SNR") is not None:
    snr = np.asarray(eval_raw[:, _feature_col("_F_SNR")], dtype=np.float32)
    bins = _safe_quantile_bins(snr, q=5)
    if bins is not None:
        lat, act = _latency_for_actions(eval_sample, actions_by_method["Fed-DDQN (Proposed)"])
        tmp = pd.DataFrame({"SNR bin": bins.astype(str), "Latency": lat[:len(bins)], "Edge": (act[:len(bins)] == 0).astype(float) * 100.0})
        grp = tmp.groupby("SNR bin", observed=False).agg({"Latency": "mean", "Edge": "mean"}).reset_index()
        fig, ax1 = plt.subplots(figsize=(10.4, 5.8))
        ax1.plot(grp["SNR bin"], grp["Latency"], marker="o", color="#A32D2D", label="Latency")
        ax1.set_ylabel("Avg latency (ms)", color="#A32D2D")
        ax1.tick_params(axis="y", labelcolor="#A32D2D")
        ax1.tick_params(axis="x", rotation=18)
        ax1.grid(alpha=0.25)
        ax2 = ax1.twinx()
        ax2.plot(grp["SNR bin"], grp["Edge"], marker="s", color="#1D9E75", label="Edge usage")
        ax2.set_ylabel("Edge usage (%)", color="#1D9E75")
        ax2.tick_params(axis="y", labelcolor="#1D9E75")
        ax1.set_title("Channel-Aware Fed-DDQN Behavior Across SNR Regimes", fontweight="bold")
        _result_save(fig, "fig_a16_snr_regime_policy_response")

# Figure A17: Packet-loss robustness.
if _has("eval_raw") and _has("eval_sample") and _has("actions_by_method") and "Fed-DDQN (Proposed)" in actions_by_method and _feature_col("_F_LOSS") is not None:
    loss = np.asarray(eval_raw[:, _feature_col("_F_LOSS")], dtype=np.float32)
    bins = _safe_quantile_bins(loss, q=5)
    if bins is not None:
        lat, act = _latency_for_actions(eval_sample, actions_by_method["Fed-DDQN (Proposed)"])
        sla = _sla_for_df(eval_sample)[:len(lat)]
        risk = (lat[:len(sla)] >= sla) | (eval_sample.get("rejection_flag", pd.Series(0, index=eval_sample.index)).to_numpy()[:len(sla)] > 0)
        tmp = pd.DataFrame({"Packet-loss bin": bins.astype(str), "Latency": lat[:len(bins)], "Risk": risk[:len(bins)].astype(float) * 100.0})
        grp = tmp.groupby("Packet-loss bin", observed=False).agg({"Latency": "mean", "Risk": "mean"}).reset_index()
        fig, ax1 = plt.subplots(figsize=(10.6, 5.8))
        ax1.bar(grp["Packet-loss bin"], grp["Latency"], color="#7F77DD", alpha=0.82)
        ax1.set_ylabel("Avg latency (ms)")
        ax1.tick_params(axis="x", rotation=18)
        ax1.grid(axis="y", alpha=0.25)
        ax2 = ax1.twinx()
        ax2.plot(grp["Packet-loss bin"], grp["Risk"], color="#D85A30", marker="o", linewidth=2.0)
        ax2.set_ylabel("SLA/rejection risk (%)", color="#D85A30")
        ax2.tick_params(axis="y", labelcolor="#D85A30")
        ax1.set_title("Fed-DDQN Robustness Under Packet-Loss Stress", fontweight="bold")
        _result_save(fig, "fig_a17_packet_loss_policy_robustness")

# Figure A18: Edge queue pressure response.
if _has("eval_raw") and _has("eval_sample") and _has("actions_by_method") and "Fed-DDQN (Proposed)" in actions_by_method and _feature_col("_F_E_QUEUE") is not None:
    queue = np.asarray(eval_raw[:, _feature_col("_F_E_QUEUE")], dtype=np.float32)
    bins = _safe_quantile_bins(queue, q=5)
    if bins is not None:
        lat, act = _latency_for_actions(eval_sample, actions_by_method["Fed-DDQN (Proposed)"])
        tmp = pd.DataFrame({"Edge-queue bin": bins.astype(str), "Latency": lat[:len(bins)], "Edge": (act[:len(bins)] == 0).astype(float) * 100.0})
        grp = tmp.groupby("Edge-queue bin", observed=False).agg({"Latency": "mean", "Edge": "mean"}).reset_index()
        fig, ax1 = plt.subplots(figsize=(10.6, 5.8))
        ax1.plot(grp["Edge-queue bin"], grp["Edge"], marker="o", color="#1D9E75")
        ax1.set_ylabel("Edge usage (%)", color="#1D9E75")
        ax1.tick_params(axis="y", labelcolor="#1D9E75")
        ax1.tick_params(axis="x", rotation=18)
        ax1.grid(alpha=0.25)
        ax2 = ax1.twinx()
        ax2.plot(grp["Edge-queue bin"], grp["Latency"], marker="s", color="#A32D2D")
        ax2.set_ylabel("Avg latency (ms)", color="#A32D2D")
        ax2.tick_params(axis="y", labelcolor="#A32D2D")
        ax1.set_title("Fed-DDQN Response to Edge Queue Contention", fontweight="bold")
        _result_save(fig, "fig_a18_edge_queue_pressure_response")

# Figure A19: Dataset3 workload and network event timeline.
if _has("tasks") and "arrival_time" in tasks.columns:
    arrivals = tasks.groupby("arrival_time").size().sort_index()
    fig, ax1 = plt.subplots(figsize=(12.4, 5.8))
    ax1.plot(arrivals.index, arrivals.values, color="#2F3A45", linewidth=1.4, label="Task arrivals")
    ax1.set_xlabel("Timestep")
    ax1.set_ylabel("Arriving tasks")
    ax1.grid(alpha=0.22)
    title_extra = ""
    if _has("network_state") and "time_step" in network_state.columns:
        ns = network_state.sort_values("time_step")
        plotted = False
        ax2 = ax1.twinx()
        if "packet_loss_rate" in ns.columns:
            ax2.plot(ns["time_step"], ns["packet_loss_rate"], color="#D85A30", alpha=0.65, linewidth=1.0, label="Packet loss")
            ax2.set_ylabel("Packet loss / event flag", color="#D85A30")
            ax2.tick_params(axis="y", labelcolor="#D85A30")
            plotted = True
        for col in ["outage", "is_outage", "network_outage", "congestion", "jitter_storm"]:
            if col in ns.columns:
                event = ns[col].astype(float).to_numpy()
                if event.max() > 0:
                    ax2.fill_between(ns["time_step"], 0, event, step="mid", alpha=0.12, color="#A32D2D")
                    title_extra = " with Network Stress Overlay"
                    plotted = True
        if not plotted:
            ax2.remove()
    ax1.set_title(f"Dataset3 Temporal Workload Burst Profile{title_extra}", fontweight="bold")
    _result_save(fig, "fig_a19_dataset3_workload_network_timeline")

# Figure A20: Stage-2 resource error CDF.
if _has("stage2_targets_feasible") and _has("residual_risk_projection"):
    target = np.asarray(stage2_targets_feasible, dtype=np.float32)
    pred = np.asarray(residual_risk_projection, dtype=np.float32)
    n = min(len(target), len(pred))
    if n:
        target = target[:n]
        pred = pred[:n]
        labels = ["CPU", "Memory", "Bandwidth"]
        fig, ax = plt.subplots(figsize=(8.8, 5.6))
        for i, label in enumerate(labels):
            err = np.abs(pred[:, i] - target[:, i])
            xs = np.sort(err)
            ys = np.linspace(0, 100, len(xs), endpoint=True)
            ax.plot(xs, ys, linewidth=2.0, label=label)
        ax.set_xlabel("Absolute allocation error")
        ax.set_ylabel("CDF (%)")
        ax.set_title("Stage-2 Residual Allocator Error Distribution", fontweight="bold")
        ax.grid(alpha=0.25)
        ax.legend(frameon=False)
        _result_save(fig, "fig_a20_allocator_resource_error_cdf")

# Figure A21: Stage-2 underallocation vs excess allocation profile.
if _has("stage2_targets_feasible") and _has("residual_risk_projection"):
    target = np.asarray(stage2_targets_feasible, dtype=np.float32)
    pred = np.asarray(residual_risk_projection, dtype=np.float32)
    n = min(len(target), len(pred))
    if n:
        diff = pred[:n] - target[:n]
        data = []
        labels = []
        for i, res in enumerate(["CPU", "Memory", "Bandwidth"]):
            data.extend([np.maximum(-diff[:, i], 0.0), np.maximum(diff[:, i], 0.0)])
            labels.extend([f"{res}\nUnder", f"{res}\nExcess"])
        fig, ax = plt.subplots(figsize=(9.8, 5.8))
        bp = ax.boxplot(data, labels=labels, patch_artist=True, showfliers=False)
        for j, patch in enumerate(bp["boxes"]):
            patch.set_facecolor("#FFF0F0" if j % 2 == 0 else "#EFFFF5")
            patch.set_edgecolor("#2F3A45")
        ax.set_ylabel("Allocation magnitude")
        ax.set_title("Stage-2 QoS Risk: Underallocation vs Excess Allocation", fontweight="bold")
        ax.grid(axis="y", alpha=0.25)
        _result_save(fig, "fig_a21_allocator_under_vs_excess_profile")

# Figure A22: End-to-end experimental methodology flowchart.
fig, ax = plt.subplots(figsize=(15.2, 6.4))
ax.set_axis_off()
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
_box(ax, (0.035, 0.61), (0.13, 0.16), "Dataset3\nIoT tasks + edge,\ncloud, network\nstates", "#EAF4FF", fontsize=8.0, wrap_width=15)
_box(ax, (0.215, 0.61), (0.13, 0.16), "Temporal split\ntrain, val, test\nwith purge gaps", "#F6F8FA", fontsize=8.0, wrap_width=15)
_box(ax, (0.395, 0.61), (0.14, 0.16), "Stage 1 training\nfederated DDQN\nzone clients", "#EFFFF5", fontsize=8.0, wrap_width=15)
_box(ax, (0.585, 0.61), (0.14, 0.16), "Test-only policy\nevaluation\nvs baselines", "#FFF3E6", fontsize=8.0, wrap_width=15)
_box(ax, (0.785, 0.61), (0.15, 0.16), "Stage 1 claims\nlatency, SLA,\nrejection, fairness", "#F5F0FF", fontsize=8.0, wrap_width=15)
_box(ax, (0.395, 0.24), (0.14, 0.16), "Ablation suite\nmulti-seed\ncomponent removal", "#E8F0FE", fontsize=8.0, wrap_width=15)
_box(ax, (0.585, 0.24), (0.14, 0.16), "Stage 2 allocator\nedge-selected\ntasks only", "#FFF0F0", fontsize=8.0, wrap_width=15)
_box(ax, (0.785, 0.24), (0.15, 0.16), "Stage 2 claims\nMAE, capacity,\nQoS resource use", "#F0FAFF", fontsize=8.0, wrap_width=15)
_arrow(ax, (0.165, 0.69), (0.215, 0.69))
_arrow(ax, (0.345, 0.69), (0.395, 0.69))
_arrow(ax, (0.535, 0.69), (0.585, 0.69))
_arrow(ax, (0.725, 0.69), (0.785, 0.69))
_arrow(ax, (0.465, 0.61), (0.465, 0.40), "robustness", label_offset=(0.075, 0.0))
_arrow(ax, (0.655, 0.61), (0.655, 0.40), "edge tasks", label_offset=(0.075, 0.0))
_arrow(ax, (0.725, 0.32), (0.785, 0.32))
ax.text(0.5, 0.92, "Evaluation Methodology for the Proposed Fed-DDQN + Adaptive Allocation System",
        ha="center", va="center", fontsize=13, fontweight="bold")
_result_save(fig, "fig_a22_experimental_methodology_flowchart")

# Figure A23: Research-claim to evidence map.
claim_rows = [
    ("Channel-aware offloading", "SNR/loss/queue features + regime plots", "A16-A18"),
    ("Federated DDQN policy", "zone clients, FedAvg/FedProx, best validation checkpoint", "A1, A13"),
    ("Fair comparison", "test-only baseline table + Oracle ceiling", "A3-A6, A14-A15"),
    ("Reliability", "multi-seed and ablation summary", "A7, A13"),
    ("Rejection-aware resource allocation", "residual allocator + capacity normalization", "A8-A12, A20-A21"),
]
fig, ax = plt.subplots(figsize=(12.8, 4.8))
ax.set_axis_off()
table_data = [[a, b, c] for a, b, c in claim_rows]
tbl = ax.table(cellText=table_data, colLabels=["Paper claim", "Evidence in notebook", "Figure(s)"],
               loc="center", cellLoc="left", colLoc="left", colWidths=[0.27, 0.53, 0.16])
tbl.auto_set_font_size(False)
tbl.set_fontsize(8.4)
tbl.scale(1.0, 1.55)
for (r, c), cell in tbl.get_celld().items():
    cell.set_edgecolor("#D0D7DE")
    if r == 0:
        cell.set_facecolor("#EAF4FF")
        cell.set_text_props(weight="bold")
    else:
        cell.set_facecolor("white" if r % 2 else "#F6F8FA")
ax.set_title("Research Claims and Supporting Evidence Map", fontweight="bold", pad=14)
_result_save(fig, "fig_a23_research_claim_evidence_map")

# Figure A24: Compact key-metric table for result review.
if _has("df_compare") and len(df_compare):
    cols = [c for c in ["Avg Latency", "SLA %", "Rejection %", "Edge Usage %", "Comm Delay (ms)", "BW Consump %"] if c in df_compare.columns]
    methods = [m for m in ["Fed-DDQN (Proposed)", "DDQN", "FL-DDPG", "Oracle"] if m in df_compare.index]
    if cols and methods:
        small = df_compare.loc[methods, cols].copy()
        small = small.applymap(lambda x: f"{_num(x):.2f}" if pd.notna(_num(x)) else str(x))
        fig, ax = plt.subplots(figsize=(12.4, 3.6 + 0.25 * len(methods)))
        ax.set_axis_off()
        tbl = ax.table(cellText=small.values, rowLabels=[_clean_method_label(m) for m in small.index],
                       colLabels=small.columns, loc="center", cellLoc="center", rowLoc="center")
        tbl.auto_set_font_size(False)
        tbl.set_fontsize(8.2)
        tbl.scale(1.0, 1.45)
        for (r, c), cell in tbl.get_celld().items():
            cell.set_edgecolor("#D0D7DE")
            if r == 0 or c == -1:
                cell.set_facecolor("#EAF4FF")
                cell.set_text_props(weight="bold")
            elif r % 2 == 0:
                cell.set_facecolor("#F6F8FA")
        ax.set_title("Compact Test-Only Policy Metrics for Manuscript Table Drafting", fontweight="bold", pad=14)
        _result_save(fig, "fig_a24_compact_policy_metric_table")

print("\nAdditional result figures generated.")
for title, png in _result_figures_saved[_extra_start:]:
    print(f"- {title}: {png} | {pdf}")
